# 캐릭터 LoRA 학습 노트북 — 영상 모델 (Wan 2.1·2.2)

이 노트북은 영상 생성 모델에 캐릭터 LoRA 를 학습한다. 베이스 모델은 얼리고 LoRA 어댑터만 학습한다.

| 단계 | 하는 일 |
|---|---|
| 0. 준비 | 라이브러리 확인, 설정(고정값), 도구·어댑터, 모델 불러오기 |
| 1. EDA | 이미지·클립의 프레임 수·fps·길이·회전·VFR, 검은·멈춘 구간, 장면 전환, 움직임, 흐림, 중복, 캡션 검사 |
| 2. 시각화 | 분포 그래프, 컨택트 시트, 클립별 신호 곡선 |
| 3. 전처리 | 모델 fps 로 리샘플, 깨끗한 구간에서 창 고르기, (프레임·높이·너비) 버킷, 잠재값·텍스트 캐시 |
| 4. 학습 | 전문가별 LoRA 학습, 정해진 단계마다 저장·검증 손실·미리보기 클립 |
| 5. 검증 | 손실 곡선, 단계별 미리보기, 저장 파일 다시 불러오기, 최종 검증 세트와 지표(움직임·깜빡임·색·정체성), 새 프롬프트 시험 |
| 6. 보완 | 지표로 증상 진단, 체크포인트 추천, 다음 실행 설정(`next_run.json`), 결과 요약 |

- 모델을 바꿀 때 고치는 값은 `0-2` 의 `USER` 처음 5개뿐이다: `TRIGGER`, `MODEL_PATH`, `RUN_NAME`, `DATASET_DIR`, `TARGET_SCOPE` (환경변수 `LORA_*` 로도 줄 수 있다).
  - 모델 계열은 `MODEL_PATH` 폴더의 설정 파일로 자동으로 정하고, 계열별 값(fps·버킷·검증 step·전문가 수)도 따라 바뀐다.
- 학습 조건은 `FIXED` 에 고정되어 있다.
- `FIXED` 를 바꾸는 방법은 하나다. `LORA_FIXED_OVERRIDES` 에 `{키: 값}` 을 준다 (6 보완 단계가 만든 `next_run.json` 의 값을 그대로 넣거나, 예: `{"token_budget": 20280}`).
  - `FIXED` 에 있는 키만 받고, 보완 규칙(`remedy.ALLOWED`)에 범위·선택지가 있는 값은 그 안에서만 받는다.
- 모델은 diffusers 형식 폴더(`model_index.json` 이 있는 폴더) 또는 Hugging Face 저장소 이름으로 불러온다.

### 지원 모델 (2026-11-05 조사, 모두 Apache-2.0)

| 계열 (자동 인식) | 학습할 모델 | 학습 데이터 | LoRA 파일 | 쓰임 |
|---|---|---|---|---|
| `wan22_t2v` | Wan2.2-T2V-A14B-Diffusers | 이미지(1프레임)·클립 | 고노이즈·저노이즈 2개 | 지금 가진 이미지로 학습하는 1단계. 블록 모양이 같아 I2V-A14B 에도 걸 수 있다 (효과는 실측 필요) |
| `wan22_i2v` | Wan2.2-I2V-A14B-Diffusers | 클립만 | 2개 | 회사 기본 영상 모델. 클립이 생기면 학습 |
| `wan22_ti2v_5b` | Wan2.2-TI2V-5B-Diffusers | 이미지·클립 | 1개 | 파일럿 (가볍다). 5B LoRA 는 A14B 에 쓸 수 없다 |
| `wan21` | Wan2.1-T2V-1.3B·14B-Diffusers | 이미지·클립 | 1개 | 코드 점검(1.3B)·대안(14B) |

- 다른 후보를 뺀 이유 (라이선스 원문 확인)
  - HunyuanVideo·HunyuanVideo-1.5·MiniMax H3: 라이선스가 대한민국을 제외한다.
  - LTX-2.x: 연매출 US$10M 이상이면 유료 라이선스, 상업적으로 출력물을 다른 모델 학습에 쓸 수 없다.
  - Kandinsky 5·6: 모델은 MIT 지만 Tencent HunyuanVideo VAE 를 포함한다 (법률 검토 필요).
  - CogVideoX·Mochi 는 오래되었거나 사용 조건이 있고, LongCat-Video·MAGI 는 diffusers·ComfyUI 지원이 없다.
- 애니 특화 Index-AniSora V3.2 (Bilibili, Wan2.2-I2V-A14B 구조, Apache-2.0 + 추가 조건) 는 diffusers 형식으로 변환하면 `wan22_i2v` 로 학습할 수 있다 [미검증].

### A14B 의 두 전문가

- Wan2.2 A14B 는 고노이즈 전문가(`transformer`)와 저노이즈 전문가(`transformer_2`)로 나뉜다. 노이즈 단계 t = 1000·σ 가 경계(T2V 875, I2V 900) 이상이면 고노이즈 전문가가 그린다.
- 전문가마다 LoRA 를 따로 학습하고 파일도 따로 저장한다. 각 전문가는 자기 구간의 σ 로만 학습한다. 구간이 어긋나면 LoRA 가 효과가 없다.
- 기본은 저노이즈(정체성·세부)에 단계의 75%, 고노이즈(구도·움직임)에 25% 를 쓴다 [제안, 실무자 경험]. 고노이즈는 기본으로 클립으로만 학습한다. 정지 이미지가 움직임을 굳힐 위험 때문이다.

### 데이터셋 폴더

```
dataset/niki_video/
  niki_01.png   niki_01.txt     # 이미지 = 1프레임 영상
  walk_01.mp4   walk_01.txt     # 클립 (mp4·mov·mkv·webm·avi·gif)
```

- 캡션: 트리거 + 바뀌는 것(샷·카메라·시간 순서의 동작·배경). 머리·옷 같은 고정 외형은 적지 않는다.
  - 예: `nkq7, medium shot, static camera, she turns from the side to face the camera, then smiles, autumn park`
- 클립은 모델 fps(Wan 16, 5B 24)로 바꾼 뒤 4n+1 프레임 창으로 잘라 쓴다. 2~5초 클립이 알맞다. 늘이거나 줄이지 않는다.
- 이미지를 반복해 만든 가짜 클립은 넣지 않는다 (움직임을 잃는 원인으로 알려져 있다).
- `wan22_i2v` 는 클립이 있어야 한다. 1프레임 이미지는 조건 채널에 정답이 그대로 들어가 배우는 것이 없다.

### 메인 서버에서 실행하기

1. 학습용 가상환경에 영상 라이브러리를 더한다: `pip install av ftfy` (선택: 애니 캐릭터 정체성 지표용 `pip install onnxruntime`)
2. 모델과 검증 지표용 작은 모델을 받는다.
   - 예: `hf download Wan-AI/Wan2.2-T2V-A14B-Diffusers --local-dir /home/mgs/models/diffusers/Wan2.2-T2V-A14B-Diffusers`
   - 지표: `hf download facebook/dinov2-small config.json model.safetensors`
   - 선택(CCIP): `hf download deepghs/ccip_onnx ccip-caformer-24-randaug-pruned/model_feat.onnx ccip-caformer-24-randaug-pruned/model_metrics.onnx ccip-caformer-24-randaug-pruned/metrics.json`
   - 파일 이름을 적어 받는다. `--include` 는 값을 하나만 받는다: 뒤에 이어 쓴 값은 파일 이름이 되어 `--include` 가 무시되고 첫 값이 빠진다 (글롭이면 오류). 패턴을 여러 개 쓰려면 값마다 `--include` 를 붙인다.
   - 실행 공간에서 인터넷이 막혀 있으면 미리 받아 두고 `HF_HUB_OFFLINE=1` 로 실행한다. 아니면 4-2 셀이 내려받기를 몇 분 기다린 뒤 그 지표를 빼고 넘어간다. 지표가 필요 없으면 `LORA_FIXED_OVERRIDES='{"use_dino": false, "use_ccip": false}'` 로 끈다.
3. 처음 한 번은 아래 '시험 실행' 으로 돌려, 리눅스에서 영상 읽기(회전·VFR·프레임 수)가 맞는지 확인한다. EDA 셀이 합성 데이터의 정답과 비교한다.
4. 실제 학습 전에 ComfyUI 대기열을 멈춰 GPU 를 비운다 (계획서 P2).
- A14B 는 전문가 2개(bf16 각 약 28.6GB)를 번갈아 GPU 에 올리고, 쉬는 쪽은 CPU 메모리에 둔다. 미리보기·최종 생성 때는 모든 구성요소(약 69GB)가 CPU 메모리에 있다.
- 0-4 셀이 모델을 불러오기 전에 GPU 가 보이는지와 메모리를 확인하고, 모자라면 필요한 양을 보여 주고 멈춘다. 메모리 확인을 건너뛰려면 `LORA_SKIP_RAM_CHECK=1` 을 준다.
  - 시스템 메모리: 가중치 + 8GB 가 비어 있어야 한다. 컨테이너 메모리 한도나 서버 RAM 이 그보다 작으면 다른 프로그램을 멈춰도 안 되므로, 원인(한도·RAM)과 들어가는 더 작은 모델을 알린다.
  - GPU 메모리: 가장 큰 구성요소 + 4GB, 학습 단계 추정(LoRA·AdamW 포함), 생성(미리보기·최종) 추정 가운데 가장 큰 값이 비어 있어야 한다. 생성 추정은 전문가를 번갈아 올리는 A14B 면 전문가 하나 + 생성 토큰, 아니면 디노이저 모두 + VAE 디코드다.
- 시간과 GPU 메모리 [추정: 실제 폭의 블록을 노트북 GPU 에서 재고 연산량·층 수로 늘려 잡음. 첫 실행의 `summary.json` 으로 확인한다]
  - A14B·이미지만·기본값이면 연산의 약 27% 가 학습, 30% 가 미리보기(9번 × 3클립), 43% 가 최종 생성(10클립)이다. 서버 GPU 의 bf16 실효 성능을 200 TFLOPS 로 잡으면 약 2시간(학습만 약 33분)이다.
  - 미리보기를 줄이려면 `LORA_FIXED_OVERRIDES='{"validate_every": 500}'` 를 준다 (9번 → 5번).
  - A14B 학습 단계의 GPU 메모리는 약 30GB + 토큰당 약 1MB 다. 클립이 있으면 기본 예산 14,040토큰에서 약 44GB 다.
  - A14B 미리보기·최종 생성은 전문가 하나 + 생성 토큰(기본 33프레임 480x832 = 14,040토큰, 토큰당 약 0.25MB)으로 잡아, 이미지만 학습해도 0-4 셀은 약 35GB 가 비어 있기를 요구한다.
  - 토큰 예산(`token_budget`)은 주로 클립에 걸린다. 이미지는 `max_area`(480x832)에 묶여 한 장이 최대 1,560토큰(약 32GB)이라, 이미지만 학습하면 `token_budget` 을 늘려도 학습이 같다. 1,560 아래로 내리면 이미지도 작아진다 (예: 1,000 이면 512x768 → 400x608).
  - 첫 실행 뒤 `summary.json` 의 `train_minutes`(학습 단계만), `preview_minutes`(학습 중 미리보기·검증·저장), `train_peak_gpu_gb`(학습 단계 최대), `max_train_tokens` 를 본다. 클립이 있으면 0-4 셀이 보여 주는 '지금 들어가는 클립 token_budget' 안에서 늘린다.

### 시험 실행 (오류 검사)

환경변수 `LORA_NB_TEST=<모델 계열>` 을 주면, 같은 구조의 아주 작은 무작위 모델과 문제를 일부러 넣은 합성 데이터셋(회전·VFR·검은 프레임·장면 전환·중복 등)으로 모든 셀을 끝까지 돌리고 EDA 결과를 확인한다. 작은 토크나이저를 내려받으므로 인터넷이 필요하다. `LORA_NB_TEST_DEVICE=cuda` 를 더하면 GPU·bf16 으로 시험한다.

```
LORA_NB_TEST=wan22_t2v jupyter nbconvert --to notebook --execute lora_training_video.ipynb --output test_run.ipynb
```

## 0. 준비

### 0-1. 라이브러리 버전 확인과 불러오기

In [ ]:
# 고정 버전: 이 조합에서 4개 계열을 작은 무작위 모델로 시험했다.
#   pip install torch==2.15.0 diffusers==0.42.1 peft==0.21.2 transformers==5.22.1 accelerate==1.15.0 \
#               av safetensors sentencepiece protobuf ftfy pillow numpy pandas matplotlib tqdm
#   (선택) pip install onnxruntime   # 애니 캐릭터 정체성 지표(CCIP)
import importlib.metadata as _md

PINNED = {"torch": "2.15.0", "diffusers": "0.42.1", "peft": "0.21.2", "transformers": "5.22.1", "accelerate": "1.15.0"}
_rows = []
for _pkg, _want in PINNED.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        raise ImportError(f"{_pkg} 가 설치되어 있지 않습니다. 위 pip 명령으로 {_pkg}=={_want} 를 설치하세요.")
    _ok = _have.split("+")[0] == _want
    _rows.append((_pkg, _want, _have, "OK" if _ok else "버전 다름"))
for _r in _rows:
    print(f"{_r[0]:<13} 고정 {_r[1]:<8} 설치 {_r[2]:<14} {_r[3]}")
if any(r[3] != "OK" for r in _rows):
    print("\n경고: 고정 버전과 다릅니다. 시험하지 않은 조합이라 오류가 날 수 있습니다.")
for _pkg, _why in [("av", "영상 읽기·쓰기 (PyAV)"), ("ftfy", "Wan 프롬프트 정리 (diffusers 가 사용)")]:
    try:
        print(f"{_pkg:<13} {_md.version(_pkg):<23} {_why}")
    except _md.PackageNotFoundError:
        raise ImportError(f"{_pkg} 가 필요합니다: pip install {_pkg}")

import contextlib, copy, gc, hashlib, importlib, json, math, os, random, re, shutil, sys, tempfile, time
from collections import Counter, defaultdict
from pathlib import Path
from types import MappingProxyType

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

import torch
import diffusers
from diffusers.optimization import get_scheduler
from peft import LoraConfig, get_peft_model_state_dict
from tqdm.auto import tqdm

if not hasattr(sys, "ps1") and "ipykernel" not in sys.modules:
    matplotlib.use("Agg")
# 그래프의 한글: 설치된 한글 글꼴을 찾아 쓴다. 없으면 한글이 네모로 보이므로 경고만 끈다.
from matplotlib import font_manager
_KO_FONTS = ("NanumGothic", "NanumBarunGothic", "Malgun Gothic", "Noto Sans KR", "Noto Sans CJK KR", "Noto Sans CJK JP",
             "Noto Sans CJK SC", "AppleGothic", "UnDotum")
_ko_font = next((f for f in _KO_FONTS if f in {x.name for x in font_manager.fontManager.ttflist}), None)
if _ko_font is None:   # matplotlib 글꼴 캐시를 만든 뒤에 설치한 글꼴: 시스템 글꼴 파일에서 직접 찾아 넣는다
    for _f in font_manager.findSystemFonts():
        if re.search(r"nanum|notosanscjk|notosanskr|malgun|applegothic|undotum", Path(_f).name, re.I):
            try:
                font_manager.fontManager.addfont(_f)
            except Exception:
                pass
    _ko_font = next((f for f in _KO_FONTS if f in {x.name for x in font_manager.fontManager.ttflist}), None)
if _ko_font:
    plt.rcParams["font.family"] = _ko_font
    try:   # 컨택트 시트(PIL)도 같은 글꼴 파일을 쓴다
        os.environ["LORA_KO_FONT"] = font_manager.findfont(font_manager.FontProperties(family=_ko_font), fallback_to_default=False)
    except Exception:
        pass
else:
    import warnings
    warnings.filterwarnings("ignore", message=r"Glyph .* missing from")
    print("참고: 한글 글꼴이 없어 그래프·컨택트 시트의 한글이 네모로 보입니다. 리눅스에서 관리자 권한이 없으면 NanumGothic.ttf 를 "
          "~/.local/share/fonts 에 복사하고 커널을 다시 시작합니다 (관리자: sudo apt install fonts-nanum).")
plt.rcParams["axes.unicode_minus"] = False
try:
    from IPython.display import Video, display
except ImportError:
    Video = display = None
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### 0-2. 설정

- 모델을 바꿀 때 고치는 값은 `USER` 의 처음 5개뿐이다: `TRIGGER`, `MODEL_PATH`, `RUN_NAME`, `DATASET_DIR`, `TARGET_SCOPE`.
  - 환경변수 `LORA_TRIGGER`, `LORA_MODEL_PATH`, `LORA_RUN_NAME`, `LORA_DATASET_DIR`, `LORA_TARGET_SCOPE` 로도 줄 수 있다 (`run_many.py` 가 이 방식을 쓴다).
  - `MODEL_PATH`: `model_index.json` 이 있는 diffusers 형식 폴더(`~` 가능) 또는 Hugging Face 저장소 이름. 모델 계열은 이 폴더의 설정 파일로 자동으로 정하고, 계열별 값(fps·버킷·검증 step·전문가 수)도 따라 바뀐다.
  - `DATASET_DIR`: 이미지·클립과 같은 이름의 캡션(.txt)이 든 폴더 (`~` 가능, 상대 경로는 이 노트북 폴더 기준).
  - `TARGET_SCOPE`: LoRA 를 붙일 층. `all`(어텐션 + FFN = 블록 안 선형층 전부, 기본), `attn`(어텐션만), `kvo`(어텐션의 k·v·o 만, 움직임 보존용). 이미지 v2 의 `attn_mlp` 는 `all`, `attn_all` 은 `attn` 으로 받는다.
  - `RUN_NAME`: 결과 폴더 이름 (`OUTPUT_DIR/RUN_NAME`). 모델·설정을 바꿀 때마다 바꾼다. 그 폴더에 저장된 LoRA 가 있으면 덮어쓰지 않게 여기서 멈춘다 (첫 저장 전에 멈춘 실행의 폴더는 이어서 쓴다).
- 아래 값은 보통 그대로 둔다.
  - `MODEL_FAMILY`: `auto` 면 위처럼 자동으로 정한다. 계열 이름을 적으면 모델 폴더와 맞는지 확인한다.
  - `EXPERT`: `auto`(A14B 는 저노이즈 → 고노이즈 순서로 둘 다, 클립이 없으면 저노이즈만. `both` 도 같다), `low`, `high`(클립 필요).
  - `CLASS_WORD`: 트리거 대신 넣는 일반 낱말 (그림체 변화 확인·보존 학습). `auto` 면 캡션에서 girl·boy·woman 등을 찾는다.
  - `FIXED_OVERRIDES`: 6 보완 단계가 만든 `next_run.json` 의 `fixed_overrides` (JSON 문자열이나 파일 경로). `FIXED` 에 있는 키만 받는다.
  - `VALIDATION_FILE`: 검증 프롬프트 JSON. 비우면 움직임이 있는 기본 4개를 쓴다.
    - 예: `{"prompts": ["full body, static camera, she waves"], "negative": "...", "seeds": [1001], "images": ["start_01.png"]}`
    - `images` 는 I2V 의 시작 이미지다 (`wan22_i2v` 에서 비우면 데이터셋 이미지나 클립 첫 프레임을 쓴다).
- `FIXED` 는 공통 시작값에 계열별 값을 덮어쓴 고정값이다. 대부분 근거가 약한 시작값이라 [제안] 이다. 출처는 주석에 있다.
- 랭크와 알파는 늘 같다. ComfyUI 는 alpha 값이 없는 LoRA 를 강도 1(알파 = 랭크)로 적용한다.

| 학습할 모델 | `MODEL_PATH` 예 (메인 서버) | `RUN_NAME` 예 | 학습 데이터 |
|---|---|---|---|
| Wan2.2-T2V-A14B | `/home/mgs/models/diffusers/Wan2.2-T2V-A14B-Diffusers` | `nkq7_wan22t2v_v1` | 이미지·클립 (이미지만 있으면 저노이즈 LoRA 만) |
| Wan2.2-I2V-A14B | `/home/mgs/models/diffusers/Wan2.2-I2V-A14B-Diffusers` | `nkq7_wan22i2v_v1` | 클립 필수 |
| Wan2.2-TI2V-5B | `/home/mgs/models/diffusers/Wan2.2-TI2V-5B-Diffusers` | `nkq7_wan22ti2v5b_v1` | 이미지·클립 |
| Wan2.1-T2V-14B·1.3B | `/home/mgs/models/diffusers/Wan2.1-T2V-14B-Diffusers` | `nkq7_wan21_14b_v1` | 이미지·클립 |

In [ ]:
TEST_FAMILY = os.environ.get("LORA_NB_TEST", "").strip()     # 시험 실행이면 모델 계열 이름
SMOKE = os.environ.get("LORA_NB_SMOKE", "").strip() == "1"   # 개발용: 실제 경로 그대로 두고 학습 크기만 시험 실행처럼 줄임

USER = MappingProxyType(dict(
    # ── 모델·실행마다 바꾸는 값: 이 5개만 고친다 ──
    TRIGGER=os.environ.get("LORA_TRIGGER", "nkq7"),
    MODEL_PATH=os.environ.get("LORA_MODEL_PATH", "/home/mgs/models/diffusers/Wan2.2-T2V-A14B-Diffusers"),
    RUN_NAME=os.environ.get("LORA_RUN_NAME", "nkq7_wan22t2v_v1"),
    DATASET_DIR=os.environ.get("LORA_DATASET_DIR", "~/lora_datasets/nkq7_final"),
    TARGET_SCOPE=os.environ.get("LORA_TARGET_SCOPE", "all"),                 # all / attn / kvo
    # ── 보통 그대로 둔다 ──
    MODEL_FAMILY=os.environ.get("LORA_MODEL_FAMILY", TEST_FAMILY or "auto"),   # auto = MODEL_PATH 의 설정 파일로 정함
    OUTPUT_DIR=os.environ.get("LORA_OUTPUT_DIR", "outputs_video"),
    EXPERT=os.environ.get("LORA_EXPERT", "auto"),                            # auto / both / low / high (A14B 만 의미 있음)
    CLASS_WORD=os.environ.get("LORA_CLASS_WORD", "auto"),                    # auto = 캡션에서 찾음 (girl·boy·woman …)
    VALIDATION_FILE=os.environ.get("LORA_VALIDATION_FILE", ""),
    FIXED_OVERRIDES=os.environ.get("LORA_FIXED_OVERRIDES", ""),              # 6 보완 단계의 fixed_overrides (JSON 문자열 또는 파일)
    CAPTION_FIXED_WORDS=os.environ.get("LORA_CAPTION_FIXED_WORDS", ""),      # 캡션에 쓰지 않을 고정 외형 낱말 (쉼표로 구분)
))
for _k in ("TRIGGER", "MODEL_PATH", "RUN_NAME", "DATASET_DIR", "TARGET_SCOPE"):
    if not USER[_k].strip():
        raise ValueError(f"{_k} 가 비어 있습니다")
if USER["TRIGGER"] != USER["TRIGGER"].strip():
    raise ValueError(f"TRIGGER='{USER['TRIGGER']}' 앞뒤에 공백이 있습니다")
if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*", USER["RUN_NAME"]):   # run_many.py 와 같은 규칙
    raise ValueError(f"RUN_NAME='{USER['RUN_NAME']}' 는 영문·숫자로 시작하고 영문·숫자와 . _ - 만 씁니다 (예: nkq7_wan22t2v_v1)")
if USER["EXPERT"] not in ("auto", "both", "low", "high"):
    raise ValueError(f"EXPERT='{USER['EXPERT']}' 는 쓸 수 없습니다. auto / both / low / high")

# 학습 범위. 이미지 v2 이름도 받는다 (Wan 블록의 선형층은 어텐션 q·k·v·o 와 FFN 이 전부라 attn_mlp = all)
SCOPE_ALIASES = {"attn_mlp": "all", "attn_all": "attn"}
SCOPE = SCOPE_ALIASES.get(USER["TARGET_SCOPE"].strip(), USER["TARGET_SCOPE"].strip())
if SCOPE not in ("all", "attn", "kvo"):
    raise ValueError(f"TARGET_SCOPE='{USER['TARGET_SCOPE']}' 는 쓸 수 없습니다. all / attn / kvo (이미지 v2 의 attn_mlp·attn_all 도 받음)")

# 모델 계열 자동 인식: 설정 파일(JSON)만 읽는다. 0-4 에서 불러올 때 어댑터가 같은 값을 한 번 더 확인한다.
WAN_MODELS = {"wan22_t2v": "Wan2.2-T2V-A14B", "wan22_i2v": "Wan2.2-I2V-A14B", "wan22_ti2v_5b": "Wan2.2-TI2V-5B",
              "wan21": "Wan2.1-T2V-1.3B·14B"}

def norm_model_path(mp):
    """MODEL_PATH 정리: 앞뒤 공백, Hugging Face 주소(https://huggingface.co/org/name/...), 저장소 이름 끝의 '/' 를 뗀다."""
    s = str(mp).strip()
    if re.match(r"https?://(?:www\.)?huggingface\.co/", s):
        s = re.sub(r"/(?:tree|blob|resolve)/.*$", "", re.sub(r"^https?://(?:www\.)?huggingface\.co/", "", s))
    if not Path(s).expanduser().exists():
        s = s.rstrip("/\\")
    return s

def model_config_reader(mp):
    """MODEL_PATH 의 설정 파일을 읽는 함수를 돌려준다. 로컬 폴더 또는 Hugging Face 저장소 이름 (설정 파일만 받는다)."""
    mp = norm_model_path(mp)
    p = Path(mp).expanduser()
    if p.is_dir():
        return lambda rel: json.loads((p / rel).read_text(encoding="utf-8")) if (p / rel).is_file() else None
    if p.exists():
        raise ValueError(f"MODEL_PATH 가 파일입니다: {p}\n  model_index.json 이 있는 diffusers 형식 폴더를 적습니다 "
                         "(ComfyUI 용 .safetensors 한 파일은 읽지 못합니다).")
    repo = str(mp).strip()
    if p.is_absolute() or repo.startswith((".", "~")) or not re.fullmatch(r"[\w.-]+/[\w.-]+", repo):
        raise FileNotFoundError(f"MODEL_PATH 폴더가 없습니다: {mp}\n  diffusers 형식 모델 폴더(model_index.json 이 있는 폴더)의 경로나 "
                                "Hugging Face 저장소 이름(org/name)을 적습니다. 예: /home/mgs/models/diffusers/Wan2.2-T2V-A14B-Diffusers")
    from huggingface_hub import hf_hub_download

    def read(rel):
        try:
            return json.loads(Path(hf_hub_download(repo, rel)).read_text(encoding="utf-8"))
        except Exception as e:
            if rel == "model_index.json":
                raise FileNotFoundError(f"Hugging Face 저장소 {repo} 의 model_index.json 을 받지 못했습니다 ({type(e).__name__}). "
                                        "저장소 이름(…-Diffusers)·인터넷·로그인을 확인합니다.") from e
            return None
    return read

def detect_family(mp):
    """model_index.json 과 transformer·vae 설정으로 Wan 계열을 정한다 (가중치는 읽지 않는다)."""
    read = model_config_reader(mp)
    idx = read("model_index.json")
    if idx is None:
        raise FileNotFoundError(f"{mp} 에 model_index.json 이 없습니다. diffusers 형식 모델(저장소 이름이 -Diffusers 로 끝남)이 필요합니다.")
    cls = idx.get("_class_name", "")
    two = bool((idx.get("transformer_2") or [None])[0]) and idx.get("boundary_ratio") is not None
    cin = (read("transformer/config.json") or {}).get("in_channels")
    zd = (read("vae/config.json") or {}).get("z_dim")
    expand, clip = bool(idx.get("expand_timesteps") or False), bool((idx.get("image_encoder") or [None])[0])
    if cls == "WanPipeline" and two and (cin, zd) == (16, 16) and not expand:
        return "wan22_t2v"
    if cls == "WanImageToVideoPipeline" and two and (cin, zd) == (36, 16) and not expand and not clip:
        return "wan22_i2v"
    if cls == "WanPipeline" and not two and (cin, zd) == (48, 48) and expand:
        return "wan22_ti2v_5b"
    if cls == "WanPipeline" and not two and (cin, zd) == (16, 16) and not expand:
        return "wan21"
    raise ValueError(f"{mp} 는 이 노트북이 학습하는 Wan 모델이 아닙니다 (파이프라인 {cls or '?'}, 전문가 {2 if two else 1}개, "
                     f"in_channels {cin}, VAE z_dim {zd}, expand_timesteps {expand}{', CLIP 이미지 인코더 있음' if clip else ''}).\n"
                     "  지원: " + ", ".join(f"{v}-Diffusers" for v in WAN_MODELS.values())
                     + " (Wan2.1 I2V·FLF2V·VACE 등은 지원하지 않음)")

if TEST_FAMILY:
    FAMILY = TEST_FAMILY
else:
    FAMILY = detect_family(USER["MODEL_PATH"])
    if USER["MODEL_FAMILY"].strip() not in ("", "auto") and USER["MODEL_FAMILY"].strip() != FAMILY:
        raise ValueError(f"MODEL_FAMILY='{USER['MODEL_FAMILY']}' 인데 MODEL_PATH 는 {FAMILY} ({WAN_MODELS[FAMILY]}) 모델입니다. "
                         "MODEL_FAMILY 를 auto 로 두면 모델 폴더에 맞춰 정합니다.")

COMMON_FIXED = dict(
    seed=42,
    # LoRA: 블록 안 선형층만 (musubi-tuner·DiffSynth·ai-toolkit 과 같은 범위).
    #   target_scope 는 USER 의 TARGET_SCOPE: all = 어텐션 q·k·v·o + FFN, attn = 어텐션만, kvo = k·v·o 만 (움직임 보존 연구 V-Warper)
    rank=16, alpha=16, dropout=0.0, init_lora_weights="gaussian", target_scope=SCOPE,
    # AdamW, 전문가마다 새로 만든다. 학습률 1e-4 (alpha = rank 기준 ai-toolkit·DiffSynth·finetrainers 예시)
    learning_rate=1e-4, betas=(0.9, 0.999), weight_decay=1e-4, eps=1e-8,
    lr_scheduler="constant_with_warmup", warmup_steps=50,
    # 학습 길이. A14B 는 저노이즈 몫(low_steps_share) → 고노이즈 나머지 순서 [제안, 실무자 경험 70~80%]
    max_train_steps=2000, low_steps_share=0.75, high_uses_images=False,
    max_grad_norm=1.0,
    # 노이즈 단계: logit-normal(0,1) → shift → 전문가 구간 안으로 거절 샘플링 (musubi preserve_distribution_shape)
    #   shift_low·shift_high 는 A14B 전문가별, shift 는 전문가가 하나인 계열 [제안, musubi 토론 #455·예시]
    logit_mean=0.0, logit_std=1.0, shift_low=1.0, shift_high=5.0, shift=3.0,
    # 데이터: 이미지·클립이 다 있을 때 이미지를 뽑을 확률, 캡션 비우기 확률, 움직임이 이보다 작은 클립 제외
    image_share=0.5, caption_dropout=0.0, min_motion=0.0, min_frames=17,
    # 버킷: 토큰 = 잠재 프레임 × (H/TOKEN_DIV) × (W/TOKEN_DIV). 14,040 = 33프레임 480x832 [제안, 파일럿 시작점]
    frame_options=(1, 17, 33, 49, 81), token_budget=14040, max_area=480 * 832,
    aspects=(480 / 832, 832 / 480, 2 / 3, 3 / 2, 1.0, 704 / 1280, 1280 / 704), allow_upscale=1.0,
    window_mode="uniform", windows_per_segment=3, analysis_side=96, ti2v_i2v_share=0.5,
    # 검증 손실용으로 학습에서 뺄 파일 수. 이미지·클립 종류마다 10개당 1개까지만 빼서, 종류마다 학습 파일이 남는다
    holdout_files=2,
    # 그림체 보존 학습(DOP, v2 노트북과 같은 방식). 기본 끔, 켜면 해당 단계마다 디노이저를 3번 계산
    preservation="off", preservation_weight=1.0, preservation_every=2,
    # 정밀도·메모리. offload_sampling: 미리보기 생성 때 두 전문가를 번갈아 GPU 에 올림
    mixed_precision="bf16", gradient_checkpointing=True, offload_sampling=True,
    # 저장·검증
    save_every=250, validate_every=250, val_loss_items=4,
    preview_prompts=2, preview_frames=17, preview_steps=20, final_frames=33, final_steps=0,   # 0 = 공식 step 수
    use_dino=True, use_ccip=True, max_nan_steps=10,
)
# 계열별 값 (여기 없는 값은 공통값). TI2V-5B 는 24fps, 32 배수, 공식 해상도 1280x704 면적까지.
#   토큰 예산 12,090 = 121프레임 480p 또는 49프레임 720p (긴 클립은 해상도를 낮춰 프레임을 지킨다)
FAMILY_FIXED = {
    "wan22_ti2v_5b": dict(frame_options=(1, 25, 49, 73, 97, 121), token_budget=12090, max_area=704 * 1280,
                          min_frames=25, preview_frames=25, final_frames=49),
}

def parse_overrides(text):
    """FIXED_OVERRIDES: JSON 문자열 또는 JSON 파일 경로. FIXED 에 있는 키만, 같은 종류의 값만 받는다."""
    if not text.strip():
        return {}
    p = Path(text).expanduser()
    raw = json.loads(p.read_text(encoding="utf-8-sig") if p.suffix == ".json" and p.is_file() else text)
    if isinstance(raw, dict) and "runs" in raw:          # next_run.json 을 통째로 준 경우
        raw = raw["runs"][0].get("fixed_overrides", {})
    if not isinstance(raw, dict):
        raise ValueError("FIXED_OVERRIDES 는 {키: 값} 이어야 합니다")
    out = {}
    for k, v in raw.items():
        if k not in COMMON_FIXED:
            raise ValueError(f"FIXED_OVERRIDES: 모르는 키 '{k}'")
        d = COMMON_FIXED[k]
        if isinstance(d, tuple):
            v = tuple(v)
        elif isinstance(d, bool) or isinstance(v, bool):
            if not isinstance(v, bool) or not isinstance(d, bool):
                raise ValueError(f"FIXED_OVERRIDES: '{k}' 값의 종류가 다릅니다 ({v!r})")
        elif isinstance(d, (int, float)):
            if not isinstance(v, (int, float)):
                raise ValueError(f"FIXED_OVERRIDES: '{k}' 는 숫자여야 합니다 ({v!r})")
            v = type(d)(v) if isinstance(d, float) or float(v).is_integer() else v
            # 0 이 뜻 있는 값만 0 을 받는다: 시드 0, 워밍업 없음, 검증용 파일 없음, 공식 step 수, 첫 NaN 에서 멈춤
            _lo = 0 if k in ("seed", "warmup_steps", "holdout_files", "final_steps", "max_nan_steps") else 1
            if isinstance(d, int) and (not isinstance(v, int) or v < _lo):
                raise ValueError(f"FIXED_OVERRIDES: '{k}' 는 {_lo} 이상의 정수여야 합니다 ({v!r})")
        elif not isinstance(v, type(d)):
            raise ValueError(f"FIXED_OVERRIDES: '{k}' 값의 종류가 다릅니다 ({v!r})")
        out[k] = v
    if "rank" in out:
        out["alpha"] = out["rank"]                       # 랭크를 바꾸면 알파도 같이
    return out

OVERRIDES = parse_overrides(USER["FIXED_OVERRIDES"])
FIXED = MappingProxyType({**COMMON_FIXED, **FAMILY_FIXED.get(FAMILY, {}), **OVERRIDES})
if FIXED["alpha"] != FIXED["rank"]:
    raise ValueError(f"alpha({FIXED['alpha']}) 와 rank({FIXED['rank']}) 가 다릅니다. 둘을 같게 둡니다.")

TEST_DEVICE = os.environ.get("LORA_NB_TEST_DEVICE", "cpu")   # 시험 실행 장치: cpu(fp32) 또는 cuda(bf16)
if TEST_FAMILY or SMOKE:   # 시험 실행: 같은 코드 경로를 아주 작은 값으로
    _mult = 32 if FAMILY == "wan22_ti2v_5b" else 16
    # 6단계: A14B 는 저노이즈 4 + 고노이즈 2 (워밍업 1단계의 학습률은 0 이라 전문가마다 2단계 이상 필요)
    FIXED = MappingProxyType({**FIXED, "max_train_steps": 6, "warmup_steps": 1, "save_every": 2, "validate_every": 2,
                              "val_loss_items": 2, "frame_options": (1, 5, 9), "min_frames": 5,
                              "max_area": (2 * _mult) ** 2, "token_budget": 3 * 4, "analysis_side": 48,
                              "preview_prompts": 1, "preview_frames": 5, "preview_steps": 2, "final_frames": 5,
                              "final_steps": 2, "windows_per_segment": 2,
                              "mixed_precision": "bf16" if TEST_DEVICE == "cuda" else "fp32"})
    if TEST_FAMILY:   # 시험 모드는 DINO·CCIP 를 받지 않는다 (작은 T5 는 받음). SMOKE 는 실제처럼 켠다 (없으면 이유를 보여 주고 끔)
        FIXED = MappingProxyType({**FIXED, "use_dino": False, "use_ccip": False})

def model_notes(family, path):
    """계열·모델 경로 이름으로 알려 주는 주의 사항."""
    p, notes = str(path).lower(), []
    if family == "wan22_i2v":
        notes.append("I2V 는 클립이 있어야 학습합니다 (이미지만 있으면 0-3 에서 멈춥니다).")
    if family == "wan22_ti2v_5b":
        notes.append("TI2V-5B 로 만든 LoRA 는 A14B 에 쓸 수 없습니다 (파일럿용).")
    if family == "wan21" and "1.3b" in p:
        notes.append("Wan2.1 1.3B 는 코드 점검용입니다. 품질 비교에는 14B 이상을 씁니다.")
    if "anisora" in p:
        notes.append("Index-AniSora 는 Apache-2.0 에 추가 조건(배상·하위 사용자 구속)이 있어 법무 확인이 필요합니다.")
    if "turbo" in p or "distill" in p or "lightx2v" in p:
        notes.append("증류(적은 step) 모델로 보입니다. 증류하지 않은 원본으로 학습하고 LoRA 는 증류판에서 씁니다.")
    return notes

OUTPUT_PATH = Path(USER["OUTPUT_DIR"]).expanduser()
DATASET_PATH = Path(USER["DATASET_DIR"]).expanduser()
if not TEST_FAMILY:
    if not DATASET_PATH.is_dir():
        raise FileNotFoundError(f"DATASET_DIR 폴더가 없습니다: {DATASET_PATH.resolve()}\n  이미지·클립과 캡션(.txt)이 든 폴더의 "
                                "경로를 적습니다 (상대 경로는 이 노트북 폴더 기준, ~ 가능).")
    # 저장된 LoRA·결과가 있는 폴더는 덮어쓰지 않는다. 첫 저장 전에 멈춘 실행의 폴더는 이어서 쓴다.
    _run_dir = OUTPUT_PATH / USER["RUN_NAME"]
    _done = sorted(_run_dir.glob("checkpoint-*/*.safetensors")) + sorted(_run_dir.glob("summary.json"))
    if _done:
        raise FileExistsError(f"{_run_dir.resolve()} 에 이전 실행의 결과({_done[0].name} 등)가 있습니다. 덮어쓰지 않도록 RUN_NAME 을 "
                              "바꿉니다 (예: _v1 → _v2). 지워도 되는 결과면 폴더를 지운 뒤 다시 실행합니다.")
    if _run_dir.is_dir() and any(_run_dir.iterdir()):
        print(f"참고: {_run_dir} 에 첫 저장 전에 멈춘 실행의 파일이 있습니다. 같은 폴더에 이어서 씁니다.")

# 검증 프롬프트: 움직임이 있는 기본 4개, 또는 VALIDATION_FILE(JSON)
_TRIG = re.compile(r"(?<!\w)" + re.escape(USER["TRIGGER"]) + r"(?![A-Za-z0-9_])")
_T = USER["TRIGGER"]
VALIDATION_PROMPTS = [
    f"{_T}, medium shot, static camera, she turns from the side to face the camera, then smiles, plain light gray background, clean anime illustration",
    f"{_T}, full body, static camera, she walks forward along a forest path, a light breeze moves the leaves, clean anime illustration",
    f"{_T}, close-up, static camera, she blinks and tilts her head, soft daylight, clean anime illustration",
    f"{_T}, full body, camera slowly pans right, she raises one arm and waves, summer meadow, clean anime illustration",
]
VALIDATION_NEGATIVE = None          # None = 어댑터의 공식 Wan 부정 프롬프트
VALIDATION_SEEDS, VALIDATION_IMAGES = None, []
if USER["VALIDATION_FILE"]:
    _vf = Path(USER["VALIDATION_FILE"]).expanduser()
    if not _vf.is_file():
        raise FileNotFoundError(f"VALIDATION_FILE 이 없습니다: {_vf}")
    _v = json.loads(_vf.read_text(encoding="utf-8-sig"))
    _p = _v.get("prompts") if isinstance(_v, dict) else None
    if not isinstance(_p, list) or not _p or not all(isinstance(x, str) and x.strip() for x in _p):
        raise ValueError(f"{_vf.name}: prompts 는 비어 있지 않은 문자열 목록이어야 합니다")
    VALIDATION_PROMPTS = [x.strip() if _TRIG.search(x) else f"{_T}, {x.strip()}" for x in _p]
    VALIDATION_NEGATIVE = _v.get("negative")
    VALIDATION_SEEDS = _v.get("seeds")
    VALIDATION_IMAGES = [str((_vf.parent / x) if not Path(x).is_absolute() else x) for x in _v.get("images", [])]
    for _im in VALIDATION_IMAGES:
        if not Path(_im).is_file():
            raise FileNotFoundError(f"검증 시작 이미지가 없습니다: {_im}")
if VALIDATION_SEEDS is None:
    VALIDATION_SEEDS = [1001 + i for i in range(len(VALIDATION_PROMPTS))]
if not isinstance(VALIDATION_SEEDS, list) or len(VALIDATION_SEEDS) != len(VALIDATION_PROMPTS) \
        or not all(isinstance(s, int) and not isinstance(s, bool) for s in VALIDATION_SEEDS):
    raise ValueError(f"검증 seeds 는 prompts 와 개수가 같은 정수 목록이어야 합니다 (prompts {len(VALIDATION_PROMPTS)}개)")
if VALIDATION_NEGATIVE is not None and not isinstance(VALIDATION_NEGATIVE, str):
    raise ValueError("검증 negative 는 문자열이어야 합니다")
FIXED_WORDS = [w.strip() for w in USER["CAPTION_FIXED_WORDS"].split(",") if w.strip()]

print(f"모델 계열: {FAMILY} ({WAN_MODELS.get(FAMILY, '?')})"
      + (" ← MODEL_PATH 에서 자동 인식" if not TEST_FAMILY and USER["MODEL_FAMILY"].strip() in ("", "auto") else ""))
if not TEST_FAMILY:
    for _note in model_notes(FAMILY, USER["MODEL_PATH"]):
        print("주의:", _note)
print(json.dumps(dict(USER), ensure_ascii=False, indent=1))
print("계열별 값:", FAMILY_FIXED.get(FAMILY) or "없음 (공통값)", "| 덮어쓴 값:", OVERRIDES or "없음")
if SCOPE != USER["TARGET_SCOPE"].strip():
    print(f"TARGET_SCOPE {USER['TARGET_SCOPE']} → {SCOPE} (영상 모델에서 같은 범위)")
if FIXED["target_scope"] != SCOPE:
    print(f"참고: FIXED_OVERRIDES 의 target_scope={FIXED['target_scope']} 를 씁니다 (TARGET_SCOPE={SCOPE} 대신)")
print(f"랭크 {FIXED['rank']} | 학습률 {FIXED['learning_rate']} | 단계 {FIXED['max_train_steps']} | 범위 {FIXED['target_scope']}"
      f" | 토큰 예산 {FIXED['token_budget']} | 검증 프롬프트 {len(VALIDATION_PROMPTS)}개")

### 0-3. 영상 도구와 모델 계열별 어댑터 (수정하지 않는다)

- `lora_video_tools/`: Wan 공통 코드(`wan_common`), 영상 읽기·분석·전처리(`video_io`), 검증 지표(`video_metrics`), 보완 규칙(`remedy`).
- `lora_video_adapters/`: 계열마다 같은 함수 이름(`encode_prompts`, `encode_video`, `forward_loss`, `save_lora`, `generate`)을 가진 어댑터. 선택한 계열 하나만 불러온다.
- Wan 학습 공식 스크립트는 diffusers 에 없어서, 각 어댑터는 diffusers v0.42.1 파이프라인의 계산과 대조했고 학습 방식은 musubi-tuner·DiffSynth-Studio·ai-toolkit 과 맞췄다. 작은 무작위 모델로 CPU·GPU 에서 처음부터 끝까지 시험했고, 실제 가중치로는 아직 학습해 보지 않았다.
- LoRA 는 ComfyUI 가 읽는 Wan 원래 이름(`diffusion_model.blocks.N.self_attn.q.lora_A.weight` 등)으로 저장한다. diffusers 도 같은 파일을 그대로 불러온다.

In [ ]:
os.makedirs("lora_video_tools", exist_ok=True)
os.makedirs("lora_video_adapters", exist_ok=True)

In [ ]:
%%writefile lora_video_tools/wan_common.py
"""Shared Wan 2.1 / 2.2 code for the video LoRA adapters (diffusers 0.42.1, peft 0.21.2).

diffusers v0.42.1 ships no Wan training script, so this mirrors the inference code the LoRA must match:
pipelines/wan/pipeline_wan.py (WanPipeline), pipeline_wan_i2v.py (WanImageToVideoPipeline),
models/transformers/transformer_wan.py, and the original Wan-Video/Wan2.1 and Wan2.2 wan/configs.
  * text: UMT5 through pipe.encode_prompt(max_sequence_length=512) (encode_prompt defaults to 226; __call__ and the
    original text_len use 512); rows after the real token count are zeros.
  * pixels: px = 2 * (uint8 / 255) - 1 as VaeImageProcessor, shape (1, 3, F, H, W), F = 4k + 1.
  * latents: x0 = (vae.encode(px).latent_dist.mode() - latents_mean) * (1 / latents_std), i.e. (z - mean) / std written
    the way prepare_latents computes it; inverse as the pipeline decode: z = x0 / (1 / latents_std) + latents_mean.
  * flow matching: x_t = (1 - s) * x0 + s * noise, target = noise - x0 (UniPC flow_prediction: x0 = x_t - s * target);
    transformer timestep t = 1000 * s as float (the pipeline passes UniPC's int64(1000 * s)).
  * "i2v_concat" (Wan2.2 I2V-A14B): hidden_states = cat([x_t, mask4, normalized latent of cat([frame0, zeros(F-1)])]),
    the condition built exactly as WanImageToVideoPipeline.prepare_latents.
  * "ti2v" (Wan2.2 TI2V-5B, expand_timesteps): per-token timestep (mask[0, 0][:, ::2, ::2] * t).flatten(); in I2V mode
    the first latent frame is the clean x0 (timestep 0 on its tokens) and is left out of the loss.
  * A14B: `transformer` (high noise) runs while t >= boundary_ratio * 1000, `transformer_2` (low noise) below.
  * ComfyUI export: original Wan names, the inverse of diffusers _convert_non_diffusers_wan_lora_to_diffusers for
    block linears, no alpha keys (train with lora_alpha == rank -> scale 1 in ComfyUI and in diffusers).
  * reload: load_lora writes the file's fp32 values back into the loaded adapter (peft casts it to the bf16 base
    dtype), so a reload is bit-exact against the fp32 LoRA trained on a bf16 base.
  * ftfy: encode_prompts requires it (diffusers' Wan prompt cleaning; WanImageToVideoPipeline fails without it).
"""
import os
import re

import numpy as np
import torch
import torch.nn.functional as F
from diffusers import (
    AutoencoderKLWan,
    DiffusionPipeline,
    UniPCMultistepScheduler,
    WanImageToVideoPipeline,
    WanTransformer3DModel,
)
from diffusers.utils import is_ftfy_available
from safetensors.torch import save_file

TEXT_LEN = 512  # wan/configs/shared_config.py text_len
NUM_TRAIN_TIMESTEPS = 1000
TINY_T5 = "hf-internal-testing/tiny-random-t5"
# Wan2.2 wan/configs/shared_config.py sample_neg_prompt, verbatim (the Wan2.1 shared_config.py has the same string).
NEGATIVE_PROMPT = (
    "色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，"
    "丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，"
    "静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走"
)

_QKVO = (("to_q", "q"), ("to_k", "k"), ("to_v", "v"), ("to_out.0", "o"))
_ATTN = [f"{a}.{d}" for a in ("attn1", "attn2") for d, _ in _QKVO]
# peft target_modules suffixes, block linears only (musubi-tuner / DiffSynth q,k,v,o,ffn.0,ffn.2 set).
LORA_TARGETS = {
    "all": _ATTN + ["ffn.net.0.proj", "ffn.net.2"],
    "attn": _ATTN,
    "kvo": [f"{a}.{d}" for a in ("attn1", "attn2") for d in ("to_k", "to_v", "to_out.0")],  # queries untouched
}
# diffusers module inside blocks.N -> original Wan / ComfyUI module
COMFY_NAMES = {f"{a}.{d}": f"{n}.{o}" for a, n in (("attn1", "self_attn"), ("attn2", "cross_attn")) for d, o in _QKVO}
COMFY_NAMES.update({"ffn.net.0.proj": "ffn.0", "ffn.net.2": "ffn.2"})
_LORA_KEY = re.compile(r"(?:transformer\.)?blocks\.(\d+)\.(.+)\.(lora_A|lora_B)\.weight")


# ----------------------------------------------------------------------------------------------- pipelines
def tiny_pipeline(pipeline_class, dtype, *, in_channels=16, out_channels=16, z_dim=16, two_experts=False,
                  boundary_ratio=None, expand_timesteps=False):
    """Random tiny Wan pipeline. Configs from diffusers v0.42.1 tests/pipelines/wan/test_wan_22.py and
    test_wan_22_image_to_video.py (get_dummy_components); UniPC flow_prediction / flow_shift 3.0 as in those tests
    (the Wan2.1 / A14B -Diffusers repos ship 3.0, TI2V-5B ships 5.0; generate() always sets the shift).
    transformer_2 is seeded differently so the two experts differ, as the real ones do."""
    from transformers import AutoConfig, AutoTokenizer, T5EncoderModel

    torch.manual_seed(0)
    vae_kw = dict(base_dim=3, z_dim=z_dim, dim_mult=[1, 1, 1, 1], num_res_blocks=1,
                  temperal_downsample=[False, True, True])
    if z_dim == 48:  # TI2V-5B VAE: 2x2 patchify, 16x spatial
        vae_kw.update(in_channels=12, out_channels=12, is_residual=True, patch_size=2, latents_mean=[0.0] * 48,
                      latents_std=[1.0] * 48, scale_factor_spatial=16, scale_factor_temporal=4)
    vae = AutoencoderKLWan(**vae_kw)
    torch.manual_seed(0)
    scheduler = UniPCMultistepScheduler(prediction_type="flow_prediction", use_flow_sigmas=True, flow_shift=3.0)
    text_encoder = T5EncoderModel(AutoConfig.from_pretrained(TINY_T5)).eval()  # eval(): T5 dropout off
    tokenizer = AutoTokenizer.from_pretrained(TINY_T5)

    def tx(seed):
        torch.manual_seed(seed)
        return WanTransformer3DModel(
            patch_size=(1, 2, 2), num_attention_heads=2, attention_head_dim=12, in_channels=in_channels,
            out_channels=out_channels, text_dim=32, freq_dim=256, ffn_dim=32, num_layers=2, cross_attn_norm=True,
            qk_norm="rms_norm_across_heads", rope_max_seq_len=32,
        )

    pipe = pipeline_class(tokenizer=tokenizer, text_encoder=text_encoder, vae=vae, scheduler=scheduler,
                          transformer=tx(0), transformer_2=tx(1) if two_experts else None,
                          boundary_ratio=boundary_ratio, expand_timesteps=expand_timesteps)
    if dtype != torch.float32:  # as load_pipeline: VAE fp32, transformer _keep_in_fp32_modules fp32, rest in dtype
        text_encoder.to(dtype=dtype)
        for m in [m for m in (pipe.transformer, pipe.transformer_2) if m is not None]:
            for name, p in m.named_parameters():
                keep = any(k in name.split(".") for k in m._keep_in_fp32_modules or [])
                p.data = p.data.to(torch.float32 if keep else dtype)
    return pipe


def check_variant(family, where, index, transformers, vae, *, two_experts, in_channels, z_dim, expand_timesteps):
    """Raise ValueError when config dicts (model_index / transformer(_2) / vae) are not the variant `family` trains."""
    bad = []
    if index.get("_class_name", "WanPipeline") not in ("WanPipeline", "WanImageToVideoPipeline"):
        bad.append(f"pipeline is {index.get('_class_name')}, not a Wan pipeline")
    br = index.get("boundary_ratio")
    has1, has2 = transformers.get("transformer") is not None, transformers.get("transformer_2") is not None
    if two_experts and (br is None or not (has1 and has2)):
        bad.append(f"needs two experts (transformer + transformer_2) and boundary_ratio; found transformer_2={has2}, "
                   f"boundary_ratio={br} (a single-expert Wan2.1 / TI2V-5B folder?)")
    if not two_experts and (br is not None or has2 or not has1):
        bad.append(f"needs one transformer and no boundary_ratio; found transformer_2={has2}, boundary_ratio={br} "
                   "(a two-expert Wan2.2 A14B folder?)")
    enc = (index.get("image_encoder") or [None, None])[-1]  # model_index entry / pipe.config: (library, class)
    if enc is not None:
        bad.append(f"image_encoder={enc} (Wan2.1 I2V with CLIP image tokens, Wan2.2 has none)")
    for name, c in transformers.items():
        if c is None:
            continue
        if c.get("_class_name", "WanTransformer3DModel") != "WanTransformer3DModel":
            bad.append(f"{name} is {c.get('_class_name')}")
        if c.get("in_channels") != in_channels:
            bad.append(f"{name}.in_channels={c.get('in_channels')}, this family needs {in_channels} "
                       "(16 T2V / Wan2.1, 36 I2V, 48 TI2V-5B)")
        if c.get("image_dim") is not None:
            bad.append(f"{name}.image_dim={c.get('image_dim')} (Wan2.1 I2V with CLIP image tokens)")
    if vae.get("z_dim") != z_dim:
        bad.append(f"vae.z_dim={vae.get('z_dim')}, this family needs {z_dim} (16 Wan2.1 VAE, 48 Wan2.2 TI2V-5B VAE)")
    if bool(index.get("expand_timesteps") or False) != expand_timesteps:
        bad.append(f"expand_timesteps={index.get('expand_timesteps')}, this family needs {expand_timesteps}")
    if bad:
        raise ValueError(f"{family}: {where} is the wrong Wan variant: " + "; ".join(bad))


def check_pipeline(pipe, family, **spec):
    tx = {n: (dict(getattr(pipe, n).config) if getattr(pipe, n, None) is not None else None)
          for n in ("transformer", "transformer_2")}
    check_variant(family, type(pipe).__name__, dict(pipe.config), tx, dict(pipe.vae.config), **spec)


def read_configs(model_path):
    """model_index.json, transformer(_2)/config.json, vae/config.json only (cheap even for a hub repo id)."""
    index = dict(DiffusionPipeline.load_config(model_path))
    present = {n for n in ("transformer", "transformer_2", "vae") if (index.get(n) or [None])[0]}
    tx = {n: (dict(WanTransformer3DModel.load_config(model_path, subfolder=n)) if n in present else None)
          for n in ("transformer", "transformer_2")}
    vae = dict(AutoencoderKLWan.load_config(model_path, subfolder="vae")) if "vae" in present else {}
    return index, tx, vae


def load_pipeline(model_path, dtype, pipeline_class, family, **spec):
    """Variant check on the JSON configs first (before 2 x 57 GB of A14B weights), then from_pretrained with the VAE in
    fp32 (as the diffusers Wan examples load it) and everything else in `dtype`, then the same check on the objects."""
    check_variant(family, model_path, *read_configs(model_path), **spec)
    pipe = pipeline_class.from_pretrained(model_path, dtype={"vae": torch.float32, "default": dtype})
    check_pipeline(pipe, family, **spec)
    return pipe


def sigma_range(pipe, expert, experts):
    if expert not in experts:
        raise ValueError(f"unknown expert {expert!r}, expected one of {experts}")
    if len(experts) == 1:
        return (0.0, 1.0)
    b = pipe.config.boundary_ratio
    if b is None:
        raise ValueError("two-expert family but pipe.config.boundary_ratio is None")
    return (float(b), 1.0) if expert == experts[0] else (0.0, float(b))


# ----------------------------------------------------------------------------------------------- encoding
def require_ftfy():
    """Raise ImportError unless diffusers found ftfy. diffusers 0.42.1 cleans Wan prompts with ftfy.fix_text:
    WanImageToVideoPipeline calls it unconditionally (NameError without ftfy), WanPipeline silently skips it (cached
    embeddings would then differ from a run with ftfy). diffusers checks at import, so install, then restart."""
    if not is_ftfy_available():
        raise ImportError("ftfy is required for diffusers' Wan prompt cleaning (WanImageToVideoPipeline raises "
                          "NameError without it, WanPipeline silently skips ftfy.fix_text): pip install ftfy, then "
                          "restart the Python kernel")


@torch.no_grad()
def encode_prompts(pipe, prompts, device, dtype):
    require_ftfy()
    out = []
    for p in prompts:  # one at a time: (1, 512, D), zeros after the real tokens
        emb, _ = pipe.encode_prompt(prompt=[p], do_classifier_free_guidance=False, max_sequence_length=TEXT_LEN,
                                    device=device, dtype=dtype)
        out.append({"prompt_embeds": emb.cpu()})
    return out


def to_pixels(frames):
    """np.uint8 (F, H, W, 3) -> float32 (1, 3, F, H, W) in [-1, 1], bit-identical to VaeImageProcessor."""
    a = np.asarray(frames)
    if a.dtype != np.uint8 or a.ndim != 4 or a.shape[-1] != 3:
        raise ValueError(f"frames must be np.uint8 (F, H, W, 3), got {a.dtype} {a.shape}")
    px = 2.0 * torch.from_numpy(a.astype(np.float32) / 255.0) - 1.0
    return px.permute(3, 0, 1, 2).unsqueeze(0).contiguous()


def latent_stats(vae, device):
    """(latents_mean, 1 / latents_std) as (1, z, 1, 1, 1) float32, built exactly like prepare_latents does."""
    z = vae.config.z_dim
    mean = torch.tensor(vae.config.latents_mean).view(1, z, 1, 1, 1).to(device, torch.float32)
    inv_std = 1.0 / torch.tensor(vae.config.latents_std).view(1, z, 1, 1, 1).to(device, torch.float32)
    return mean, inv_std


def normalize(vae, z):
    mean, inv_std = latent_stats(vae, z.device)
    return (z.to(torch.float32) - mean) * inv_std


def denormalize(vae, x0):
    mean, inv_std = latent_stats(vae, x0.device)
    return x0.to(torch.float32) / inv_std + mean


@torch.no_grad()
def vae_mode(vae, px, device):
    """Posterior mode (retrieve_latents(..., sample_mode="argmax")) as float32."""
    return vae.encode(px.to(device=device, dtype=vae.dtype)).latent_dist.mode().to(torch.float32)


@torch.no_grad()
def i2v_condition(pipe, first_px, num_frames, device):
    """(1, 4 + z, f, h, w) float32 = cat([mask4, normalized latent of cat([frame0, zeros(F - 1)])]), exactly
    WanImageToVideoPipeline.prepare_latents for batch 1 without last_image. first_px: (1, 3, 1, H, W) in [-1, 1]."""
    vae, tf, sf = pipe.vae, pipe.vae_scale_factor_temporal, pipe.vae_scale_factor_spatial
    hh, ww = first_px.shape[-2:]
    h, w = hh // sf, ww // sf
    video = torch.cat([first_px, first_px.new_zeros(1, 3, num_frames - 1, hh, ww)], dim=2)
    lat = normalize(vae, vae_mode(vae, video, device))
    mask = torch.ones(1, 1, num_frames, h, w)
    mask[:, :, 1:] = 0
    mask = torch.cat([torch.repeat_interleave(mask[:, :, 0:1], dim=2, repeats=tf), mask[:, :, 1:]], dim=2)
    mask = mask.view(1, -1, tf, h, w).transpose(1, 2).to(lat.device)
    return torch.cat([mask, lat], dim=1)


@torch.no_grad()
def encode_video(pipe, frames, device, *, conditioning, multiple, condition=False):
    """{"x0": (1, z, f, h, w)} float32 CPU; plus {"cond"} for "i2v_concat" with condition=True ("ti2v" conditions on
    x0[:, :, :1] itself, "t2v" has no condition). The VAE runs in its own dtype on `device`."""
    px = to_pixels(frames)
    n, hh, ww = px.shape[2:]
    tf = pipe.vae_scale_factor_temporal
    if n % tf != 1 or hh % multiple or ww % multiple:
        raise ValueError(f"frames {n}x{hh}x{ww}: need F = {tf}k+1 and H, W multiples of {multiple}")
    out = {"x0": normalize(pipe.vae, vae_mode(pipe.vae, px, device)).cpu()}
    if condition and conditioning == "i2v_concat":
        out["cond"] = i2v_condition(pipe, px[:, :, :1], n, device).cpu()
    return out


# ----------------------------------------------------------------------------------------------- loss
def flow_loss(denoiser, x0, noise, sigma, text, *, conditioning, mode, cond=None):
    """One denoiser call; fp32 MSE against noise - x0 (see module docstring for the three conditioning types)."""
    ok = {"t2v": ("t2v",), "i2v_concat": ("i2v",), "ti2v": ("t2v", "i2v")}
    if mode not in ok.get(conditioning, ()):
        raise ValueError(f"mode {mode!r} is not available for conditioning {conditioning!r}")
    dev = x0.device
    wdtype = getattr(denoiser, "dtype", None) or next(denoiser.parameters()).dtype
    x0 = x0.to(torch.float32)
    noise = noise.to(device=dev, dtype=torch.float32)
    b, _, f, h, w = x0.shape
    if mode == "i2v" and f < 2:
        raise ValueError("mode 'i2v' needs a clip with at least 2 latent frames (F >= 5)")
    sig = sigma.to(device=dev, dtype=torch.float32).reshape(-1)
    s = sig.view(-1, 1, 1, 1, 1)
    x_t = (1.0 - s) * x0 + s * noise
    target = noise - x0
    t = (sig * NUM_TRAIN_TIMESTEPS).expand(b)  # float, same unit as the pipeline's int64 UniPC timesteps
    first = 0
    if conditioning == "i2v_concat":
        if cond is None or "cond" not in cond:
            raise ValueError("i2v_concat needs cond from encode_video(..., condition=True)")
        c = cond["cond"].to(device=dev, dtype=torch.float32)
        if c.shape[2:] != x0.shape[2:]:
            raise ValueError(f"cond {tuple(c.shape)} does not match x0 {tuple(x0.shape)}")
        inp, timestep = torch.cat([x_t, c.expand(b, -1, -1, -1, -1)], dim=1), t
    elif conditioning == "ti2v":
        pt, ph, pw = getattr(getattr(denoiser, "config", None), "patch_size", (1, 2, 2))
        m = torch.ones(1, 1, f, h, w, device=dev)
        if mode == "i2v":
            m[:, :, 0] = 0  # clean first latent frame
            first = 1
        inp = (1 - m) * x0 + m * x_t
        timestep = m[0, 0][::pt, ::ph, ::pw].flatten()[None] * t[:, None]  # (b, tokens), 0 on the clean frame
    else:
        inp, timestep = x_t, t
    emb = text["prompt_embeds"].to(device=dev, dtype=wdtype)
    if emb.shape[0] != b:
        emb = emb.expand(b, -1, -1)
    pred = denoiser(hidden_states=inp.to(wdtype), timestep=timestep, encoder_hidden_states=emb, return_dict=False)[0]
    return F.mse_loss(pred[:, :, first:].float(), target[:, :, first:])


# ----------------------------------------------------------------------------------------------- LoRA files
def lora_to_comfy(peft_state_dict):
    """peft names ("blocks.0.attn1.to_q.lora_A.weight", optional "transformer.") -> original Wan / ComfyUI names
    ("diffusion_model.blocks.0.self_attn.q.lora_A.weight"). Raises on anything else; writes no alpha keys."""
    out, parts = {}, {}
    for k, v in peft_state_dict.items():
        m = _LORA_KEY.fullmatch(k)
        if not m or m.group(2) not in COMFY_NAMES:
            raise ValueError(f"lora_to_comfy: cannot map {k!r}; only lora_A/lora_B weights of the block linears "
                             f"{sorted(COMFY_NAMES)}")
        mod = f"diffusion_model.blocks.{m.group(1)}.{COMFY_NAMES[m.group(2)]}"
        key = f"{mod}.{m.group(3)}.weight"
        if key in out:
            raise ValueError(f"lora_to_comfy: {k!r} maps to {key!r} twice")
        out[key] = v
        parts.setdefault(mod, set()).add(m.group(3))
    unpaired = [mod for mod, ab in parts.items() if ab != {"lora_A", "lora_B"}]
    if not out or unpaired:
        raise ValueError(f"lora_to_comfy: empty state dict or lora_A/lora_B missing for {unpaired[:3]}")
    return out


def save_lora(peft_state_dict, path, metadata):
    """lora_to_comfy(...) as fp32 contiguous safetensors (one file per expert); metadata values become strings."""
    sd = {k: v.detach().to(device="cpu", dtype=torch.float32).contiguous()
          for k, v in lora_to_comfy(peft_state_dict).items()}
    meta = {"format": "pt", **{str(k): str(v) for k, v in (metadata or {}).items()}}
    if "lora_adapter_metadata" in meta:  # diffusers would read it as the LoraConfig and override alpha == rank
        raise ValueError("metadata key 'lora_adapter_metadata' is reserved by diffusers")
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    save_file(sd, path, metadata=meta)
    return path


def load_lora(pipe, path, expert, adapter_name):
    """pipe.load_lora_weights(path, adapter_name=..., load_into_transformer_2=(expert == "transformer_2")), then the
    file's exact fp32 values written back into the new adapter. peft casts a loaded adapter to its base layer's dtype
    (bf16 on the GPU pipeline) while training keeps the LoRA fp32 on a bf16 base; without the write-back the reload is
    a bf16-rounded copy (outputs off by a few % of the LoRA effect), with it the reload is bit-exact."""
    if expert not in ("transformer", "transformer_2") or getattr(pipe, expert, None) is None:
        raise ValueError(f"the pipeline has no expert {expert!r}")
    den = getattr(pipe, expert)
    before = set(getattr(den, "peft_config", None) or {})
    pipe.load_lora_weights(path, adapter_name=adapter_name, load_into_transformer_2=(expert == "transformer_2"))
    name = adapter_name
    if name is None:  # diffusers named it (default_<n>)
        (name,) = set(den.peft_config) - before
    # The values load_lora_weights loaded: lora_state_dict applies the same conversion (ComfyUI -> diffusers names),
    # and load_lora_adapter keeps only the "transformer." keys, prefix removed, for either expert
    # (load_into_transformer_2 loads the same "transformer." keys). peft names the parameter
    # <module>.lora_A.<adapter>.weight; matching the adapter right before ".weight" keeps adapter names that look like
    # module names ("attn1", "0") from matching elsewhere.
    sd = pipe.lora_state_dict(path)
    sd = sd[0] if isinstance(sd, tuple) else sd
    vals = {k[len("transformer."):]: v for k, v in sd.items() if k.startswith("transformer.")}
    pat = re.compile(rf"(.+)\.(lora_A|lora_B)\.{re.escape(name)}\.(weight|bias)")
    done, miss = 0, []
    for n, p in den.named_parameters():
        m = pat.fullmatch(n)
        if m is None:
            continue
        v = vals.get(f"{m.group(1)}.{m.group(2)}.{m.group(3)}")
        if v is None or v.shape != p.shape:
            miss.append(n)
            continue
        p.data = v.to(device=p.device, dtype=torch.float32, copy=True)  # own memory, never the file's buffer
        done += 1
    if miss or not done:
        raise ValueError(f"load_lora: {path} has no matching value for {miss[:3] or 'any parameter'} of adapter "
                         f"{name!r} in {expert}")


# ----------------------------------------------------------------------------------------------- sampling
def as_i2v(pipe):
    """The same modules as a WanImageToVideoPipeline (TI2V-5B ships as WanPipeline). Built directly because
    from_pipe() casts every shared module to its default dtype (float32)."""
    if isinstance(pipe, WanImageToVideoPipeline):
        return pipe
    return WanImageToVideoPipeline(**pipe.components, boundary_ratio=pipe.config.boundary_ratio,
                                   expand_timesteps=pipe.config.expand_timesteps)


def with_flow_shift(scheduler, shift):
    """Copy of a flow-sigma scheduler (UniPC as in the -Diffusers repos) with flow_shift = shift."""
    c = scheduler.config
    if not (c.get("use_flow_sigmas") and c.get("prediction_type") == "flow_prediction" and "flow_shift" in c):
        raise ValueError(f"{type(scheduler).__name__} is not a flow_prediction scheduler with use_flow_sigmas")
    return type(scheduler).from_config(c, flow_shift=float(shift))


@torch.no_grad()
def generate(pipe, text, negative, *, height, width, num_frames, generator, image, steps, shift, guidance, device,
             offload, conditioning):
    """uint8 (T, H, W, 3) through the pipeline with cached embeddings, the scheduler temporarily set to `shift`.
    guidance = (high-noise expert, low-noise expert) -> guidance_scale, guidance_scale_2 (single expert: [0] only).
    offload=True: diffusers model CPU offload on CUDA for this call, hooks removed afterwards; elsewhere a plain call.
    device=None: CUDA when offload=True and CUDA is available (the device enable_model_cpu_offload() picks by itself),
    otherwise the device the modules are on (pipe._execution_device). Without that default, offload=True with the
    modules parked on CPU would silently denoise on CPU."""
    if conditioning == "t2v" and image is not None:
        raise ValueError("text-to-video family: image must be None")
    if conditioning == "i2v_concat" and image is None:
        raise ValueError("image-to-video family: pass the first frame as a PIL image")
    run = as_i2v(pipe) if image is not None else pipe
    g_hi, g_lo = (float(g) for g in guidance)
    two = run.config.boundary_ratio is not None
    if two and g_hi <= 1.0 < g_lo:
        raise ValueError("diffusers WanPipeline runs CFG only when guidance_scale (high-noise expert) > 1")
    if g_hi > 1.0 and negative is None:
        raise ValueError("guidance > 1 needs cached negative embeddings (encode_prompts(pipe, [NEGATIVE_PROMPT]))")
    if device is None:
        device = "cuda" if offload and torch.cuda.is_available() else run._execution_device
    device = torch.device(device)
    den = run.transformer if run.transformer is not None else run.transformer_2
    kw = dict(prompt_embeds=text["prompt_embeds"].to(device, den.dtype), height=height, width=width,
              num_frames=num_frames, num_inference_steps=steps, guidance_scale=g_hi, generator=generator,
              output_type="np")
    if g_hi > 1.0:
        kw["negative_prompt_embeds"] = negative["prompt_embeds"].to(device, den.dtype)
    if two:
        kw["guidance_scale_2"] = g_lo
    if image is not None:
        kw["image"] = image
    sched, hooked = run.scheduler, offload and device.type == "cuda"
    run.scheduler = with_flow_shift(sched, shift)
    try:
        if hooked:
            run.enable_model_cpu_offload(device=device)
        frames = run(**kw).frames[0]
    finally:
        if hooked:
            run.remove_all_hooks()  # modules stay on CPU; the caller re-places them
        run.scheduler = sched
    return (np.asarray(frames) * 255).round().clip(0, 255).astype(np.uint8)

In [ ]:
%%writefile lora_video_tools/video_io.py
"""영상 LoRA 노트북의 영상 입출력·EDA·전처리 도구 (numpy, Pillow, PyAV 만 사용).

따르는 기준과 계산식:
  * 메타데이터: 헤더 값(stream.frames, average_rate)은 mkv·webm·VFR 에서 틀린다. 디코딩 없이 패킷 PTS 를 모아
    frames = 패킷 수, fps = 1/median(dt), duration = (마지막 - 처음) + median(dt), VFR = std(dt)/mean(dt) > 0.05
    (webm 은 1ms 단위 시각이라 CFR 이어도 0.01 안팎이 나온다). 재인코딩 없이 자른 mp4·mov(ffmpeg -ss … -c copy)는 편집 목록
    (edit list) 앞의 패킷을 디코더가 버리므로(is_discard) 세지 않는다. 회전·HDR 만 첫 프레임 1장을 디코딩해 읽는다.
  * 디코딩: PyAV 는 회전을 적용하지 않으므로 frame.rotation(DISPLAYMATRIX, 반시계 각도)만큼 np.rot90(k=rotation/90)
    (ffmpeg 자동 회전과 화소까지 같음). SAR≠1 이면 폭을 coded_w*SAR 로 늘리고, 알파는 흰 배경에 합성한다.
    FFmpeg 기본 vp8·vp9 디코더는 webm 의 알파(alpha_mode=1, BlockAdditional)를 버리므로 그런 클립은 libvpx 로 디코딩한다.
    움직이는 GIF·PNG(APNG)·WebP 는 확장자가 이미지여도 클립이다 (is_animated_image, 노트북도 같은 판정으로 나눈다).
    imageio-ffmpeg 는 VFR·GIF 를 CFR 로 복제해 내보내므로(30장 → 60장) 쓰지 않는다.
  * 이미지: 이미지 노트북과 같게 EXIF 회전, 16비트·실수 → 8비트, 투명 → 흰 배경.
  * 검은 프레임 = ffmpeg blackdetect (휘도 ≤ 0.10*255 인 화소 비율 ≥ 0.98). 멈춤 = ffmpeg freezedetect: 멈춤 후보의 첫 프레임과의
    평균 |dY|/255 < 0.001 이 0.25초 이상 (첫 프레임과 비교하므로 느린 움직임은 차이가 쌓여 멈춤이 풀린다). 시각을 주면 VFR·GIF 에서
    한 그림이 화면에 떠 있는 시간으로 잰다. 컷 = PySceneDetect ContentDetector 와 같은 식: HSV 평균 차 ≥ 27 (단 색상은 PIL 0~255
    척도라 OpenCV 0~179 보다 색상 차가 약 1.42배 크게 들어가 조금 더 민감하다), 또는 차 ≥ 15 이면서 (Y 히스토그램 상관 ≤ 0.80
    이거나 앞뒤 4전환 평균의 3배 이상). 히스토그램 기준에 15 조건을 붙인 것은 평평한 셀화·GIF 팔레트의 오검출을 막기 위함.
    셀 애니의 2·3프레임 홀드는 결함이 아니므로 hold_ratio·drawing_fps = fps*(1-hold_ratio) 로 보고만 한다.
  * 움직임 = 긴 변 96px 휘도에서 8x8 블록 정합(±4px). 블록 변위의 중앙값 = 카메라, 상위 5% = 피사체 (VBench dynamic degree).
    이웃 프레임(빠른 움직임), 1/16초(30·60fps 에서 이웃은 1px 미만, 0.5초는 ±4px 를 넘는 중간 속도), 0.5초(1px 미만의 느린
    움직임) 떨어진 프레임을 비교해 큰 값을 쓴다. 단위는 '초당 화면 긴 변의 비율'(0.5 = 1초에 긴 변의 절반을 움직임)이라
    해상도·fps 가 달라도 비교할 수 있다 (16·24·30·60fps 화면 전체 팬에서 초당 긴 변의 0.8 까지는 참값의 약 0.8~1.7배,
    그보다 빠르면 이웃 비교의 ±4px 한계로 작게 읽는다: 16fps 에서 최대 약 0.67).
  * 중복 = 같은 상대 위치 8프레임의 pHash (imagehash.phash 와 같은 계산) 평균 해밍 거리, DUP_BITS(8) 이하면 중복 의심.
  * 리샘플 = ffmpeg fps 필터(round=near)와 같다: 격자 시각 t_k = k/target_fps 에 화면에 떠 있는 원본 프레임 (시작 시각을
    출력 단위로 반올림한 값이 k 이하인 마지막 프레임). 시각 기준이라 VFR·GIF 의 긴 프레임은 떠 있는 시간만큼 반복된다.
  * Wan VAE: 잠재 프레임 = 1 + (F-1)//4 이고 4k+1 을 넘는 프레임은 경고 없이 버려진다 → round_4k1 로 미리 자른다.
  * 버킷: 토큰 = (1+(F-1)//4) * (H/TOKEN_DIV) * (W/TOKEN_DIV) ≤ 예산, H*W ≤ max_area, 업스케일 없음.
    표준 비율(832/480, 1280/704)을 그대로 넣으면 정확히 480x832·704x1280 이 나온다 (16/9 를 넣으면 464x832).
  * 저장: libx264 yuv420p 는 짝수 크기만 받으므로 가장자리를 복제해 맞춘다. Pillow 는 GIF·WebP 에서 같은 프레임을 합치고
    GIF 는 10ms 단위라 재생 속도가 바뀌므로, 지표는 메모리 배열로 재고 미리보기는 mp4·WebP 로 저장한다.
  * 시험: make_test_dataset 이 알려진 문제(회전·VFR·홀수 크기·검은/멈춘 구간·컷·중복·정지·편집 목록·알파 webm·움직이는 WebP·
    VFR 긴 정지)를 넣은 합성 데이터와 기대값을 만들고, check_test_expectations 가 노트북 EDA 결과와 대조한다.
    플랫폼·FFmpeg 빌드 차이를 실행할 때 잡기 위함.
"""
import math
import os
import re
import warnings
from fractions import Fraction
from functools import lru_cache
from pathlib import Path

import av
import numpy as np
from av.video.format import VideoFormat
from PIL import Image, ImageDraw, ImageFont, ImageOps

IMG_EXT = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tif", ".tiff"}
VID_EXT = {".mp4", ".mov", ".m4v", ".mkv", ".webm", ".avi", ".gif", ".mpg", ".mpeg", ".ts", ".mts", ".m2ts",
           ".wmv", ".flv", ".3gp"}

VFR_CV = 0.05                       # 프레임 간격의 변동계수가 이보다 크면 VFR
BLACK_PIX, BLACK_PIC = 0.10, 0.98   # blackdetect pix_th, pic_th
FREEZE_MAD = 1e-3                   # freezedetect noise (-60dB)
CUT_CONTENT, CUT_HIST, CUT_RATIO, CUT_MIN = 27.0, 0.20, 3.0, 15.0
STATIC_MOTION = 0.05                # subject_motion(초당 화면 긴 변의 비율)이 이보다 작으면 '거의 정지' (초당 긴 변의 5%)
MOTION_SPAN = 0.5                   # 느린 움직임을 잴 프레임 간격(초): 96px 에서 0.5초에 1px = 초당 긴 변의 2%
MOTION_MID = 1 / 16                 # 중간 간격(초): 30·60fps 에서 이웃 비교(1px 미만은 0)와 0.5초 비교(±4px 에서 포화) 사이
CAMERA_MOVING = 0.10                # camera_motion(초당 화면 긴 변의 비율)이 이보다 크면 'static camera' 캡션과 모순
DUP_BITS = 8.0                      # 서명 거리(64비트 중)가 이 이하이면 중복 의심 (자동 삭제는 하지 않음)
BLUR_SIDE = 256                     # 흐림(라플라시안 분산)을 재는 긴 변
CAPTION_WORDS = (6, 120)
ASPECT_TOL = math.log(1.1)          # pick_bucket: 가장 가까운 비율보다 10% 안쪽으로 어긋난 버킷은 같은 비율로 본다


# ------------------------------------------------------------------ 읽기
def _has_alpha(pix_fmt):
    """화소 형식에 알파가 있는지 (bgra·rgba·yuva420p·ya8·pal8). 이름에 'a' 가 있는지로 보면 gray 도 걸린다."""
    try:
        f = VideoFormat(pix_fmt)
    except Exception:
        return False
    return bool(f.has_palette or any(c.is_alpha for c in f.components))


def _sar(vs):
    s = vs.sample_aspect_ratio or vs.codec_context.sample_aspect_ratio
    return float(s) if s and float(s) > 0 else 1.0


@lru_cache(None)
def _codec_ok(name, mode="r"):
    """이 PyAV(FFmpeg) 빌드에 name 디코더(r)·인코더(w)가 있는지"""
    try:
        av.Codec(name, mode)
        return True
    except Exception:
        return False


def _alpha_decoder(vs):
    """알파가 있는 VP8·VP9 webm(alpha_mode=1)이면 알파째 디코딩할 libvpx 디코더 이름, 아니면 None.
    FFmpeg 기본 vp8·vp9 디코더는 BlockAdditional 에 담긴 알파를 버려 투명한 곳에 숨은 색이 나온다.
    libvpx 디코더가 없는 빌드면 '' (부르는 쪽이 경고하고 알파 없이 읽는다)."""
    meta = {str(k).lower(): str(v).strip() for k, v in (vs.metadata or {}).items()}
    name = vs.codec_context.name
    if meta.get("alpha_mode") != "1" or name not in ("vp8", "vp9", "libvpx", "libvpx-vp9"):
        return None
    lib = "libvpx-vp9" if name in ("vp9", "libvpx-vp9") else "libvpx"
    return lib if _codec_ok(lib, "r") else ""


def _warn_no_alpha(p):
    warnings.warn(f"{Path(p).name}: 알파가 있는 VP8·VP9 webm 인데 이 PyAV(FFmpeg)에 libvpx 디코더가 없어 알파 없이 읽음 "
                  "(투명한 곳이 흰색이 아니라 숨은 색으로 나옴)", stacklevel=3)


def _stream_rotation(vs):
    """스트림 표시 행렬(display_matrix)의 회전, 반시계 각도. 따로 연 디코더(libvpx)의 프레임에는 이 값이 붙지 않는다."""
    try:
        m = np.frombuffer(vs.codec_context.coded_side_data["display_matrix"], np.int32)[:5] / 65536.0
        return int(round(-math.degrees(math.atan2(m[1] / math.hypot(m[1], m[4]), m[0] / math.hypot(m[0], m[3])))))
    except Exception:
        return 0


def is_animated_image(path):
    """움직이는 GIF·PNG(APNG)·WebP 이면 True (확장자가 아니라 내용으로 본다). probe·read_frames 는 이런 파일을 확장자가
    이미지여도 클립으로 읽는다: Pillow 로 열면 첫 프레임만 나와 정지 이미지로 학습되기 때문이다 (ComfyUI 'Save Animated
    WEBP'·save_webp 출력이 이 형식). 열 수 없는 파일, 1프레임 파일, 여러 쪽 TIFF·MPO(카메라 JPEG)는 False."""
    try:
        with Image.open(path) as im:
            return im.format in ("GIF", "PNG", "WEBP") and bool(getattr(im, "is_animated", False))
    except Exception:
        return False


def _is_still_image(p):
    return p.suffix.lower() in IMG_EXT and not is_animated_image(p)


def _decoded(c, vs, decoder=None):
    """스트림 vs 의 프레임. decoder(예: 'libvpx-vp9')를 주면 기본 디코더 대신 그것으로 디코딩한다 (시각은 패킷에서)."""
    if not decoder:
        yield from c.decode(vs)
        return
    cc = av.CodecContext.create(decoder, "r")
    for pkt in c.demux(vs):
        if pkt.size:                             # 빈 패킷(끝)은 아래에서 한 번만 보낸다
            yield from cc.decode(pkt)
    yield from cc.decode(None)


def probe(path):
    """메타데이터 dict: kind('image'·'video'), frames, fps, duration, vfr, rotation, width, height(회전·SAR 적용한 표시 크기),
    codec, pix_fmt, header_frames(헤더의 프레임 수, mkv·webm 은 0), avg_rate(헤더 fps), has_alpha, hdr.
    frames 는 디코딩되는 프레임 수다 (편집 목록 밖 패킷은 세지 않음). has_alpha 는 디코딩한 화소에 알파가 있어 흰 배경에
    합성된다는 뜻이다 (GIF·움직이는 WebP 디코더는 늘 알파 형식, 알파 VP8·VP9 webm 은 libvpx 로 읽음).
    이미지는 1프레임으로 돌려준다: rotation 은 EXIF 방향의 회전 부분이고 EXIF 2·4·5·7 은 좌우 뒤집힘도 함께 적용된다.
    움직이는 GIF·PNG·WebP 는 확장자가 이미지여도 클립(kind='video')으로 읽는다 (is_animated_image)."""
    p = Path(path)
    if _is_still_image(p):
        with Image.open(p) as im:
            o = im.getexif().get(0x0112, 1)
            w, h = (im.height, im.width) if o in (5, 6, 7, 8) else im.size
            return dict(kind="image", frames=1, fps=0.0, duration=0.0, vfr=False,
                        rotation={3: 180, 4: 180, 5: 90, 6: -90, 7: -90, 8: 90}.get(o, 0),
                        width=w, height=h, codec=(im.format or "").lower(), pix_fmt=im.mode, header_frames=1, avg_rate=0.0,
                        has_alpha=im.mode in ("RGBA", "LA", "PA") or "transparency" in im.info, hdr=False)
    with av.open(str(p)) as c:
        if not c.streams.video:
            raise ValueError(f"영상 스트림이 없음: {p.name}")
        vs = c.streams.video[0]
        cc = vs.codec_context
        dec = _alpha_decoder(vs)
        if dec == "":
            _warn_no_alpha(p)
        info = dict(kind="video", container=c.format.name, codec=cc.name, pix_fmt="yuva420p" if dec is not None else cc.pix_fmt,
                    coded_w=cc.width, coded_h=cc.height, sar=_sar(vs), header_frames=int(vs.frames or 0),
                    avg_rate=float(vs.average_rate or 0), has_alpha=dec is not None or _has_alpha(cc.pix_fmt),
                    has_audio=bool(c.streams.audio))
        rate, tb, pts, n = float(vs.guessed_rate or vs.average_rate or 0), float(vs.time_base or 0), [], 0
        for pkt in c.demux(vs):                  # 디코딩 없이 패킷만 (1 패킷 = 1 프레임)
            if pkt.size == 0 or getattr(pkt, "is_discard", False):   # 끝을 알리는 빈 패킷, 편집 목록 밖 (디코더가 버림)
                continue
            n += 1
            t = pkt.pts if pkt.pts is not None else pkt.dts
            if t is not None:
                pts.append(t)
    with av.open(str(p)) as c:                   # 회전·HDR 은 프레임 정보라 첫 프레임만 디코딩
        f = next(c.decode(video=0), None)
    info["rotation"] = int(f.rotation) if f is not None else 0
    info["hdr"] = f is not None and int(f.color_trc) in (16, 18)    # PQ·HLG: rgb24 로 바꾸면 톤 매핑 없이 색이 바램
    ts = np.sort(np.asarray(pts, np.float64)) * tb
    dt = np.diff(ts)
    dt = dt[dt > 0]
    if len(dt):
        med = float(np.median(dt))
        info.update(fps=1.0 / med, duration=float(ts[-1] - ts[0]) + med, vfr=bool(dt.std() / dt.mean() > VFR_CV))
    else:                                        # 1프레임이거나 시각이 없는 스트림
        info.update(fps=rate, duration=n / rate if rate else 0.0, vfr=False)
    w, h = info["coded_w"], info["coded_h"]
    if info["sar"] != 1.0:                       # 애너모픽: 표시 폭 = coded_w * SAR
        w = max(1, round(w * info["sar"]))
    if info["rotation"] % 180:
        w, h = h, w
    info.update(frames=n, width=w, height=h)
    return info


def _over_white(rgba):
    """알파를 흰 배경에 합성 (이미지 노트북 flatten_alpha 와 같은 결과)"""
    a = rgba[..., 3:4].astype(np.uint16)
    return ((rgba[..., :3].astype(np.uint16) * a + 255 * (255 - a) + 127) // 255).astype(np.uint8)


def read_frames(path, max_side=None):
    """((T,H,W,3) uint8, (T,) 초) — 회전·SAR 적용, 알파는 흰 배경. 시각은 첫 프레임이 0.
    max_side 를 주면 긴 변을 그 이하로 줄여 읽는다 (EDA 용). 이미지는 1프레임 클립으로 돌려준다.
    움직이는 GIF·PNG·WebP 는 모든 프레임을 읽는다. 알파 VP8·VP9 webm 은 libvpx 로 디코딩한다 (없으면 경고).
    모든 프레임을 메모리에 올리므로 긴 영상은 max_side 를 주거나 미리 잘라 넣는다."""
    p = Path(path)
    if _is_still_image(p):
        a = load_image(p)
        if max_side and max(a.shape[:2]) > max_side:
            s = max_side / max(a.shape[:2])
            a = np.array(Image.fromarray(a).resize((max(1, round(a.shape[1] * s)), max(1, round(a.shape[0] * s))),
                                                   Image.Resampling.LANCZOS))
        return a[None], np.zeros(1)
    frames, ts, size = [], [], None
    with av.open(str(p)) as c:
        vs = c.streams.video[0]
        vs.thread_type = "AUTO"
        dec = _alpha_decoder(vs)
        if dec == "":
            _warn_no_alpha(p)
        fmt = "rgba" if dec or _has_alpha(vs.codec_context.pix_fmt) else "rgb24"
        rot0 = _stream_rotation(vs) if dec else 0    # 따로 연 디코더의 프레임에는 회전 정보가 없다
        sar, rate = _sar(vs), float(vs.guessed_rate or vs.average_rate or 0) or 16.0
        for i, f in enumerate(_decoded(c, vs, dec)):
            if size is None:                     # 첫 프레임 기준 크기 (중간에 크기가 바뀌는 스트림도 같은 크기로)
                w, h = f.width * sar, f.height
                s = min(1.0, max_side / max(w, h)) if max_side else 1.0
                size = (max(1, round(w * s)), max(1, round(h * s)))
                interp = "AREA" if size[0] <= f.width and size[1] <= f.height else "BICUBIC"
            a = f.to_ndarray(format=fmt, width=size[0], height=size[1], interpolation=interp)
            if fmt == "rgba":
                a = _over_white(a)
            k = int(round((f.rotation or rot0) / 90)) % 4  # 반시계 각도
            frames.append(np.ascontiguousarray(np.rot90(a, k)) if k else a)
            ts.append(float(f.time) if f.time is not None else i / rate)    # raw 스트림은 시각이 없음
    if not frames:
        raise ValueError(f"프레임을 읽지 못함: {p.name}")
    ts = np.asarray(ts, np.float64)
    order = np.argsort(ts, kind="stable")
    return np.stack([frames[i] for i in order]), ts[order] - ts[order[0]]


def to_8bit(im):
    """16비트 흑백(I;16)·정수(I)·실수(F) 이미지를 0~255 로 줄인다. 그대로 RGB 로 바꾸면 255 에서 잘려 하얗게 된다."""
    if im.mode.startswith("I;16"):
        a = np.asarray(im, dtype=np.float32) / 257.0
    elif im.mode in ("I", "F"):
        a = np.asarray(im, dtype=np.float32)
        hi = float(np.nanmax(a)) if a.size else 0.0
        if im.mode == "F" and hi <= 1.0:
            a = a * 255.0                  # 0~1 실수 이미지
        elif hi > 255.0:
            a = a / hi * 255.0             # 범위를 모르는 큰 값은 최댓값 기준
    else:
        return im
    return Image.fromarray(np.nan_to_num(a).clip(0, 255).astype(np.uint8), "L")


def flatten_alpha(im):
    """투명한 부분을 흰 배경에 합성한 RGB. EDA 와 학습이 같은 이미지를 보게 한다."""
    if im.mode in ("RGBA", "LA", "PA") or "transparency" in im.info:
        im = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im.convert("RGBA"))
    return im.convert("RGB")


def load_image(path):
    """(H,W,3) uint8. 이미지 노트북 load_rgb 와 같은 처리: EXIF 회전 → 8비트 → 투명은 흰 배경."""
    with Image.open(path) as im:
        return np.array(flatten_alpha(to_8bit(ImageOps.exif_transpose(im))))


# ------------------------------------------------------------------ 프레임 신호와 사건
def _u8(frames):
    """uint8 배열로 (0~1 실수를 그냥 uint8 로 바꾸면 검게 잘리므로 막는다)"""
    a = np.asarray(frames)
    if a.dtype != np.uint8:
        raise TypeError(f"uint8(0~255) 프레임이 필요함: {a.dtype} (0~1 실수는 (x * 255).round().astype(np.uint8))")
    return a


def _luma(rgb):
    """BT.601 휘도 (0~255, float32)"""
    rgb = rgb.astype(np.float32)
    return rgb[..., 0] * 0.299 + rgb[..., 1] * 0.587 + rgb[..., 2] * 0.114


def _shrink(frames, side):
    """긴 변이 side 보다 크면 모든 프레임을 같은 크기로 줄인다 (BOX 평균)"""
    h, w = frames.shape[1:3]
    s = side / max(h, w)
    if s >= 1:
        return frames
    size = (max(1, round(w * s)), max(1, round(h * s)))
    return np.stack([np.asarray(Image.fromarray(f).resize(size, Image.Resampling.BOX)) for f in frames])


def _hist_corr(a, b, bins=128):
    """Y 히스토그램 상관 (cv2.compareHist HISTCMP_CORREL 과 같음)"""
    ha = np.histogram(a, bins, (0, 256))[0].astype(np.float64)
    hb = np.histogram(b, bins, (0, 256))[0].astype(np.float64)
    ha -= ha.mean()
    hb -= hb.mean()
    d = math.sqrt(float((ha * ha).sum() * (hb * hb).sum()))
    return float((ha * hb).sum() / d) if d > 0 else 1.0


_OFFSETS = sorted(((dy, dx) for dy in range(-4, 5) for dx in range(-4, 5)), key=lambda o: math.hypot(*o))


def _block_motion(g0, g1, block=8, bias=0.5):
    """8x8 블록 정합: g1 의 각 블록이 g0 의 ±4px 안 어디서 왔는지 찾아 변위 크기(px)를 돌려준다.
    평평하거나 잡음뿐인 블록이 엉뚱한 변위를 고르지 않게 변위 1px 마다 화소당 bias 만큼 비용을 더한다."""
    r = 4
    bh, bw = (g0.shape[0] - 2 * r) // block, (g0.shape[1] - 2 * r) // block
    if bh < 1 or bw < 1:
        return np.zeros(1, np.float32)
    ref = g1[r:r + bh * block, r:r + bw * block]
    best = np.full((bh, bw), np.inf, np.float32)
    best_d = np.zeros((bh, bw), np.float32)
    for dy, dx in _OFFSETS:                     # 가까운 변위부터 → 비용이 같으면 작은 변위
        d = math.hypot(dy, dx)
        cand = g0[r + dy:r + dy + bh * block, r + dx:r + dx + bw * block]
        cost = np.abs(ref - cand).reshape(bh, block, bw, block).sum((1, 3)) + bias * block * block * d
        better = cost < best
        best[better], best_d[better] = cost[better], d
    return best_d.ravel()


def frame_signals(frames, fps, analysis_side=96):
    """프레임별·전환별 신호 (detect_events·summarize_clip 의 입력). blur 를 뺀 값은 긴 변 analysis_side 로 줄여 잰다.
    프레임별 (T,): mean_luma(0~1), black(검은 화소 비율), blur(라플라시안 분산, 긴 변 BLUR_SIDE), sat(HSV 채도 0~255), rgb (T,3)
    전환별 (T-1,): mad(이웃 프레임과의 평균 |dY|/255, 홀드용), mad_ref(멈춤 후보의 첫 프레임과의 평균 |dY|/255, freezedetect 처럼
      FREEZE_MAD 이상 달라지면 그 프레임이 새 기준. 느린 움직임이 묻히지 않게 긴 변 BLUR_SIDE 에서 잰다),
      content(HSV 평균 차), hist_corr(Y 히스토그램 상관),
      motion_med·motion_p95(블록 변위 px)
    motion_med_s·motion_p95_s (T-s,), motion_med_k·motion_p95_k (T-k,): 프레임 i 와 i+s (s = motion_s, 약 MOTION_MID 초),
      i+k (k = motion_k, 약 MOTION_SPAN 초) 비교를 간격으로 나눈 프레임당 변위. 이웃 비교는 96px 에서 1px 미만을 0 으로 읽고
      (16fps 에서 초당 긴 변의 17%, 60fps 에서 62% 미만), k 간격은 ±4px 를 넘으면 포화하므로 30·60fps 의 중간 속도는 s 간격이 잰다.
    side = 분석 프레임의 긴 변 (움직임을 '긴 변 비율' 로 바꿀 때 씀). 1프레임(이미지)이면 전환별 값은 없다."""
    frames = _u8(frames)
    T = len(frames)
    small = _shrink(frames, analysis_side)
    Y = _luma(small)
    hsv = np.stack([np.asarray(Image.fromarray(f).convert("HSV"), np.int16) for f in small])
    yb = _luma(_shrink(frames, BLUR_SIDE))
    lap = yb[:, 1:-1, 1:-1] * -4 + yb[:, :-2, 1:-1] + yb[:, 2:, 1:-1] + yb[:, 1:-1, :-2] + yb[:, 1:-1, 2:]
    sig = dict(fps=float(fps), side=int(max(Y.shape[1:])), mean_luma=Y.mean((1, 2)) / 255.0,
               black=(Y <= BLACK_PIX * 255).mean((1, 2)), blur=lap.reshape(T, -1).var(1) if lap[0].size else np.zeros(T),
               sat=hsv[..., 1].mean((1, 2)), rgb=small.reshape(T, -1, 3).mean(1))
    if T > 1:
        sig["mad"] = np.abs(np.diff(Y, axis=0)).mean((1, 2)) / 255.0
        ref, mr = 0, np.empty(T - 1)             # freezedetect: 멈춤 후보의 첫 프레임과 비교 (느린 움직임은 쌓여서 풀림)
        for i in range(1, T):                    # 96px 에서는 0.3px/프레임 같은 느린 팬이 3배 느리게 쌓여 멈춤으로 읽힌다
            mr[i - 1] = np.abs(yb[i] - yb[ref]).mean() / 255.0
            if mr[i - 1] >= FREEZE_MAD:
                ref = i
        sig["mad_ref"] = mr
        sig["content"] = np.abs(np.diff(hsv, axis=0)).mean((1, 2, 3))   # PIL 색상은 0~255 (OpenCV 0~179)
        sig["hist_corr"] = np.array([_hist_corr(Y[i], Y[i + 1]) for i in range(T - 1)])
        k = sig["motion_k"] = max(2, round(fps * MOTION_SPAN))
        s = sig["motion_s"] = max(1, math.ceil(fps * MOTION_MID - 1e-6))   # 23.976fps 도 2 (round 면 1)
        done = {}
        for suffix, step in (("", 1), ("_s", s), ("_k", k)):
            if step not in done:                 # 16fps 이하는 s = 1 이라 이웃 비교와 같다
                mv = [_block_motion(Y[i], Y[i + step]) / step for i in range(T - step)]
                done[step] = (np.array([np.median(m) for m in mv]), np.array([np.percentile(m, 95) for m in mv]))
            sig["motion_med" + suffix], sig["motion_p95" + suffix] = done[step]
    return sig


def _runs(mask, min_len=1):
    """True 가 이어지는 구간 [(시작, 끝)) 목록"""
    m = np.concatenate([[0], np.asarray(mask, np.int8), [0]])
    e = np.flatnonzero(np.diff(m))
    return [(int(a), int(b)) for a, b in zip(e[::2], e[1::2]) if b - a >= min_len]


def detect_events(sig, fps, min_scene_s=0.5, freeze_min_s=0.25, timestamps=None):
    """black·freeze: 프레임 구간 [(시작, 끝)), cuts: 프레임 번호 i (i-1 → i 에서 장면이 바뀜),
    hold_ratio: 멈춤·검은 구간 밖에서 바로 앞 프레임과 같은 전환의 비율 (셀 애니 on twos ≈ 0.5, 24→30fps 변환 ≈ 0.2),
    drawing_fps = fps * (1 - hold_ratio) (실제로 새 그림이 바뀌는 횟수/초). 검은 화면과의 경계는 컷으로 세지 않는다.
    멈춤 = 첫 프레임과 같은 그림(mad_ref < FREEZE_MAD)이 이어지는 묶음이 freeze_min_s 이상 (ffmpeg freezedetect 처럼 첫 프레임의
    시작부터 마지막 같은 프레임의 시작까지, 곧 같은 전환 round(freeze_min_s*fps) 개 이상).
    timestamps(read_frames 의 시각)를 주면 묶음 길이를 화면에 떠 있는 시간으로 잰다: 표시 시간(다음 다른 프레임까지,
    마지막 프레임은 1/fps)을 프레임 수로 반올림한 값 - 1. 그래서 VFR·GIF 에서 한 프레임이 오래 떠 있는 정지 화면
    (리샘플하면 같은 프레임이 연달아 들어감)도 멈춤이 되고 clean_segments 가 뺀다. CFR 시각이면 주지 않았을 때와 같다."""
    blk = np.asarray(sig["black"]) >= BLACK_PIC
    T = len(blk)
    ev = dict(black=_runs(blk), freeze=[], cuts=[], hold_ratio=float("nan"), drawing_fps=float("nan"))
    if "mad" not in sig:
        return ev
    still = sig["mad"] < FREEZE_MAD               # 홀드: 바로 앞 프레임과 같음
    valid = ~(blk[:-1] & blk[1:])                 # 검은 프레임끼리의 전환은 멈춤·홀드로 세지 않는다
    same = (np.asarray(sig.get("mad_ref", sig["mad"])) < FREEZE_MAD) & valid
    g0 = np.flatnonzero(np.concatenate([[True], ~same]))             # 같은 그림 묶음 [g0, g1) 의 시작 프레임
    g1 = np.append(g0[1:], T)
    length = (g1 - g0 - 1).astype(np.float64)                        # 묶음 안 전환 수
    if timestamps is not None and fps > 0:
        t = np.asarray(timestamps, np.float64)
        if t.shape != (T,):
            raise ValueError(f"timestamps 길이 {t.shape} 가 프레임 수 {T} 와 다름")
        end = np.append(t[1:], t[-1] + 1.0 / fps)                    # 프레임이 화면에서 내려가는 시각
        length = np.floor((end[g1 - 1] - t[g0]) * fps + 0.5) - 1     # 표시 시간을 프레임 수로 (CFR 이면 위와 같음)
    need = max(1, round(freeze_min_s * fps))
    for a, b, n in zip(g0, g1, length):
        if n >= need and not blk[a:b].all():
            ev["freeze"].append((int(a), int(b)))
            valid[a:b - 1] = False                # 멈춘 묶음 안의 전환은 홀드 비율에서 뺀다
    if valid.any():
        ev["hold_ratio"] = float(still[valid].mean())
        ev["drawing_fps"] = float(fps) * (1.0 - ev["hold_ratio"])
    c = sig["content"]
    pad = np.pad(c, 2, mode="edge")
    neigh = (pad[:-4] + pad[1:-3] + pad[3:-1] + pad[4:]) / 4.0
    # 히스토그램·이웃 대비 기준은 HSV 차가 CUT_MIN 이상일 때만 쓴다: 평평한 셀화·그라데이션·GIF 팔레트 변화는
    # 히스토그램이 뾰족해 움직임만으로 상관이 0.8 아래로 떨어진다 (합성 GIF 0.44, yuv444 첫 전환 0.75 에서 확인)
    assist = (sig["hist_corr"] <= 1.0 - CUT_HIST) | (c / np.maximum(neigh, 1e-6) >= CUT_RATIO)
    cut = (c >= CUT_CONTENT) | ((c >= CUT_MIN) & assist)
    last, gap = -10 ** 9, max(1, round(min_scene_s * fps))
    for i in np.flatnonzero(cut & ~(blk[:-1] | blk[1:])):     # 검은 화면과의 경계는 black 구간이 맡는다
        if i + 1 - last >= gap:                   # 장면은 min_scene_s 이상
            ev["cuts"].append(int(i + 1))
            last = i + 1
    return ev


def clean_segments(n_frames, events):
    """컷으로 나누고 검은·멈춘 프레임을 뺀 깨끗한 구간 [(시작, 끝)) 목록. 학습 창은 이 안에서만 고른다."""
    keep = np.ones(n_frames, bool)
    for a, b in list(events.get("black", [])) + list(events.get("freeze", [])):
        keep[a:b] = False
    bounds = [0] + sorted(c for c in events.get("cuts", []) if 0 < c < n_frames) + [n_frames]
    out = []
    for a, b in zip(bounds[:-1], bounds[1:]):
        out += [(a + s, a + e) for s, e in _runs(keep[a:b])]
    return out


def summarize_clip(frames, fps, sig=None, events=None, timestamps=None):
    """EDA 표의 클립 요약값. subject_motion·camera_motion 은 깨끗한 구간 안에서 잰 평균 움직임 (초당 화면 긴 변의 비율):
    이웃 프레임, motion_s(약 1/16초), motion_k(약 0.5초) 간격 비교 평균 가운데 가장 큰 값 (빠른 움직임은 이웃, 30·60fps 의
    중간 속도는 1/16초, 1px 미만의 느린 움직임은 0.5초 간격이 맞다). static = 클립인데 subject_motion < STATIC_MOTION.
    blur 는 프레임 중앙값, sat(0~255)·mean_rgb(0~255) 는 평균. 1프레임(이미지)도 받는다: 움직임 0, static False,
    hold_ratio·drawing_fps 는 nan. timestamps 는 events 를 주지 않을 때 detect_events 에 넘긴다."""
    sig = frame_signals(frames, fps) if sig is None else sig
    ev = detect_events(sig, fps, timestamps=timestamps) if events is None else events
    T = len(sig["black"])
    out = dict(subject_motion=0.0, camera_motion=0.0, hold_ratio=ev["hold_ratio"], drawing_fps=ev["drawing_fps"],
               blur=float(np.median(sig["blur"])), n_cuts=len(ev["cuts"]), n_black=sum(b - a for a, b in ev["black"]),
               n_freeze=sum(b - a for a, b in ev["freeze"]), sat=float(np.mean(sig["sat"])),
               mean_rgb=tuple(round(float(v), 1) for v in np.mean(sig["rgb"], 0)))
    if T > 1:
        steps = [("", 1)] + [(sfx, sig[key]) for sfx, key in (("_s", "motion_s"), ("_k", "motion_k")) if key in sig]
        ok = {sfx: np.zeros(max(T - st, 0), bool) for sfx, st in steps}
        for a, b in clean_segments(T, ev):        # 비교하는 두 프레임이 같은 깨끗한 구간 안에 있을 때만
            for sfx, st in steps:
                ok[sfx][a:max(a, b - st)] = True
        for key, name in (("subject_motion", "motion_p95"), ("camera_motion", "motion_med")):
            m = [sig[name + sfx][ok[sfx]].mean() for sfx, _ in steps if ok[sfx].any()]
            out[key] = float(max(m, default=0.0) * fps / sig["side"])
    out["static"] = bool(T > 1 and out["subject_motion"] < STATIC_MOTION)
    return out


# ------------------------------------------------------------------ 중복·캡션
_DCT = {}


def _phash(rgb, hs=8, hf=4):
    """imagehash.phash 와 같은 64비트 (32x32 회색 → 정규화 안 한 DCT-II → 왼쪽 위 8x8 이 중앙값보다 큰지)"""
    n = hs * hf
    if n not in _DCT:
        k, i = np.meshgrid(np.arange(n), np.arange(n), indexing="ij")
        _DCT[n] = 2.0 * np.cos(np.pi * k * (2 * i + 1) / (2 * n))
    g = np.asarray(Image.fromarray(rgb).convert("L").resize((n, n), Image.Resampling.LANCZOS), np.float64)
    low = (_DCT[n] @ g @ _DCT[n].T)[:hs, :hs]
    return (low > np.median(low)).ravel()


def clip_signature(frames, k=8, skip_first=False):
    """같은 상대 위치(0~1) k 프레임의 pHash (k,64). fps·길이·재인코딩이 달라도 같은 클립이면 가깝다.
    같은 시작 이미지로 만든 I2V 클립끼리는 0번 프레임이 같으므로 skip_first=True 로 비교한다."""
    idx = np.linspace(0, len(frames) - 1, k).round().astype(int)
    return np.stack([_phash(_u8(frames[i])) for i in (idx[1:] if skip_first else idx)])


def signature_distance(a, b):
    """같은 위치끼리의 평균 해밍 거리 (64비트 중). 같은 클립 0, 재인코딩본 ~4, 다른 클립 20 이상, DUP_BITS 이하면 중복 의심."""
    n = min(len(a), len(b))
    return float((a[:n] != b[:n]).sum(1).mean())


# 동작 = 움직이는 동사 형태만. 정지 상태(standing, sitting, holding, looking at, pointed ears, eyes closed, open field,
# close-up, wavy hair, bow tie, lower body)는 동작이 아니다. 영어는 단어 경계, 한국어는 어간(뒤에 어미가 붙음)으로 찾는다.
_MOTION_EN = (
    r"walk\w*|run(?:s|ning)?|ran|jog\w*|sprint\w*|hurr(?:y|ies|ying|ied)|rush(?:es|ed|ing)|wander\w*|"
    r"turn\w*|spin(?:s|ning)?|spun|twirl\w*|rotat\w*|wav(?:e|es|ed|ing)|nod(?:s|ded|ding)?|shak(?:e|es|ing)|shook|"
    r"blink\w*|wink\w*|smil(?:es|ed)|laugh\w*|giggl\w*|cr(?:y|ies|ied|ying)|yawn\w*|sigh(?:s|ed|ing)|"
    r"glanc\w*|look(?:s|ed)? (?:around|up|down|back|away|over)|looking around|"
    r"rais(?:e|es|ing)|lower(?:s|ing)|lift(?:s|ed|ing)|reach(?:es|ed|ing)|point(?:s|ing)|gestur(?:e|es|ed|ing)|clap\w*|"
    r"(?:sits?|sitting|sat) (?:down|up)|(?:stands?|standing|stood|gets?|getting|got) up|kneels(?: down)?|knelt|kneeling down|"
    r"jump\w*|hop(?:s|ped|ping)?|skip(?:s|ped|ping)|leans|leaning (?:forward|in|over|back)|"
    r"(?:steps?|stepped|stepping) (?:forward|back|backward|aside|closer|in|out|up|down|onto|off|toward|towards|into|over)|"
    r"bowing|bows down|tilt(?:s|ing)|sway\w*|bounc\w*|open(?:s|ing)|clos(?:es|ing)|grab\w*|throw\w*|threw|toss\w*|"
    r"catch(?:es|ing)|caught|push(?:es|ed|ing)|pull(?:s|ed|ing)|kick\w*|punch\w*|hug(?:s|ged|ging)|stretch(?:es|ed|ing)|"
    r"danc\w*|climb\w*|fall(?:s|ing)|fell|slid(?:e|es|ing)|swinging|swung|swings (?:her|his|its|their|back|forward|around)|"
    r"swim(?:s|ming)?|swam|crawl\w*|"
    r"fl(?:y|ies|ying|ew)|float\w*|drift\w*|roll(?:s|ing)|flutter\w*|blow(?:s|ing)|ripple\w*|flap\w*|wag(?:s|ged|ging)|"
    r"wiggl\w*|twitch\w*|mov(?:e|es|ed|ing)|approach\w*|enter(?:s|ed|ing)|exit(?:s|ed|ing)|leav(?:e|ing)|"
    r"eat(?:s|ing)|drink(?:s|ing)|chew\w*|talk(?:s|ed|ing)|speak(?:s|ing)|sing(?:s|ing)|transform\w*")
_MOTION_KO = (
    r"걷는|걷고|걷다|걸어가|걸어오|걸어간|걸어온|걸음을|달린다|달리는|달리고|달려가|달려오|달려간|달려온|뛴다|뛰는|뛰고|뛰며|"
    r"뛰어(?!난)|점프|춤|움직|흔들|흔든|끄덕|깜빡|깜박|윙크|돌아보|돌아서|돌아선|뒤돌|돌린|돌며|빙글|회전|웃는다|웃으며|미소 ?짓|"
    r"일어나|일어서|일어선|앉는다|앉으며|날아|날갯짓|떨어지|떨어진다|기울이|기울인|다가오|다가가|다가서|다가온|다가간|"
    r"들어 ?올리|들어 ?올린|내려놓|던지|던진|잡는다|잡으며|펄럭|흩날|휘날|나부끼|미끄러|굴러|구른다|튀어 ?오르|튄다|흘러|"
    r"떠다니|손짓|하품|박수|손을 흔|고개를 (?:끄덕|돌|숙이|젓|기울|든다|들어)")
# 카메라·샷 = 고정 카메라, 카메라 움직임('the camera slowly pans', 'camera is panning'), 샷 크기·앵글(close-up·closeup·close up,
# long shot, full body, from above …)
_STATIC_CAM_EN = (r"(?:static|fixed|locked(?:[- ]off)?|stationary|still|tripod)[- ](?:camera|shot)"
                  r"|camera (?:is|stays|remains|holds) (?:\w+ )?(?:static|still|fixed|locked|stationary|steady)(?! \w+ing\b)"
                  r"|camera does(?: not|n['’]t) move|no camera (?:movement|motion)")
_STATIC_CAM_KO = r"(?:고정(?:된)?|정지(?:된)?) ?카메라|카메라(?:는|가|도)? ?(?:고정|정지|움직이지 않)"
_CAMERA_EN = (
    _STATIC_CAM_EN
    + r"|camera (?:is |then |also |still |now |\w+ly ){0,2}(?:pan|tilt|zoom|push|pull|track|follow|mov|orbit|rotat|dolly|truck|crane|"
      r"circl|ris|descend|ascend|shak|sway|drift|glid|swing|swoop|hold|stay|remain)\w*"
    + r"|(?:pan(?:s|ning)?|tilt(?:s|ing)?|zoom(?:s|ing)?|dolly(?:ing)?) (?:in|out|left|right|up|down|forward|back|away)"
    + r"|(?:extreme |medium |big )?close[- ]?ups?"
    + r"|(?:wide|full|long|medium|establishing|tracking|aerial|overhead|cowboy|bust|two|pov|point[- ]of[- ]view|low[- ]angle|"
      r"high[- ]angle|dutch[- ]angle|bird'?s[- ]eye|over[- ]the[- ]shoulder|first[- ]person|drone|insert|static)[- ]shots?"
    + r"|(?:upper|full|lower)[- ]body|full[- ]length|hand[- ]?held|(?:low|high|dutch|eye[- ]level)[- ]angle"
    + r"|(?:bird'?s[- ]eye|aerial|overhead|top[- ]down) view|from (?:above|below|behind)|pov")
_CAMERA_KO = (
    _STATIC_CAM_KO
    + r"|카메라(?:가|는|도) [^,.;]{0,20}?(?:이동|움직|따라|돌|회전|다가|멀어|올라|내려|흔들|패닝|팬|줌|틸트|당기|밀|비추|비춘)"
    + r"|카메라 ?(?:이동|워크|무빙|움직임|패닝|줌|틸트)"
    + r"|클로즈 ?업|근접 ?(?:촬영|샷)|(?:미디엄|바스트|풀|전신|와이드|롱|트래킹|오버 ?더 ?숄더|시점|부감|항공|드론) ?샷"
    + r"|상반신|하반신|전신|패닝|줌 ?(?:인|아웃)|틸트 ?(?:업|다운)|부감|앙각|하이 ?앵글|로우 ?앵글|핸드 ?헬드")
MOTION_RE = re.compile(r"\b(?:" + _MOTION_EN + r")\b|" + _MOTION_KO, re.I)
# 부정한 움직임('does not move', 'without moving', '움직이지 않는다')은 동작이 아니다
_NEG_MOTION = re.compile(r"\b(?:does(?: not|n['’]t)|do(?: not|n['’]t)|did(?: not|n['’]t)|not|never|without|no) "
                         r"(?:\w+ly |really |even )?mov(?:e|es|ed|ing)\b"
                         r"|움직이지 ?(?:않|마)|움직임(?:이|은)? ?없", re.I)
CAMERA_RE = re.compile(r"\b(?:" + _CAMERA_EN + r")\b|" + _CAMERA_KO, re.I)
_STATIC_CAM = re.compile(r"\b(?:" + _STATIC_CAM_EN + r")\b|" + _STATIC_CAM_KO, re.I)


def caption_issues(caption, trigger_re, is_clip, fixed_words=(), summary=None):
    """캡션 문제 목록 (비면 문제 없음). 영상 캡션은 트리거 + 동작(시간 순서) + 카메라·샷 표현, 고정 외형은 쓰지 않는다.
    동작·카메라 표현은 영어와 한국어의 흔한 표현을 찾는다 (MOTION_RE·CAMERA_RE, 정지 상태 낱말은 동작으로 보지 않음).
    summary(summarize_clip 결과)를 주면 EDA 값과의 모순도 본다."""
    cap, out = caption or "", []
    if not trigger_re.search(cap):
        out.append("트리거 없음")
    n = len(cap.split())
    if not CAPTION_WORDS[0] <= n <= CAPTION_WORDS[1]:
        out.append(f"단어 수 {n} (권장 {CAPTION_WORDS[0]}~{CAPTION_WORDS[1]})")
    for w in fixed_words:                      # 한글 뒤 조사는 허용 (이미지 노트북 TRIGGER_RE 와 같은 경계)
        if re.search(r"(?<!\w)" + re.escape(w) + r"(?![A-Za-z0-9_])", cap, re.I):
            out.append(f"고정 특징 단어: {w} (외형은 트리거가 배우므로 빼기)")
    if is_clip:
        moving = MOTION_RE.search(_NEG_MOTION.sub(" ", _STATIC_CAM.sub(" ", cap)))
        if not moving:
            out.append("동작 표현 없음 (예: turns, walks, waves)")
        if not CAMERA_RE.search(cap):
            out.append("카메라·샷 표현 없음 (예: static camera, camera pans left, medium shot)")
        if summary is not None:
            if _STATIC_CAM.search(cap) and summary["camera_motion"] > CAMERA_MOVING:
                out.append(f"static camera 라고 했지만 카메라가 움직임 ({summary['camera_motion']:.2f})")
            if moving and summary["subject_motion"] < STATIC_MOTION:
                out.append(f"동작을 적었지만 거의 정지된 클립 ({summary['subject_motion']:.3f})")
    return out


# ------------------------------------------------------------------ 전처리
def resample_to_fps(timestamps, target_fps):
    """격자 시각 t_k = k/target_fps 에 화면에 떠 있는 원본 프레임 번호 (ffmpeg fps 필터 round=near 와 같음): 시작 시각을
    출력 단위로 반올림한 값(반은 올림, 그래서 32fps → 16fps 는 [0,2,4,...])이 k 이하인 마지막 프레임.
    길이 = round((마지막 시각 + 원본 프레임 길이(dt 중앙값)) * target_fps) (ffmpeg 처럼 끝도 반올림).
    VFR·GIF 의 긴 프레임은 떠 있는 시간만큼, 원본 fps 가 더 낮으면 같은 프레임이 반복된다
    (예: 12fps → 16fps 는 [0,1,1,2,3,4,4,5,...]). 섞인 순서로 줘도 원래 번호를 돌려준다."""
    ts = np.asarray(timestamps, np.float64)
    order = np.argsort(ts, kind="stable")
    t = ts[order] - ts[order[0]]
    dt = np.diff(t)
    dt = dt[dt > 0]
    dur = float(np.median(dt)) if len(dt) else 1.0 / target_fps
    n = max(1, int(math.floor((t[-1] + dur) * target_fps + 0.5 + 1e-9)))
    r = np.searchsorted(t, (np.arange(n) + 0.5) / target_fps - 1e-9, side="left") - 1   # 반올림 시각 < k + 0.5 인 마지막
    return order[np.clip(r, 0, len(t) - 1)]


def round_4k1(n, temporal=4):
    """n 이하에서 가장 큰 temporal*k+1 (80 → 77, 83 → 81). Wan VAE 는 이를 넘는 프레임을 경고 없이 버린다."""
    return 0 if n < 1 else (int(n) - 1) // temporal * temporal + 1


def windows(seg_len, F, mode="uniform", n=3, rng=None):
    """길이 seg_len 구간 안 F 프레임 창의 시작 위치. head: 처음 1개, uniform: 고르게 n 개, chunk: 겹치지 않게 전부,
    slide: F//2 간격, random: 무작위 n 개 (에폭마다 다시 부르면 다른 창 → I2V 는 다른 시작 이미지 증강)."""
    if F < 1 or seg_len < F:
        return []
    last = seg_len - F
    if mode == "head":
        return [0]
    if mode == "uniform":
        return sorted(set(np.linspace(0, last, n).round().astype(int).tolist()))
    if mode == "chunk":
        return list(range(0, last + 1, F))
    if mode == "slide":
        return list(range(0, last + 1, max(1, F // 2)))
    if mode == "random":
        rng = np.random.default_rng() if rng is None else rng
        return sorted(int(x) for x in rng.choice(last + 1, size=min(n, last + 1), replace=False))
    raise ValueError(f"모르는 창 방식: {mode}")


def make_buckets(token_budget, frame_options, aspects, max_area, token_div, temporal, multiple):
    """(F, H, W, tokens) 목록. tokens = (1+(F-1)//temporal) * (H//token_div) * (W//token_div) ≤ token_budget,
    H*W ≤ max_area, H·W 는 multiple 의 배수. aspects 는 W/H (예: 832/480, 480/832)."""
    out = set()
    for F in frame_options:
        if F < 1 or (F - 1) % temporal:
            raise ValueError(f"프레임 수는 {temporal}k+1 이어야 함: {F}")
        lat_f = 1 + (F - 1) // temporal
        area = min(max_area, token_budget / lat_f * token_div * token_div)
        for a in aspects:
            w = int(math.sqrt(area * a) + 1e-6) // multiple * multiple
            h = int(math.sqrt(area / a) + 1e-6) // multiple * multiple
            if w >= multiple and h >= multiple:
                out.add((F, h, w, lat_f * (h // token_div) * (w // token_div)))
    return sorted(out)


def pick_bucket(buckets, avail_frames, w, h, prefer="frames", allow_upscale=1.0):
    """원본 w x h, 쓸 수 있는 프레임 avail_frames 에 맞는 (F, H, W, tokens). 없으면 None.
    덮도록 키우는 배율 max(W/w, H/h) 가 allow_upscale 이하인 것 가운데 비율이 가장 가까운 것과 ASPECT_TOL 안의 것을 같은
    비율로 보고 (토큰 예산 때문에 줄어든 버킷은 배수 맞춤으로 비율이 몇 % 어긋남), prefer='frames' 면 F, 'area' 면 넓이가 큰 것."""
    la = math.log(w / h)
    cand = [b for b in buckets if b[0] <= avail_frames and max(b[2] / w, b[1] / h) <= allow_upscale + 1e-9]
    if not cand:
        return None
    err = {b: round(abs(math.log(b[2] / b[1]) - la), 9) for b in cand}
    best = min(err.values())
    cand = [b for b in cand if err[b] <= best + ASPECT_TOL]
    if prefer == "frames":
        return max(cand, key=lambda b: (b[0], -err[b], b[1] * b[2]))
    return max(cand, key=lambda b: (b[1] * b[2], -err[b], b[0]))


def resize_crop_frames(frames, h, w):
    """모든 프레임에 같은 배율·같은 가운데 자르기 (프레임마다 다르면 흔들림을 배운다). 줄일 때 LANCZOS, 키울 때 BICUBIC."""
    frames = _u8(frames)
    h0, w0 = frames.shape[1:3]
    s = max(w / w0, h / h0)
    bw, bh = min(w0, w / s), min(h0, h / s)       # 원본에서 잘라 쓸 크기 (w/(w/w0) 가 부동소수 오차로 w0 를 넘지 않게)
    box = ((w0 - bw) / 2, (h0 - bh) / 2, (w0 + bw) / 2, (h0 + bh) / 2)
    rs = Image.Resampling.LANCZOS if s < 1 else Image.Resampling.BICUBIC
    return np.stack([np.asarray(Image.fromarray(f).resize((w, h), rs, box=box)) for f in frames])


# ------------------------------------------------------------------ 보기·저장
@lru_cache(None)
def _font(size):
    """한글이 보이는 글꼴: 노트북 0-1 셀이 찾은 글꼴 파일(환경변수 LORA_KO_FONT), Windows 맑은 고딕, Linux 나눔·Noto CJK
    (Pillow 는 ~/.local/share/fonts 도 파일 이름으로 찾는다), 없으면 기본 글꼴"""
    for name in (os.environ.get("LORA_KO_FONT", ""), "malgun.ttf", "NanumGothic.ttf", "NotoSansCJK-Regular.ttc",
                 "NotoSansKR-Regular.ttf", "AppleSDGothicNeo.ttc", "DejaVuSans.ttf"):
        if not name:
            continue
        try:
            return ImageFont.truetype(name, size)
        except OSError:
            pass
    return ImageFont.load_default()


def contact_sheet(rows, labels=None, n=8, thumb_h=120, pad=4):
    """행마다 클립 1개(예: 체크포인트별 같은 시드 결과)에서 고르게 n 장을 뽑아 붙인 PIL.Image. labels 는 행 이름."""
    thumbs = []
    for clip in rows:
        clip = _u8(clip)
        clip = clip[None] if clip.ndim == 3 else clip
        tw = max(1, round(clip.shape[2] * thumb_h / clip.shape[1]))
        idx = np.linspace(0, len(clip) - 1, min(n, len(clip))).round().astype(int)
        thumbs.append([Image.fromarray(clip[i]).resize((tw, thumb_h), Image.Resampling.LANCZOS) for i in idx])
    font = _font(max(10, min(18, thumb_h // 6)))
    lab = [str(x)[:40] for x in labels] if labels is not None else []
    measure = ImageDraw.Draw(Image.new("RGB", (1, 1)))
    lab_w = int(max(measure.textlength(s, font=font) for s in lab)) + 2 * pad if lab else 0
    tw, cols = max(t[0].width for t in thumbs), max(len(t) for t in thumbs)
    sheet = Image.new("RGB", (lab_w + pad + cols * (tw + pad), pad + len(thumbs) * (thumb_h + pad)), "white")
    d = ImageDraw.Draw(sheet)
    for r, row in enumerate(thumbs):
        y = pad + r * (thumb_h + pad)
        if lab:
            d.text((pad, y + (thumb_h - getattr(font, "size", 10)) // 2), lab[r], fill="black", font=font)
        for k, im in enumerate(row):
            sheet.paste(im, (lab_w + pad + k * (tw + pad), y))
    return sheet


def _encode(path, frames, fps, codec, pix_fmt, options=None, pts_ms=None, rotation=0):
    """PyAV 로 영상 쓰기. 프레임은 (H,W,3) RGB 또는 (H,W,4) RGBA. pts_ms 를 주면 1ms 단위 시각(VFR, 음수면 mp4 편집 목록이
    생겨 그 프레임은 재생·디코딩에서 빠진다), rotation 은 표시 회전(반시계)."""
    h, w = frames[0].shape[:2]
    with av.open(str(path), "w") as c:
        st = c.add_stream(codec, rate=Fraction(fps).limit_denominator(1001))
        st.width, st.height, st.pix_fmt = w, h, pix_fmt
        st.options = dict(options or {})
        if pts_ms is not None:
            st.codec_context.time_base = st.time_base = Fraction(1, 1000)
        if rotation:
            st.set_display_rotation(rotation)
        for i, a in enumerate(frames):
            a = np.ascontiguousarray(_u8(a))
            f = av.VideoFrame.from_ndarray(a, format="rgba" if a.shape[-1] == 4 else "rgb24")
            if pts_ms is not None:
                f.pts, f.time_base = int(pts_ms[i]), Fraction(1, 1000)
            for pk in st.encode(f):
                c.mux(pk)
        for pk in st.encode():
            c.mux(pk)
    return str(path)


def save_mp4(path, frames, fps, crf=18):
    """H.264(yuv420p) mp4 (노트북에서는 IPython.display.Video(path, embed=True) 로 본다).
    홀수 크기는 가장자리를 복제해 짝수로 맞춘다 (libx264 yuv420p 는 홀수에서 'Generic error')."""
    frames = _u8(frames)
    frames = frames[None] if frames.ndim == 3 else frames
    H, W = frames.shape[1:3]
    if H % 2 or W % 2:
        frames = np.pad(frames, ((0, 0), (0, H % 2), (0, W % 2), (0, 0)), mode="edge")
    if "libx264" in av.codecs_available:
        return _encode(path, frames, fps, "libx264", "yuv420p", {"crf": str(crf), "preset": "medium"})
    return _encode(path, frames, fps, "mpeg4", "yuv420p", {"b": "8M"})       # libx264 가 없는 FFmpeg 빌드


def save_webp(path, frames, fps):
    """움직이는 WebP 미리보기 (GIF 보다 작고 색이 온전함). 프레임 길이는 누적 반올림(16fps → 62,63,63,62ms)이라
    전체 재생 시간이 정확하다. 같은 프레임이 이어지면 Pillow 가 합치지만 재생 시간은 그대로다."""
    frames = _u8(frames)
    ims = [Image.fromarray(f) for f in (frames[None] if frames.ndim == 3 else frames)]
    ms = np.diff(np.round(np.arange(len(ims) + 1) * 1000.0 / fps)).astype(int).tolist()
    ims[0].save(path, format="WEBP", save_all=True, append_images=ims[1:], duration=ms, loop=0, quality=80, method=4)
    return str(path)


# ------------------------------------------------------------------ 시험 데이터
def _is_red(a, box, inset=3):
    y0, y1, x0, x1 = box
    m = a[y0 + inset:y1 - inset, x0 + inset:x1 - inset].reshape(-1, 3).mean(0)
    return bool(m[0] > 170 and m[1] < 90 and m[2] < 90)


def check_test_expectations(df, events, dup_pairs, exp):
    """노트북 TEST 모드: make_test_dataset 기대값과 EDA 결과를 대조한다 (틀리면 AssertionError).
    df 열: file, kind, width, height, frames, vfr, rotation, subject_motion (+ 있으면 decoded, has_alpha, has_caption,
    caption_issues). 움직이는 WebP 처럼 확장자가 이미지인 클립도 kind='video' 행이어야 한다 (is_animated_image 로 나눔).
    events = {클립: detect_events(sig, fps, timestamps=read_frames 시각) 결과} (VFR 긴 정지는 시각을 줘야 멈춤으로 잡힘),
    dup_pairs = [(a, b, 거리), ...].
    회전 방향(빨간 표시가 네 모서리 중 기대한 곳에만)·투명 → 흰색·16비트 밝기는 exp["root"] 의 파일을 다시 읽어 화소로 본다."""
    r, root = df.set_index("file"), Path(exp["root"])
    for n in exp["images"] + exp["clips"]:
        assert n in r.index, f"{n}: EDA 표에 없음"
        assert (int(r.loc[n, "height"]), int(r.loc[n, "width"])) == tuple(exp["size"][n]), f"{n}: 표시 크기가 기대와 다름"
        want = "image" if n in exp["images"] else "video"
        assert r.loc[n, "kind"] == want, (f"{n}: {r.loc[n, 'kind']} 로 읽음 (기대 {want}"
                                          + (", 움직이는 이미지는 is_animated_image 로 클립 경로에)" if n in exp.get("animated", []) else ")"))
    for n in exp["clips"]:
        x, ev = r.loc[n], events[n]
        want = exp["events"].get(n, dict(black=[], freeze=[], cuts=[]))
        hint = " (편집 목록 밖 패킷까지 셈)" if n in exp.get("discard", {}) else ""
        assert int(x["frames"]) == exp["frames"][n], f"{n}: 프레임 수 {x['frames']} != {exp['frames'][n]}{hint}"
        assert "decoded" not in r.columns or int(x["decoded"]) == int(x["frames"]), f"{n}: 패킷 수와 디코딩 수가 다름"
        assert bool(x["vfr"]) == exp["vfr"][n] and int(x["rotation"]) == exp["rotation"].get(n, 0), f"{n}: VFR·회전 판정 오류"
        assert "has_alpha" not in r.columns or bool(x["has_alpha"]) == (n in exp["alpha"]), f"{n}: 알파 판정 오류"
        assert [ev[k] for k in ("black", "freeze", "cuts")] == [want[k] for k in ("black", "freeze", "cuts")], f"{n}: 사건 {ev}"
        assert (float(x["subject_motion"]) < STATIC_MOTION) == (n in exp["static"]), f"{n}: 정지 판정 오류"
    assert {frozenset(p[:2]) for p in dup_pairs} == {frozenset(p) for p in exp["duplicates"]}, f"중복 쌍 {dup_pairs}"
    for n in exp["portrait"]:
        assert r.loc[n, "height"] > r.loc[n, "width"], f"{n}: 세로로 보이지 않음"
    for n, box in exp["red"].items():
        a = read_frames(root / n)[0][0]
        H, W = a.shape[:2]
        corner = [(0, 16, 0, 16), (0, 16, W - 16, W), (H - 16, H, 0, 16), (H - 16, H, W - 16, W)]
        assert [c for c in corner if _is_red(a, c)] == [tuple(box)], f"{n}: 회전 방향이 틀림"
    for n, (y0, y1, x0, x1) in exp["white"].items():
        assert read_frames(root / n)[0][:, y0:y1, x0:x1].min() >= 250, f"{n}: 투명한 부분이 흰색이 아님"
    lo, hi = exp["gray16_mean"]
    assert lo < load_image(root / "img_gray16.png").mean() < hi, "16비트 이미지가 하얗거나 검게 잘림"
    for n in exp["no_caption"] if "has_caption" in r.columns else []:
        assert not r.loc[n, "has_caption"], f"{n}: 캡션이 없어야 함"
    for n, words in exp["caption_issues"].items() if "caption_issues" in r.columns else []:
        assert all(w in str(r.loc[n, "caption_issues"]) for w in words), f"{n}: 캡션 문제 {words} 를 못 찾음"


def _scene(t, kind, h=96, w=160, speed=3.0, color=(245, 245, 235)):
    """kind 마다 다른 부드러운 무늬 배경 위를 좌우로 오가는 검은 테두리 사각형 (t 번째 프레임, 끝에서 튀지 않게 왕복).
    1차원 그라데이션은 pHash 계수가 0 근처에 몰려 서명이 불안정하므로 대비가 큰 2차원 무늬를 쓴다."""
    g = np.random.default_rng(100 + kind).random((3, 5)).astype(np.float32)
    g = np.asarray(Image.fromarray(g, "F").resize((w, h), Image.Resampling.BICUBIC)).clip(0, 1)[..., None]
    img = (np.array([10, 20, 60]) * (1 - g) + np.array([240, 230, 200]) * g).astype(np.uint8)
    s = h // 3
    p = (t * speed) % (2 * (w - s))
    x0, y0 = int(min(p, 2 * (w - s) - p)), h // 2 - s // 2
    img[y0:y0 + s, x0:x0 + s] = 0                 # 테두리가 있어 배경 밝기와 상관없이 움직임이 보인다
    img[y0 + 3:y0 + s - 3, x0 + 3:x0 + s - 3] = color
    return img


def make_test_dataset(root, trigger):
    """알려진 문제를 넣은 작은 시험 데이터 (이미지 4장 + 클립 14개, 161px·60프레임 이하, PyAV·Pillow 만, 1초 안팎).
    돌려주는 기대값으로 TEST 모드가 probe·read_frames·사건 검출·중복·정지·캡션 검사를 assert 한다 (예: test_video_io.py).
      images·clips: 파일 이름 (클립에는 확장자가 이미지인 움직이는 WebP 도 있다) / frames·fps·vfr: 클립별 디코딩되는 프레임 수
      (= 편집 목록 밖을 뺀 패킷 수)·중앙값 fps·VFR 여부 / header_zero: 헤더 프레임 수 0 / discard: 편집 목록 밖이라 디코더가 버리는
      패킷 수 (ffmpeg -ss … -c copy 로 자른 mp4) / animated: 확장자는 이미지지만 움직이는 파일 (클립으로 읽어야 함)
      size: 표시 크기 (h, w) / rotation·portrait: 회전 클립·세로로 보여야 할 파일 / red: 빨간 표시가 보여야 할 영역
      (y0, y1, x0, x1), 다른 모서리에 있으면 회전 방향이 틀림 / alpha: 디코딩한 화소에 알파가 있는 파일 (GIF·움직이는 WebP 는
      늘 알파 형식) / white: 투명해서 흰색이어야 할 영역 (알파 VP9 webm 은 그 아래 빨간색이 숨어 있음)
      gray16_mean: 16비트 이미지 평균 밝기 범위 / events: black·freeze·cuts·segments (read_frames 시각을 detect_events 에 준
      결과, hold.gif 는 1.2초 떠 있는 프레임이 멈춤) / duplicates: 중복 쌍 / static: 거의 정지
      no_caption: 캡션 없는 파일 / caption_issues: 파일 → caption_issues 결과에 들어 있어야 할 문구 / fixed_words, trigger
    크기·영역은 회전을 적용한 뒤 기준."""
    need = [("libx264", "w"), ("libvpx-vp9", "w"), ("libvpx-vp9", "r")]
    missing = [f"{n} {'인코더' if m == 'w' else '디코더'}" for n, m in need if not _codec_ok(n, m)]
    if missing:
        raise RuntimeError(f"시험 데이터에 필요한 코덱이 PyAV 에 없음: {missing} (PyPI 의 av 휠에는 들어 있음)")
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(0)
    fixed = ["black bob", "wine-red cape"]
    caps = {}
    # 이미지: 보통 png, EXIF 회전 jpg (저장은 가로, 보이는 것은 세로), 16비트 흑백 png, 왼쪽 절반이 투명한 png
    blocks = Image.fromarray(rng.integers(0, 256, (12, 16, 3), dtype=np.uint8))
    blocks.resize((128, 96), Image.Resampling.NEAREST).save(root / "img_plain.png")
    a = rng.integers(0, 256, (64, 96, 3), dtype=np.uint8)
    a[:16, :16] = (255, 0, 0)                     # 저장 기준 왼쪽 위 → 보이는 기준 오른쪽 위
    exif = Image.Exif()
    exif[0x0112] = 6
    Image.fromarray(a).save(root / "img_exif.jpg", exif=exif.tobytes(), quality=95)
    Image.fromarray((np.linspace(0, 65535, 64)[None, :].repeat(80, 0)).astype(np.uint16)).save(root / "img_gray16.png")
    a = rng.integers(0, 256, (96, 96, 4), dtype=np.uint8)
    a[..., 3] = 255
    a[:, :48, 3] = 0
    Image.fromarray(a, "RGBA").save(root / "img_alpha.png")
    caps.update({"img_plain.png": f"{trigger}, standing in a sunny park, full shot",
                 "img_exif.jpg": f"{trigger}, seen from the side, close-up portrait",
                 "img_gray16.png": f"{trigger}, grayscale sketch of the character, full shot",
                 "img_alpha.png": f"{trigger}, character cutout on a transparent background"})
    # 클립
    base = [_scene(t, 0) for t in range(48)]
    _encode(root / "cfr24.mp4", base, 24, "libx264", "yuv420p")
    fr, ts = read_frames(root / "cfr24.mp4")      # 거의 같은 클립: 축소·30fps·저화질 재인코딩
    idx = resample_to_fps(ts, 30)
    _encode(root / "cfr24_dup.mp4", resize_crop_frames(fr[idx], 76, 128), 30, "libx264", "yuv420p", {"crf": "30"})
    pts = np.concatenate([[0], np.cumsum([33, 66, 100] * 10)[:-1]])       # 간격 33/66/100ms 반복
    yel = (250, 230, 60)
    _encode(root / "vfr_mp4.mp4", [_scene(t, 5) for t in range(30)], 30, "libx264", "yuv420p", pts_ms=pts)
    _encode(root / "vfr_mkv.mkv", [_scene(t, 6, color=yel) for t in range(30)], 30, "libx264", "yuv420p", pts_ms=pts)
    rot = [_scene(t, 7) for t in range(36)]
    for f in rot:
        f[:16, :16] = (255, 0, 0)                 # 저장 기준 왼쪽 위 → 반시계 90° 표시 기준 왼쪽 아래
    _encode(root / "rot90 세로.mp4", rot, 24, "libx264", "yuv420p", rotation=90)
    _encode(root / "vp9.webm", [_scene(t, 4) for t in range(33)], 24, "libvpx-vp9", "yuv420p",
            {"deadline": "realtime", "cpu-used": "8"})
    _encode(root / "odd_444.mp4", [_scene(t, 1, h=97, w=161) for t in range(25)], 16, "libx264", "yuv444p")
    gif = []
    for t in range(12):
        g = np.dstack([_scene(t, 8, speed=4), np.full((96, 160), 255, np.uint8)])
        g[:16, :16, 3] = 0                        # 투명 → 흰색으로 읽혀야 함
        gif.append(Image.fromarray(g, "RGBA"))
    gif[0].save(root / "anim.gif", save_all=True, append_images=gif[1:], duration=[40, 40, 80, 80, 120, 120] * 2,
                loop=0, disposal=2)
    b = [_scene(t, 3, speed=4, color=yel) for t in range(24, 40)]   # 장면 B (t = 24~39)
    ev = [np.zeros((96, 160, 3), np.uint8)] * 3 + [_scene(t, 2) for t in range(3, 24)]   # 검은 0~2, 장면 A 3~23
    ev += b[:8] + [b[7]] * 9 + b[8:]              # 컷 24, B 24~31, 31~40 멈춤, 41~48 이어서 움직임
    _encode(root / "events.mp4", ev, 16, "libx264", "yuv420p")
    still = _scene(0, 9, speed=0).astype(np.float32)   # 거의 정지: 그림은 그대로, 약한 잡음만 (멈춤으로 잡히지 않게)
    noisy = [np.clip(still + rng.normal(0, 3.0, still.shape), 0, 255).astype(np.uint8) for _ in range(40)]
    _encode(root / "static.mp4", noisy, 24, "libx264", "yuv420p", {"crf": "10"})
    # 재인코딩 없이 자른 클립(ffmpeg -ss … -c copy)과 같은 mp4 편집 목록: 앞 4프레임은 시각이 음수라 디코더가 버린다
    _encode(root / "trim_copy.mp4", [_scene(t, 10, speed=2) for t in range(36)], 24, "libx264", "yuv420p",
            pts_ms=(np.arange(36) - 4) * 1000 / 24)
    va = [np.dstack([_scene(t + 8, 11, speed=4), np.full((96, 160), 255, np.uint8)]) for t in range(24)]
    for a in va:
        a[:, :48] = (200, 40, 40, 0)              # 투명한 왼쪽 48px 아래 숨은 빨간색 (기본 vp9 디코더는 알파를 버려 빨갛게 나옴)
    _encode(root / "vp9_alpha.webm", va, 24, "libvpx-vp9", "yuva420p", {"deadline": "realtime", "cpu-used": "8"})
    wp = []                                       # 움직이는 WebP (ComfyUI 'Save Animated WEBP'·save_webp 처럼 손실 압축, 16fps)
    for t in range(24):
        a = np.dstack([_scene(t, 12, speed=5), np.full((96, 160), 255, np.uint8)])
        a[:16, :16] = (200, 40, 40, 0)            # 왼쪽 위 투명
        wp.append(Image.fromarray(a, "RGBA"))
    wp[0].save(root / "anim_loop.webp", format="WEBP", save_all=True, append_images=wp[1:], loop=0, quality=80, method=4,
               duration=np.diff(np.round(np.arange(25) * 1000 / 16)).astype(int).tolist())
    hold = [Image.fromarray(_scene(t, 13, speed=5)) for t in range(16)]   # VFR 긴 정지: 8번 프레임이 1.2초 떠 있음
    hold[0].save(root / "hold.gif", save_all=True, append_images=hold[1:], duration=[80] * 8 + [1200] + [80] * 7, loop=0)
    shot = f"{trigger}, medium shot, static camera,"
    caps.update({"cfr24.mp4": f"{shot} a white box slides left and right",
                 "cfr24_dup.mp4": f"{shot} a white box slides left and right",
                 "vfr_mp4.mp4": f"{trigger}, wide shot, static camera, a white box moves across a blotchy wall",
                 "rot90 세로.mp4": f"{trigger}, full shot, static camera, a white box slides down the tall frame",
                 "vp9.webm": "a white box bounces across the frame, static camera, wide shot",                 # 트리거 없음
                 "odd_444.mp4": f"{trigger} with a black bob and wine-red cape walks to the left, static camera, medium shot",
                 "anim.gif": f"{trigger}, close-up, static camera, a white box jumps in uneven steps",
                 "events.mp4": f"{shot} a white box slides right, then the scene changes to a yellow box",
                 "static.mp4": f"{trigger}, close-up, static camera, she turns her head and waves",          # 동작 vs 정지
                 "trim_copy.mp4": f"{trigger}, long shot, locked-off camera, a white box drifts to the right",
                 "vp9_alpha.webm": f"{trigger}, full body, the camera stays still while a white box bounces",
                 "anim_loop.webp": f"{trigger}, 클로즈업, 고정 카메라, 흰 상자가 오른쪽으로 미끄러진다",             # 한국어 캡션
                 "hold.gif": f"{trigger}, medium close-up, the camera is static, a white box stops and then moves on"})
    for name, text in caps.items():               # vfr_mkv.mkv 는 캡션 없음 (캡션은 같은 이름 .txt 라 이름 앞부분이 겹치면 안 됨)
        (root / name).with_suffix(".txt").write_text(text, encoding="utf-8")
    clips = ["cfr24.mp4", "cfr24_dup.mp4", "vfr_mp4.mp4", "vfr_mkv.mkv", "rot90 세로.mp4", "vp9.webm", "odd_444.mp4", "anim.gif",
             "events.mp4", "static.mp4", "trim_copy.mp4", "vp9_alpha.webm", "anim_loop.webp", "hold.gif"]
    return dict(
        root=str(root), trigger=trigger, fixed_words=fixed,
        images=["img_plain.png", "img_exif.jpg", "img_gray16.png", "img_alpha.png"], clips=clips,
        frames={"cfr24.mp4": 48, "cfr24_dup.mp4": len(idx), "vfr_mp4.mp4": 30, "vfr_mkv.mkv": 30, "rot90 세로.mp4": 36,
                "vp9.webm": 33, "odd_444.mp4": 25, "anim.gif": 12, "events.mp4": 49, "static.mp4": 40, "trim_copy.mp4": 32,
                "vp9_alpha.webm": 24, "anim_loop.webp": 24, "hold.gif": 16},
        fps={"cfr24.mp4": 24, "cfr24_dup.mp4": 30, "vfr_mp4.mp4": 1000 / 66, "vfr_mkv.mkv": 1000 / 66, "rot90 세로.mp4": 24,
             "vp9.webm": 24, "odd_444.mp4": 16, "anim.gif": 12.5, "events.mp4": 16, "static.mp4": 24, "trim_copy.mp4": 24,
             "vp9_alpha.webm": 24, "anim_loop.webp": 16, "hold.gif": 12.5},
        vfr={c: c in ("vfr_mp4.mp4", "vfr_mkv.mkv", "anim.gif", "hold.gif") for c in clips},
        header_zero=["vfr_mkv.mkv", "vp9.webm", "vp9_alpha.webm", "anim_loop.webp"],   # 헤더 프레임 수 0 (패킷으로 세야 함)
        discard={"trim_copy.mp4": 4}, animated=["anim_loop.webp"],
        size={"img_plain.png": (96, 128), "img_exif.jpg": (96, 64), "img_gray16.png": (80, 64), "img_alpha.png": (96, 96),
              "cfr24.mp4": (96, 160), "cfr24_dup.mp4": (76, 128), "vfr_mp4.mp4": (96, 160), "vfr_mkv.mkv": (96, 160),
              "rot90 세로.mp4": (160, 96), "vp9.webm": (96, 160), "odd_444.mp4": (97, 161), "anim.gif": (96, 160),
              "events.mp4": (96, 160), "static.mp4": (96, 160), "trim_copy.mp4": (96, 160), "vp9_alpha.webm": (96, 160),
              "anim_loop.webp": (96, 160), "hold.gif": (96, 160)},
        rotation={"rot90 세로.mp4": 90}, portrait=["img_exif.jpg", "rot90 세로.mp4"],
        red={"img_exif.jpg": (0, 16, 48, 64), "rot90 세로.mp4": (144, 160, 0, 16)},       # 회전 방향 확인용 빨간 표시
        alpha=["img_alpha.png", "anim.gif", "vp9_alpha.webm", "anim_loop.webp", "hold.gif"],
        white={"img_alpha.png": (0, 96, 0, 48), "anim.gif": (0, 16, 0, 16), "vp9_alpha.webm": (0, 96, 0, 40),
               "anim_loop.webp": (0, 16, 0, 16)},
        gray16_mean=(60, 200),                    # 16비트가 하얗게 잘리지 않았는지 (평균 밝기 범위)
        events={"events.mp4": dict(black=[(0, 3)], freeze=[(31, 41)], cuts=[24], segments=[(3, 24), (24, 31), (41, 49)]),
                "hold.gif": dict(black=[], freeze=[(8, 9)], cuts=[], segments=[(0, 8), (9, 16)])},
        duplicates=[("cfr24.mp4", "cfr24_dup.mp4")], static=["static.mp4"], no_caption=["vfr_mkv.mkv"],
        caption_issues={"vp9.webm": ["트리거 없음"], "odd_444.mp4": ["고정 특징"], "static.mp4": ["거의 정지"]},
    )

In [ ]:
%%writefile lora_video_tools/video_metrics.py
"""영상 검증 지표. 체크포인트 클립을 같은 시드·프롬프트의 베이스(0 단계) 클립과 비교한다 (numpy·PIL).

선택 모델 두 가지는 없으면 load() 가 이유를 출력하고 None 을 돌려준다: Dino (transformers), CCIP (onnxruntime).
  * motion_amount    : 긴 변 64px 휘도 (BOX 축소) 의 프레임 차 평균 mean|Y[t+1]-Y[t]| (0~1). 밝기 깜빡임에도 커진다 (참고용).
  * block_motion_p95 : 움직임. 단위: 원본 해상도 px/프레임. clip_metrics 의 motion. 전이마다 8x8 블록 변위의 상위 5% 평균
                       (VBench dynamic_degree 와 같은 통계) 을 내고, 클립 값은 잴 수 있는 전이의 평균.
                       0) 분석 휘도: 긴 변 224px 로 LANCZOS 축소 (MOTION_RESAMPLE, 줄이는 배율만큼 넓힌 앤티에일리어스 필터.
                          BOX 는 화면 주기 3~8px 망점을 접어 느린 팬을 몇 배로 읽었고, BILINEAR 는 주기 5~10px 망점에서
                          그랬다). 클립 대비 c (8x8 블록 표준편차의 98 백분위, 프레임 중앙값) 가 CONTRAST_REF 60 보다 낮으면
                          휘도에 60/c 배 (최대 CONTRAST_GMAX 4) 를 곱한다. 아래 문턱 tau·변위 벌점 lam 이 절대 휘도라 어둡거나
                          흐린 클립은 같은 속도가 느리게 읽혔다 (선화 x0.15 의 1px/프레임이 0). 보통 그림은 이득 1 이다
                          (실제 그림 46장의 가로·세로 화면 92개 가운데 89개, c 중앙값 78).
                       1) 정수 탐색: ±6px 과 그 바깥 ±8px 의 짝수 변위 (분석 px). 비용은 블록 평균을 뺀 SAD (밝기 변화를
                          움직임으로 보지 않음) + 변위 1px 마다 화소당 lam 1 휘도 (반복 무늬·가장자리의 먼 오정합 방지).
                          대비·노출 깜빡임은 화면 전체 표준편차 비 (GAIN_RANGE 0.6~1.6 안) 로 앞 프레임의 대비를 맞춰 지운다.
                       2) 1px 아래: 흐린 휘도 ([1,2,1] 이항) 에서 대칭 기울기 Lucas-Kanade 두 번. 시작점은 정수 최선 변위와 0
                          이고, 다듬은 뒤 남은 SAD + 변위 벌점이 작은 쪽을 쓴다.
                       3) 개구 문제: 블록 기울기 구조 텐서의 고윳값 비 λ2/λ1 이 작은 (선 하나뿐인) 블록은 변위의 선 방향 성분을
                          그 블록에서 정할 수 없으므로 법선 성분만 쓰고, 선 방향 성분은 이웃 3x3 의 모서리·무늬 블록 변위로 채운다.
                       4) 움직인 블록 = 0 변위 SAD 가 화소당 tau 2 휘도를 넘고, 다듬은 뒤 남은 SAD (그 블록의 잡음 바닥) 의
                          2 배를 넘는 블록. 어떤 변위로도 줄지 않는 변화 (프레임마다 다른 입자 잡음 — 화면 일부만이어도 —,
                          대비 깜빡임) 는 움직임이 아니다. 대비 이득 g 는 매끈한 (낮은 주파수) 시간 잡음까지 키우므로, 전이마다
                          밋밋한 블록 (두 프레임 8x8 표준편차의 작은 쪽이 하위 25%) 에서 블록 평균을 뺀 |프레임 차| 의 중앙값
                          n 으로 tau = max(2, min(2g, 2n)), lam = max(1, min(g, n)) 을 쓴다 (상한 2g·g 는 이득 전 휘도로 tau 2·
                          lam 1. 이득 1 클립은 n 을 재지 않고 그대로. 이 규칙 전에는 흐린 시간 잡음이 낀 어두운 정지 클립을 20px
                          넘게 읽었다). 무늬가 화면 전체에 있는 그림이 움직이면 밋밋한 블록도 바뀌므로, n 은 화면 전체 정수
                          변위 (위상 상관) 로 맞춘 뒤 블록마다 LK 한 번으로 설명하고 남은 차이로도 재서 작은 쪽을 쓴다 (잡음은
                          남고 움직임은 설명된다. 안 맞추면 무늬 빽빽한 어두운 팬의 1~3px/프레임을 0 으로 읽었다). 밋밋한 하위
                          25% 블록이 잡음 없는 고른 영역 (완전히 검은 레터박스 띠 등, 화면의 약 25% 넘게) 이면 n 이 0 이라 이
                          규칙이 꺼진다 (흐린 잡음 σ2 의 어두운 정지 클립을 14px 로 읽음).
                       5) 화면 표준편차가 한 전이에서 GAIN_RANGE 밖으로 바뀌면 (검은·흰 프레임, 페이드 끝, 대비가 크게 다른
                          장면 전환) 두 프레임을 견줄 수 없어 그 전이는 평균에서 뺀다 (예전 판: 정지 클립 가운데 검은 프레임
                          2장을 1.7px 움직임으로 읽음). 남은 전이가 하나도 없으면 (프레임마다 번갈아 오는 ±30% 넘는 대비·노출
                          깜빡임, 한 장 건너 검은 프레임) 대비 비가 GAIN_RANGE 안인 한 칸 건너 같은 위상의 두 프레임 (t, t+2)
                          을 견줘 변위를 2 로 나누고, 짝·홀 위상별 평균의 큰 쪽을 쓴다 (그런 짝도 없으면 표준편차가 0 이 아닌
                          전이를 모두 잰다). 모두 빼면 움직임이 0 이 되어 거짓 motion_loss 가, 대비가 안 맞는 두 프레임을 그대로
                          견주면 2~19 배로 부푼 값이 나왔다. 전이가 하나라도 남으면 남은 전이만 잰다.
                       합성 벤치 (vmh/bench 2,168 사례: 실제 선화 4장·합성 무늬 팬, 정지 선화 위 스프라이트 (화면의 5~40%),
                       832x480·480x832, 17·33프레임): 1~16px/프레임에서 참값의 0.9~1.4 배 (1px 쪽이 큼), 계열마다 속도에
                       단조, 정지·입자 (전체·일부·어두운 곳) 는 0.06px 아래, 24~32px/프레임 피사체는 0.85 배 이상. 벤치 밖
                       재검증: 어둡게 (x0.15·x0.3)·흐리게 (대비 x0.12·x0.25) 한 같은 팬은 원본의 0.8~1.0 배 (예전 판 0~1.0),
                       화면 주기 3~6.5px 망점 팬은 망점 없는 팬의 1.0~1.3 배 (예전 판 최대 5 배), 어두운 정지 클립 + 입자·x264
                       왕복 (기본 GOP)·페이드·검은 프레임은 0.1px 아래, 흐린 시간 잡음이 낀 어두운 정지 클립은 1px 아래.
                       한계: 화면의 5% 보다 작은 피사체는 상위 5% 에 움직이지 않는 블록이 섞여 작게 나온다 (속도에 단조는
                       그대로). 평평한 배경 위 회전·줌은 무늬가 있는 곳의 흐름만 잰다. 빠른 쪽은 분석 크기에 묶인다 (탐색 ±8
                       분석 px + 다듬기): 832x480 은 약 32~36px/프레임, 624x624 는 약 24px/프레임 위에서 포화한다. 화면
                       전체가 규칙 무늬이면 한 주기 어긋난 정수 해를 고를 수 있다 (주기 11~22px 줄무늬의 1px/프레임이 11px
                       로, 화면 주기 13~16px 망점의 6~8px/프레임이 약 22px 로. 날카로운 망점은 화면 주기 약 7~10px (분석
                       해상도의 나이퀴스트 근처) 에서도 생긴다. 화면 일부의 무늬는 괜찮다).
                       밋밋하고 큰 실루엣이 1px/프레임으로 들어오면 0.3~0.55 배로 작게 읽는다. 0 이나 255 로 잘린 곳이 많은
                       그림 (흰 종이 선화) 의 대비·노출 깜빡임은 전체 이득으로 다 지우지 못해, ±20% 까지는 정지 클립 0.4px
                       아래지만 번갈아 노출 ±26~30% 는 정지 1.8~2.4px·움직이는 클립 1.8~2.4 배로 읽고, 255 로 잘리는
                       번쩍임 프레임 (노출 x2) 은 한 장에 1~3px, 여러 장이면 9px 까지 움직임으로 읽는다 (둘 다 flicker_rel
                       로는 걸린다). 대비 이득은 클립 하나에 하나 (c 는 98 백분위의 프레임 중앙값) 다: 화면 블록의 약 2% 넘는
                       밝은 정지 요소 (달·등불·창·자막) 가 있으면 이득이 1 이 되어 어두운 부분의 느린 움직임을 예전처럼 작게
                       읽고, 어두운 프레임과 밝은 프레임이 반씩인 클립 (불 켜기·끄기) 은 어느 쪽이 많은지에 따라 값이 달라지며,
                       c 가 15 (= 60/4) 아래인 무늬 없는 부드러운 피사체는 상한에 걸려 0~0.6 배로 읽는다. 시간 잡음이 낀
                       어둡거나 흐린 클립은 문턱이 잡음만큼 (이득 전 앞 판 수준까지) 올라가 움직임을 작게 읽는다: 흐린 잡음
                       σ1 휘도 정도면 대비가 아주 낮은 (x0.12) 실제 영상은 8~17px/프레임에서도 0.5~0.6 배, 어두운 선화는
                       2px/프레임에서 0.2~0.4 배다 (motion_loss 기준 0.6 근처·아래). 강하게 압축한 (x264
                       crf 35) 어두운 클립은 압축 잡음까지 이득을 받아 최대 1.4 배로 크게 읽고 (crf 23 은 그대로), 키프레임
                       간격이 짧은 (GOP 4) crf 40 어두운 정지 클립은 3px 까지 읽을 수 있다 (노트북은 압축 전 프레임을 잰다).
                       1px/프레임 아래는 들쭉날쭉하다 (베이스가 MOTION_FLOOR 아래면 motion_ratio 는 None).
  * flicker          : 긴 변 32px 휘도 (BOX 축소). flicker_ratio = mean|Y[t+2]-2Y[t+1]+Y[t]| / mean|Y[t+1]-Y[t]|
                       (부드러운 움직임 < 1, 프레임마다 오가는 떨림·깜빡임 ≈ 2),
                       luma_flicker = 프레임 평균 휘도 m (0~255) 의 고역 성분 m[t] - (m[t-1]+m[t+1])/2 의 RMS (밝기 깜빡임).
  * contrast_flicker : 같은 32px 휘도의 프레임 표준편차 s (대비, 0~255) 의 고역 성분 s[t] - (s[t-1]+s[t+1])/2 의 RMS 인데,
                       제곱이 가장 큰 고역 표본 2개를 빼고 잰다 (표본 3개 이상일 때. 2프레임 이하는 0). 장면 전환 한 번은 s 의
                       계단 하나라 큰 표본이 2개 (±계단/2) 뿐이어서 지워지고, 2·4프레임 주기로 오가는 대비 깜빡임은 표본
                       대부분이 커서 남는다. 2~4프레임 디졸브나 두 번 장면 전환은 큰 표본이 4개 넘게 생겨 대비 깜빡임으로
                       걸릴 수 있다. 128 둘레로 대비만 오가는 깜빡임 (±5~15%) 은 평균 휘도가 거의 그대로라
                       luma_flicker 로는 안 보인다. clip_metrics 는 contrast_flicker 키로 내고, flicker_rel = max(luma_flicker·
                       contrast_flicker 를 각각 베이스 값 (하한 FLICKER_FLOOR) 으로 나눈 값). 밝기 쪽은 예전 flicker_rel 그대로,
                       깨끗한 클립의 대비 성분은 FLICKER_FLOOR 아래. 8프레임 주기처럼 느린 대비 변화는 약하게 잡고, 대비가
                       낮은 (어둡거나 흐린) 클립의 ±5~10% 깜빡임은 절대값이 FLICKER_FLOOR (1 휘도) 근처라 놓칠 수 있다.
  * color_stats      : 고르게 고른 8프레임의 평균 R·G·B 와 HSV 채도·명도 (0~255).
  * Dino             : facebook/dinov2-small pooler_output (layernorm 한 CLS), L2 정규화. letterbox 224 (기본 전처리의
                       가운데 자르기는 832x480 프레임의 양옆을 잘라 낸다), ImageNet 평균·표준편차. torchvision 없이.
  * subject_consistency: VBench subject_consistency 식. t>=1 평균 (max(0,cos(e_t,e_t-1)) + max(0,cos(e_t,e_0))) / 2.
  * style_drift      : 1 - 같은 번호 프레임 cos 평균. 트리거 없는(class word) 프롬프트를 베이스와 비교하면 그림체 변화.
  * CCIP             : deepghs/ccip_onnx ccip-caformer-24-randaug-pruned. imgutils.metrics.ccip 와 같은 계산 (흰 배경 RGB,
                       384x384 bilinear, CLIP 평균·표준편차, model_feat → model_metrics 차이 행렬). 차이 <= threshold
                       (metrics.json, 0.1785) 면 같은 캐릭터. 정체성은 DINO 가 아니라 CCIP 로 잰다 (DINO 는 구도 변화와 못 가름).
출처: VBench subject_consistency.py·temporal_flickering.py, imgutils metrics/ccip.py. 기준값은 remedy.THR.
"""
import json

import numpy as np
from PIL import Image

MOTION_SIDE = 224     # block_motion_p95 분석 해상도 (긴 변 px)
RING = 2              # 블록 정합 정수 탐색 ±6px 바깥에 ±8px 까지 짝수 변위 고리 (빠른 피사체, 1px 아래 다듬기가 사이를 메움)
MOTION_RESAMPLE = Image.Resampling.LANCZOS    # block_motion_p95 축소 필터 (줄이는 배율만큼 넓힌 앤티에일리어스 필터)
CONTRAST_REF = 60.0   # 클립 대비 기준 (분석 휘도 8x8 블록 표준편차의 98 백분위, 프레임 중앙값). 이보다 낮으면 휘도를 키운다
CONTRAST_GMAX = 4.0   # 대비 이득 상한 (아주 어두운 클립의 잡음·8비트 계단을 너무 키우지 않게)
GAIN_RANGE = (0.6, 1.6)   # 전이마다 화면 대비 비 (뒤 / 앞 표준편차): 이 안이면 앞 프레임에 곱해 맞추고, 밖이면 재지 않는다
MOTION_FLOOR = 1.0    # 베이스 움직임이 이보다 작으면 (px/프레임) motion_ratio 를 내지 않는다 (None)
FLICKER_FLOOR = 1.0   # flicker_rel 분모 하한 (휘도 0~255). 베이스가 매끄러우면 LoRA 가 1.5 를 넘어야 깜빡임
SAT_FLOOR = 10.0      # color_shift 채도 분모 하한 (회색에 가까운 베이스)


_W601 = np.array([0.299, 0.587, 0.114], np.float32)   # PIL 'L' 과 같은 휘도 가중치


def _empty(frames):
    """프레임이 없음 (None·빈 배열·빈 목록)"""
    return frames is None or len(frames) == 0


def _luma(frames, side, resample=Image.Resampling.BOX):
    """(T,H,W,3) uint8 → 긴 변이 side 이하인 휘도 (T,h,w) float32 0~255. 실수 그대로 축소 (8비트 반올림 잡음 없음).
    resample: BOX (기본, 깜빡임·motion_amount) 또는 MOTION_RESAMPLE (block_motion_p95). 프레임이 없으면 (0,1,1)."""
    if _empty(frames):
        return np.zeros((0, 1, 1), np.float32)
    out = []
    for f in frames:
        im = Image.fromarray(np.asarray(f, np.float32) @ _W601)          # mode 'F'
        s = side / max(im.size)
        if s < 1:
            im = im.resize((max(8, round(im.width * s)), max(8, round(im.height * s))), resample)
        out.append(np.asarray(im, np.float32))
    return np.stack(out)


def _pick(n, k):
    """0..n-1 에서 고르게 k 개 (중복 없음)"""
    return np.unique(np.linspace(0, n - 1, min(n, k)).round().astype(int))


def motion_amount(frames):
    Y = _luma(frames, 64) / 255.0
    return float(np.abs(np.diff(Y, axis=0)).mean()) if len(Y) > 1 else 0.0


def _fine(g):
    """1px 아래 다듬기용 (흐린 휘도, x 기울기, y 기울기) float32. [1,2,1]/4 분리형 흐림 (가장자리 복제) 은 선화의 날카로운
    선과 축소 계단 (aliasing) 을 눅여 선형 보간·기울기 근사가 1px 아래 변위에서도 맞게 한다."""
    p = np.pad(np.asarray(g, np.float32), 1, mode="edge")
    f = (p[:-2] + 2 * p[1:-1] + p[2:]) * 0.25
    f = (f[:, :-2] + 2 * f[:, 1:-1] + f[:, 2:]) * 0.25
    gy, gx = np.gradient(f)
    return f, gx, gy


def _bilinear(flat, W, H, yy, xx):
    """평평하게 편 (H, W) 영상 flat 의 실수 좌표 (yy, xx) 선형 보간 (밖은 가장자리)"""
    yy = np.clip(yy, 0, H - 1.001)
    xx = np.clip(xx, 0, W - 1.001)
    y0, x0 = yy.astype(np.int64), xx.astype(np.int64)
    fy, fx = (yy - y0).astype(np.float32), (xx - x0).astype(np.float32)
    i = y0 * W + x0
    a, b, c, d = flat[i], flat[i + 1], flat[i + W], flat[i + W + 1]
    top = a + (b - a) * fx
    return top + (c + (d - c) * fx - top) * fy


def _block_disp(g0, g1, block=8, radius=6, tau=2.0, lam=1.0, noise=2.0, fine0=None, fine1=None):
    """g1 의 블록마다 g0 에서 온 변위 크기 (분석 해상도 px, 블록 수 길이). fine0·fine1 = _fine(g0)·_fine(g1) (없으면 만든다).

    0) 대비 이득: g0 에 화면 전체 표준편차 비 std(g1)/std(g0) (0.6~1.6) 를 곱해 맞춘다. 밝기는 블록 평균을 빼서, 대비·노출
       깜빡임은 이 이득으로 지운다 (움직임만 있으면 이득은 1 근처).
    1) 정수 탐색 ±radius 와 그 바깥 ±(radius+RING) 의 짝수 변위 (g0 은 가장자리를 복제해 늘림, 블록 격자는 그대로):
       비용 = 블록 평균을 뺀 SAD + lam·|변위|·화소 수 (비슷한 SAD 면 짧은 변위: 반복 무늬·가장자리의 먼 오정합 방지).
       최선 SAD 가 0 변위 SAD 의 0.8 배보다 크면 0.
    2) 1px 아래 다듬기 (흐린 휘도): 정수 최선 변위와 0 두 곳에서 대칭 기울기 Lucas-Kanade 두 번 (둘째는 다듬은 자리에서
       선형 보간). 다듬은 뒤 남은 SAD + 같은 변위 벌점이 작은 쪽을 쓴다. 1px 아래로 느린 선은 정수 변위 어디에도 맞지 않아
       선을 따라 먼 정수 변위가 뽑히곤 했다 (선화 팬 2px/프레임이 10px 로 읽힘). 0 에서 다듬으면 실제 변위가 남은 SAD 도
       작고 더 짧다.
    3) 개구 문제: 블록 기울기 구조 텐서 J 의 고윳값 비 q = λ2/λ1 이 0.15 아래 (선 하나뿐인 블록) 면 변위의 선 방향 성분은
       그 블록에서 정해지지 않으므로 버리고 (법선 성분만 씀), 0.4 위 (모서리·무늬) 면 모두 쓴다. 사이는 선형으로 섞는다.
       버린 선 방향 성분은 이웃 3x3 블록 가운데 믿을 만한 블록 (q 가중) 의 평균 변위에서 채운다: 세로 선이 많은 옷이
       세로로 움직일 때 (작은 피사체·세로 화면)·회전·줌에서 법선 성분만으로는 작게 나오던 것.
    4) 움직인 블록 = 0 변위 SAD > tau·화소 수 이고, 0 변위 SAD > noise × 다듬은 뒤 남은 SAD (흐린 휘도, 블록마다).
       남은 SAD 가 그 블록의 잡음 바닥이다: 프레임마다 다른 입자 잡음 (화면 일부만이어도)·대비 깜빡임은 어떤 변위로도
       줄지 않아 움직임이 아니고, 깨끗한 움직임은 남은 SAD 가 0 근처라 그대로다. noise=0 이면 이 문턱을 끈다."""
    r, b = radius, block
    H, W = g0.shape
    nby, nbx = (H - 2 * r) // b, (W - 2 * r) // b
    if nby < 1 or nbx < 1:
        return np.zeros(1, np.float32)
    ga = np.float32(np.clip(np.std(g1) / max(float(np.std(g0)), 1e-6), *GAIN_RANGE))   # 화면 전체 대비 이득

    def zb(a):   # 블록 평균을 뺀 (nby, b, nbx, b)
        a = a.reshape(nby, b, nbx, b)
        return a - a.mean(axis=(1, 3), keepdims=True)

    rr = r + RING                                             # 바깥 고리: ±(r+2) 의 짝수 변위 (빠른 피사체)
    cand = np.array([(y, x) for y in range(-rr, rr + 1) for x in range(-rr, rr + 1)
                     if max(abs(y), abs(x)) <= r or (y % 2 == 0 and x % 2 == 0)])
    ref, g0a = zb(g1[r:r + nby * b, r:r + nbx * b]), np.pad(g0 * ga, RING, mode="edge")
    S = np.stack([np.abs(ref - zb(g0a[rr + y:rr + y + nby * b, rr + x:rr + x + nbx * b])).sum(axis=(1, 3))
                  for y, x in cand])                                                  # (후보 수, nby, nbx)
    D = np.hypot(cand[:, 0], cand[:, 1])[:, None, None]                               # 후보 변위 크기
    C = S + lam * b * b * D
    best = np.where(C <= C.min(0) + 1e-3, D, np.inf).argmin(0)                       # 비용이 같으면 작은 변위
    i0 = len(cand) // 2                                                               # (0, 0) 후보
    s00 = S[i0]
    best = np.where(np.take_along_axis(S, best[None], 0)[0] > 0.8 * s00, i0, best)   # 별로 안 나아지면 0
    out = np.zeros(nby * nbx, np.float32)
    sel = np.flatnonzero(s00.ravel() > tau * b * b)          # 0 변위에서도 거의 같은 블록은 움직이지 않은 것
    if sel.size == 0:
        return out
    iy, ix = cand[best.ravel()[sel]].T + r
    f0, gx0, gy0 = _fine(g0) if fine0 is None else fine0
    f1, gx1, gy1 = _fine(g1) if fine1 is None else fine1
    F0, GX0, GY0 = f0.ravel() * ga, gx0.ravel() * ga, gy0.ravel() * ga
    byi, bxi = np.divmod(sel, nbx)
    oy, ox = np.mgrid[0:b, 0:b]
    Py, Px = (r + byi * b)[:, None, None] + oy, (r + bxi * b)[:, None, None] + ox      # (K, b, b) 블록 화소 좌표
    lin = Py * W + Px
    R = f1.ravel()[lin]
    R = R - R.mean((1, 2), keepdims=True)
    Rgx, Rgy = gx1.ravel()[lin], gy1.ravel()[lin]
    A = F0[lin]
    s00f = np.abs(A - A.mean((1, 2), keepdims=True) - R).sum((1, 2))                   # 흐린 휘도의 0 변위 SAD

    def refine(sy, sx):
        """정수 시작 변위 (sy, sx) 에서 대칭 기울기 LK 두 번 → (dy, dx, 남은 SAD, Jxx, Jxy, Jyy)"""
        li = np.clip(Py + sy[:, None, None], 0, H - 1) * W + np.clip(Px + sx[:, None, None], 0, W - 1)
        A = F0[li]
        e = A - A.mean((1, 2), keepdims=True) - R
        Gx, Gy = 0.5 * (GX0[li] + Rgx), 0.5 * (GY0[li] + Rgy)     # 두 프레임 기울기 평균: 1차 근사가 2차까지 맞음
        Gx -= Gx.mean((1, 2), keepdims=True)
        Gy -= Gy.mean((1, 2), keepdims=True)
        Jxx, Jxy, Jyy = (Gx * Gx).sum((1, 2)), (Gx * Gy).sum((1, 2)), (Gy * Gy).sum((1, 2))
        reg = 1e-3 * (Jxx + Jyy) + 1e-6                           # 선 하나뿐인 블록: 선 방향으로는 거의 움직이지 않음
        a, c = Jxx + reg, Jyy + reg
        det = a * c - Jxy * Jxy

        def step(e):
            bx, by = (Gx * e).sum((1, 2)), (Gy * e).sum((1, 2))
            return np.clip((Jxy * bx - a * by) / det, -1, 1), np.clip((Jxy * by - c * bx) / det, -1, 1)

        ddy, ddx = step(e)
        dy, dx = sy + ddy, sx + ddx
        A = _bilinear(F0, W, H, Py + dy[:, None, None], Px + dx[:, None, None])
        e = A - A.mean((1, 2), keepdims=True) - R
        ddy, ddx = step(e)
        res = np.abs(e + Gx * ddx[:, None, None] + Gy * ddy[:, None, None]).sum((1, 2))   # 선형 근사로 다듬은 뒤 SAD
        return dy + ddy, dx + ddx, res, Jxx, Jxy, Jyy

    dy, dx, res, Jxx, Jxy, Jyy = refine(iy - r, ix - r)
    z = np.zeros(sel.size, np.int64)
    zy, zx, zres, zJxx, zJxy, zJyy = refine(z, z)
    use = zres + lam * b * b * np.hypot(zy, zx) <= res + lam * b * b * np.hypot(dy, dx)
    dy, dx, res = np.where(use, zy, dy), np.where(use, zx, dx), np.where(use, zres, res)
    Jxx, Jxy, Jyy = np.where(use, zJxx, Jxx), np.where(use, zJxy, Jxy), np.where(use, zJyy, Jyy)
    tr, disc = Jxx + Jyy, np.sqrt((Jxx - Jyy) ** 2 * 0.25 + Jxy ** 2)
    q = (0.5 * tr - disc) / np.maximum(0.5 * tr + disc, 1e-9)                         # λ2 / λ1
    th = 0.5 * np.arctan2(2 * Jxy, Jxx - Jyy)                                          # 기울기 (법선) 방향
    pn, pt = dx * np.cos(th) + dy * np.sin(th), dy * np.cos(th) - dx * np.sin(th)     # 법선·선 방향 성분
    w = np.clip((q - 0.15) / 0.25, 0, 1)                                              # 선 방향 성분을 믿는 정도
    moving = s00f > noise * res if noise > 0 else np.ones(sel.size, bool)
    # 선 블록의 선 방향 성분은 이웃 3x3 블록 가운데 믿을 만한 블록 (w 가중, 정지로 판정된 블록은 변위 0) 의 평균 변위에서
    V = np.zeros((3, nby * nbx))
    V[:, sel] = w, w * np.where(moving, dx, 0.0), w * np.where(moving, dy, 0.0)
    V = np.pad(V.reshape(3, nby, nbx), ((0, 0), (1, 1), (1, 1)))
    V = sum(V[:, i:i + nby, j:j + nbx] for i in range(3) for j in range(3)).reshape(3, -1)[:, sel]
    vx, vy = V[1] / np.maximum(V[0], 1e-9), V[2] / np.maximum(V[0], 1e-9)
    tn = vy * np.cos(th) - vx * np.sin(th)
    mag = np.sqrt(pn * pn + w * pt * pt + (1 - w) * np.minimum(V[0], 1.0) * tn * tn)
    out[sel] = np.where(moving, mag, 0.0)
    return out


def _top5(d):
    """상위 5% 평균 (VBench dynamic_degree 의 흐름 크기 통계). 움직이는 블록이 5% 근처여도 값이 무너지지 않는다."""
    k = max(1, int(np.ceil(0.05 * d.size)))
    return float(np.sort(d)[-k:].mean())


def _contrast_gain(Y):
    """클립 대비 이득 (스칼라). 대비 c = 프레임마다 8x8 블록 표준편차의 98 백분위 (가장 무늬가 짙은 블록들), 그 프레임
    중앙값. c 가 CONTRAST_REF 보다 낮으면 CONTRAST_REF / c 배 (최대 CONTRAST_GMAX), 아니면 1 (보통 그림은 그대로).
    _block_disp 의 문턱 tau·변위 벌점 lam 은 절대 휘도라, 어둡거나 흐린 클립에서는 같은 속도가 느리게 읽혔다."""
    T, h, w = Y.shape
    h8, w8 = h // 8 * 8, w // 8 * 8
    sd = Y[:, :h8, :w8].reshape(T, h8 // 8, 8, w8 // 8, 8).std(axis=(2, 4)).reshape(T, -1)
    c = float(np.median(np.percentile(sd, 98, axis=1)))
    return np.float32(np.clip(CONTRAST_REF / max(c, 1e-6), 1.0, CONTRAST_GMAX))


def _motion_luma(frames, side=MOTION_SIDE, with_gain=False):
    """block_motion_p95 의 분석 휘도: MOTION_RESAMPLE 축소 (스크린톤 무아레 방지) 에 클립 대비 이득을 곱한 것"""
    Y = _luma(frames, side, MOTION_RESAMPLE)
    g = _contrast_gain(Y) if len(Y) and min(Y.shape[1:]) >= 8 else np.float32(1.0)
    return (Y * g, float(g)) if with_gain else Y * g


def _int_shift(a, b):
    """b 가 a 를 화면 전체로 옮긴 정수 변위 (dy, dx): 위상 상관 피크"""
    h, w = a.shape
    A, B = np.fft.rfft2(a - a.mean()), np.fft.rfft2(b - b.mean())
    R = B * np.conj(A)
    c = np.fft.irfft2(R / np.maximum(np.abs(R), 1e-9), s=(h, w))
    iy, ix = np.unravel_index(int(np.argmax(c)), c.shape)
    return (iy if iy <= h // 2 else iy - h), (ix if ix <= w // 2 else ix - w)


def _lk_noise(a, b):
    """같은 크기 두 프레임: 블록마다 대칭 기울기 LK 한 번으로 설명하고 남은 프레임 차 (블록 평균 뺌) 평균 |e| 를, 두 프레임
    8x8 표준편차의 작은 쪽이 하위 25% 인 블록에서 중앙값"""
    h, w = a.shape
    h8, w8 = h // 8 * 8, w // 8 * 8
    if h8 < 16 or w8 < 16:
        return np.inf
    gya, gxa = np.gradient(a)
    gyb, gxb = np.gradient(b)
    blk = lambda x: x[:h8, :w8].reshape(h8 // 8, 8, w8 // 8, 8)
    zm = lambda x: x - x.mean(axis=(1, 3), keepdims=True)
    A, B = blk(a), blk(b)
    D = zm(B - A)
    Gx, Gy = zm(blk(0.5 * (gxa + gxb))), zm(blk(0.5 * (gya + gyb)))
    jxx, jxy, jyy = (Gx * Gx).sum((1, 3)), (Gx * Gy).sum((1, 3)), (Gy * Gy).sum((1, 3))
    bx, by = -(Gx * D).sum((1, 3)), -(Gy * D).sum((1, 3))
    reg = 1e-3 * (jxx + jyy) + 1e-6
    p, r = jxx + reg, jyy + reg
    det = p * r - jxy * jxy
    dx, dy = (r * bx - jxy * by) / det, (p * by - jxy * bx) / det
    e = D + Gx * dx[:, None, :, None] + Gy * dy[:, None, :, None]
    md = np.abs(e).mean(axis=(1, 3)).ravel()
    s = np.minimum(A.std(axis=(1, 3)), B.std(axis=(1, 3))).ravel()
    return float(np.median(md[s <= np.percentile(s, 25)]))


def _flat_noise(Y):
    """전이마다 시간 잡음 (분석 휘도): 두 프레임 8x8 표준편차의 작은 쪽이 하위 25% 인 블록에서 블록 평균을 뺀 프레임 차
    평균 |dY| 의 중앙값. 무늬가 있는 그림이 움직이면 밋밋한 블록도 바뀌므로, 화면 전체 정수 변위로 맞춘 뒤 블록마다
    LK 한 번으로 설명하고 남은 차이로도 재서 작은 쪽을 쓴다 (잡음은 남고, 움직임은 설명된다)."""
    T, h, w = Y.shape
    h8, w8 = h // 8 * 8, w // 8 * 8
    Z = Y[:, :h8, :w8].reshape(T, h8 // 8, 8, w8 // 8, 8)
    sd = Z.std(axis=(2, 4)).reshape(T, -1)
    D = np.diff(Z, axis=0)
    md = np.abs(D - D.mean(axis=(2, 4), keepdims=True)).mean(axis=(2, 4)).reshape(T - 1, -1)
    s = np.minimum(sd[:-1], sd[1:])
    q = np.percentile(s, 25, axis=1, keepdims=True)
    out = np.array([np.median(md[i][s[i] <= q[i]]) for i in range(T - 1)])
    for i in range(T - 1):
        dy, dx = _int_shift(Y[i], Y[i + 1])
        a = Y[i, max(0, -dy):h - max(0, dy), max(0, -dx):w - max(0, dx)]       # b[y, x] ~ a[y - dy, x - dx]
        b = Y[i + 1, max(0, dy):h - max(0, -dy), max(0, dx):w - max(0, -dx)]
        out[i] = min(out[i], _lk_noise(a, b))
    return out


def block_motion_p95(frames, side=MOTION_SIDE):
    if _empty(frames) or len(frames) < 2:
        return 0.0
    frames = np.asarray(frames)
    Y, g = _motion_luma(frames, side, with_gain=True)
    scale = frames.shape[2] / Y.shape[2]                      # 분석 px → 원본 px
    if min(Y.shape[1:]) < 20:                                 # 8x8 블록 ±6px 이 들어가지 않음
        return 0.0
    # 화면 대비 (표준편차) 가 한 전이에서 GAIN_RANGE 밖으로 바뀌면 (검은·흰 프레임, 페이드 끝, 대비가 다른 장면 전환)
    # 두 프레임을 견줄 수 없어 움직임을 재지 않고 평균에서 뺀다
    sd = Y.reshape(len(Y), -1).astype(np.float64).std(axis=1)
    ok = [i for i in range(len(Y) - 1) if sd[i] > 0 and GAIN_RANGE[0] <= sd[i + 1] / sd[i] <= GAIN_RANGE[1]]
    pairs = [(i, i + 1) for i in ok]
    if not ok:
        pairs = [(i, i + 2) for i in range(len(Y) - 2) if sd[i] > 0 and GAIN_RANGE[0] <= sd[i + 2] / sd[i] <= GAIN_RANGE[1]]
        pairs = pairs or [(i, i + 1) for i in range(len(Y) - 1) if sd[i] > 0]
    if not pairs:
        return 0.0
    F = {i: _fine(Y[i]) for i in sorted({j for p in pairs for j in p})}
    nz = _flat_noise(Y) if g > 1 else np.zeros(len(Y) - 1)
    out = {}
    for a, b in pairs:
        n = float(max(nz[a:b]))
        tau, lam = max(2.0, min(2.0 * g, 2.0 * n)), max(1.0, min(g, n))
        v = _top5(_block_disp(Y[a], Y[b], tau=tau, lam=lam, fine0=F[a], fine1=F[b])) / (b - a)
        out.setdefault(a % 2 if b - a == 2 else 0, []).append(v)
    return float(max(np.mean(v) for v in out.values()) * scale)


def _rms_hp(v, trim=0):
    """고역 성분 h[t] = v[t] - (v[t-1]+v[t+1])/2 의 RMS. trim: 제곱이 가장 큰 표본 trim 개를 빼고 잰다 (표본이 3개
    아래면 빼지 않음). 표본이 없으면 0."""
    h2 = (v[1:-1] - (v[:-2] + v[2:]) / 2) ** 2
    if trim and h2.size >= 3:
        h2 = np.sort(h2)[:-trim]
    return float(np.sqrt(h2.mean())) if h2.size else 0.0


def _flicker_y(Y):
    """긴 변 32px 휘도 (T,h,w) 0~1 → (flicker_ratio, luma_flicker, contrast_flicker). 3프레임 아래면 모두 0.
    luma_flicker = 프레임 평균 휘도 (밝기, 0~255) 의 고역 성분 RMS (매끈한 변화는 0 근처, 프레임마다 오가면 커짐).
    contrast_flicker = 프레임 휘도 표준편차 (대비, 0~255) 의 고역 성분 RMS 인데, 제곱이 가장 큰 표본 2개를 빼고 잰다
    (고역 표본 3개 이상일 때): 장면 전환 한 번은 계단이라 고역 표본 2개 (±계단/2) 만 크고, 대비가 프레임마다 오가는
    깜빡임은 모든 표본이 크다."""
    if len(Y) < 3:
        return 0.0, 0.0, 0.0
    d1 = np.abs(np.diff(Y, axis=0)).mean()
    d2 = np.abs(Y[2:] - 2 * Y[1:-1] + Y[:-2]).mean()
    return float(d2 / (d1 + 1e-6)), _rms_hp(Y.mean(axis=(1, 2)) * 255), _rms_hp(Y.std(axis=(1, 2)) * 255, trim=2)


def flicker(frames):
    r, lf, _ = _flicker_y(_luma(frames, 32) / 255.0)
    return dict(flicker_ratio=r, luma_flicker=lf)


def contrast_flicker(frames):
    """대비 깜빡임 (휘도 0~255): 프레임 휘도 표준편차의 고역 성분 RMS (가장 큰 2개를 뺌, _flicker_y). 128 둘레로 대비만
    오가는 깜빡임은 평균 휘도가 거의 그대로라 luma_flicker 에 안 보인다. clip_metrics 는 이 값을 contrast_flicker 키로
    내고, 베이스 값으로 나눠 flicker_rel 에 넣는다."""
    return _flicker_y(_luma(frames, 32) / 255.0)[2]


def color_stats(frames, k=8):
    if _empty(frames):
        return dict.fromkeys(("r", "g", "b", "sat", "val"), float("nan"))
    sub = np.asarray(frames)[_pick(len(frames), k)]
    rgb = sub.reshape(-1, 3).mean(0)
    hsv = np.stack([np.asarray(Image.fromarray(np.ascontiguousarray(f)).convert("HSV"), np.float32) for f in sub])
    return dict(r=float(rgb[0]), g=float(rgb[1]), b=float(rgb[2]), sat=float(hsv[..., 1].mean()), val=float(hsv[..., 2].mean()))


def _color_shift(c, cb):
    """채도 상대 변화와 '평균색 거리 / 베이스 평균색 크기' 가운데 큰 값 (0.10 = 10%)."""
    sat = abs(c["sat"] - cb["sat"]) / max(cb["sat"], SAT_FLOOR)
    a, b = (np.array([d["r"], d["g"], d["b"]]) for d in (c, cb))
    return float(max(sat, np.linalg.norm(a - b) / max(np.linalg.norm(b), 1.0)))


class Dino:
    """프레임 임베딩 (T, 384). Dino.load(device) 가 None 이면 (설치·내려받기 실패) 시간 일관성·그림체 지표 없이 진행한다."""
    NAME = "facebook/dinov2-small"
    MEAN = np.array([0.485, 0.456, 0.406], np.float32)
    STD = np.array([0.229, 0.224, 0.225], np.float32)
    PAD = (124, 116, 104)     # ImageNet 평균색: 정규화하면 0

    def __init__(self, model, device):
        self.model, self.device = model, device

    @classmethod
    def load(cls, device="cpu", name=NAME):
        try:
            import torch
            from transformers import Dinov2Model
            model = Dinov2Model.from_pretrained(name, dtype=torch.float32).to(device).eval()
        except Exception as e:      # transformers·torch 없음, 내려받기 실패 등
            print(f"DINOv2 지표를 쓰지 않습니다 ({name}): {type(e).__name__}: {str(e)[:300]}")
            return None
        return cls(model, device)

    @classmethod
    def letterbox(cls, img, size=224):
        """비율을 지키며 긴 변을 size 로 줄이고 남는 곳을 평균색으로 채운다 (프레임 전체가 남는다)."""
        im = img if isinstance(img, Image.Image) else Image.fromarray(np.ascontiguousarray(img))
        im = im.convert("RGB")
        s = size / max(im.size)
        im = im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.Resampling.BICUBIC)
        canvas = Image.new("RGB", (size, size), cls.PAD)
        canvas.paste(im, ((size - im.width) // 2, (size - im.height) // 2))
        return np.asarray(canvas)

    def embed(self, frames, batch=32):
        """(T,H,W,3) uint8 또는 그림 목록 (크기가 달라도 됨) → (T, 384) float32, 행마다 L2 정규화. 없으면 (0, 384)."""
        if _empty(frames):
            return np.zeros((0, getattr(getattr(self.model, "config", None), "hidden_size", 384)), np.float32)
        import torch
        out = []
        with torch.inference_mode():
            for i in range(0, len(frames), batch):
                x = np.stack([(self.letterbox(f) / 255.0 - self.MEAN) / self.STD for f in frames[i:i + batch]])
                x = torch.from_numpy(x.transpose(0, 3, 1, 2).astype(np.float32)).to(self.device)
                e = self.model(pixel_values=x).pooler_output.float()
                out.append(torch.nn.functional.normalize(e, dim=-1).cpu().numpy())
        return np.concatenate(out)


def _unit(E):
    """(T, D) 행마다 L2 정규화. None·빈 목록은 (0, 1), 벡터 하나 (D,) 는 (1, D)."""
    E = np.asarray([] if E is None else E, np.float32)
    if E.ndim == 1:
        E = E[None] if E.size else np.zeros((0, 1), np.float32)
    return E / np.maximum(np.linalg.norm(E, axis=1, keepdims=True), 1e-12)


def subject_consistency(E):
    E = _unit(E)
    if len(E) < 2:
        return 1.0
    prev = np.clip((E[1:] * E[:-1]).sum(1), 0, None)
    first = np.clip(E[1:] @ E[0], 0, None)
    return float(((prev + first) / 2).mean())


def style_drift(E_a, E_b):
    """한쪽이라도 비면 NaN"""
    a, b = _unit(E_a), _unit(E_b)
    n = min(len(a), len(b))
    if n == 0:
        return float("nan")
    return float(1.0 - (a[:n] * b[:n]).sum(1).mean())


class CCIP:
    """애니 캐릭터 정체성. CCIP.load() 가 None 이면 (onnxruntime 없음·내려받기 실패) 정체성 지표 없이 진행한다."""
    REPO = "deepghs/ccip_onnx"
    MODEL = "ccip-caformer-24-randaug-pruned"
    MEAN = np.array([0.48145466, 0.4578275, 0.40821073], np.float32)[:, None, None]
    STD = np.array([0.26862954, 0.26130258, 0.27577711], np.float32)[:, None, None]

    def __init__(self, feat, metric, threshold):
        self.feat, self.metric, self.threshold = feat, metric, threshold

    @classmethod
    def load(cls, model=MODEL, providers=("CPUExecutionProvider",)):
        try:
            import onnxruntime as ort
        except ImportError as e:
            print(f"CCIP 정체성 지표를 쓰지 않습니다: onnxruntime 이 없습니다 ({e}). 쓰려면 pip install onnxruntime")
            return None
        except Exception as e:      # 깨진 설치: DLL 을 못 찾는 OSError 등
            print(f"CCIP 정체성 지표를 쓰지 않습니다: onnxruntime 을 불러오지 못했습니다 ({type(e).__name__}: {str(e)[:300]})."
                  " 다시 설치하려면 pip install --force-reinstall onnxruntime")
            return None
        try:
            from huggingface_hub import hf_hub_download
            path = lambda f: hf_hub_download(cls.REPO, f"{model}/{f}")
            feat = ort.InferenceSession(path("model_feat.onnx"), providers=list(providers))
            metric = ort.InferenceSession(path("model_metrics.onnx"), providers=list(providers))
            with open(path("metrics.json"), encoding="utf-8") as f:
                threshold = float(json.load(f)["threshold"])
        except Exception as e:
            print(f"CCIP 정체성 지표를 쓰지 않습니다 ({cls.REPO}/{model}): {type(e).__name__}: {str(e)[:300]}")
            return None
        return cls(feat, metric, threshold)

    @classmethod
    def _prep(cls, img):
        im = img if isinstance(img, Image.Image) else Image.fromarray(np.ascontiguousarray(img))
        if im.mode in ("RGBA", "LA", "PA") or (im.mode == "P" and "transparency" in im.info):   # 투명 → 흰 배경
            im = im.convert("RGBA")
            im = Image.alpha_composite(Image.new("RGBA", im.size, (255, 255, 255, 255)), im)
        x = np.asarray(im.convert("RGB").resize((384, 384), Image.Resampling.BILINEAR), np.float32)
        return (x.transpose(2, 0, 1) / 255.0 - cls.MEAN) / cls.STD

    def features(self, images, batch=16):
        """그림 목록 (PIL 또는 uint8 배열, (N,H,W,3) 배열도 됨) → (N, 768). 없으면 (0, 768)."""
        images = [] if images is None else images
        out = [self.feat.run(["output"], {"input": np.stack([self._prep(im) for im in images[i:i + batch]]).astype(np.float32)})[0]
               for i in range(0, len(images), batch)]
        return np.concatenate(out) if out else np.zeros((0, 768), np.float32)

    def differences(self, feats):
        """쌍별 차이 행렬 (작을수록 닮음)"""
        return self.metric.run(["output"], {"input": np.asarray(feats, np.float32)})[0]

    def identity(self, frames, ref_feats, max_frames=16):
        """프레임마다 가장 가까운 참조 그림과의 차이 d. id_pass = mean(d <= threshold), id_mean = mean(d),
        id_slope = 프레임 번호에 대한 d 의 기울기 (+ 면 클립이 진행될수록 덜 닮아 감). 프레임·참조가 없으면 모두 None."""
        if _empty(frames) or ref_feats is None or len(ref_feats) == 0:
            return dict(id_pass=None, id_mean=None, id_slope=None)
        idx = _pick(len(frames), max_frames)
        f = self.features([frames[i] for i in idx])
        d = self.differences(np.concatenate([f, np.asarray(ref_feats, np.float32)]))[:len(idx), len(idx):].min(1)
        slope = float(np.polyfit(idx, d, 1)[0]) if len(idx) > 2 else 0.0
        return dict(id_pass=float((d <= self.threshold).mean()), id_mean=float(d.mean()), id_slope=slope)


def clip_metrics(frames, base=None, dino=None, ccip=None, ref_feats=None):
    """클립 하나의 지표 (평평한 dict, 못 잰 값은 None).

    base: 같은 시드·프롬프트(·시작 이미지)로 만든 베이스 클립 프레임. *_ratio·*_rel·color_shift·drift 는 base 가 있어야 나온다.
    luma_flicker·contrast_flicker: 이 클립의 밝기·대비 깜빡임 (휘도 0~255, _flicker_y). 어느 쪽이 깜빡였는지 표에서 본다.
    flicker_rel = max(밝기: luma_flicker / max(베이스 luma_flicker, FLICKER_FLOOR),
                      대비: contrast_flicker / max(베이스 contrast_flicker, FLICKER_FLOOR)). 밝기 쪽은 예전 flicker_rel 그대로.
    drift·color_shift 는 트리거 없는(class word) 프롬프트에서 그림체 변화로 읽는다 (트리거 프롬프트는 캐릭터가 바뀌는 게 정상).
    ref_feats: ccip.features(참조 캐릭터 그림들).
    frames 가 None·빈 것이면 motion 0.0 에 나머지는 모두 None (생성 실패 등). 빈 base 는 None 과 같다.
    """
    if _empty(frames):
        return dict(motion=0.0, motion_ratio=None, luma_flicker=None, contrast_flicker=None, flicker_rel=None,
                    flicker_ratio=None, sat=None, color_shift=None, consistency=None, drift=None, id_pass=None,
                    id_mean=None, id_slope=None)
    if base is not None and len(base) == 0:
        base = None
    frames = np.asarray(frames)
    fr, lf, cf = _flicker_y(_luma(frames, 32) / 255.0)
    col = color_stats(frames)
    E = dino.embed(frames) if dino is not None else None
    out = dict(motion=block_motion_p95(frames), motion_ratio=None, luma_flicker=lf, contrast_flicker=cf, flicker_rel=None,
               flicker_ratio=fr, sat=col["sat"], color_shift=None,
               consistency=subject_consistency(E) if E is not None else None, drift=None,
               id_pass=None, id_mean=None, id_slope=None)
    if base is not None:
        base = np.asarray(base)
        bm = block_motion_p95(base)
        out["motion_ratio"] = out["motion"] / bm if bm >= MOTION_FLOOR else None
        _, blf, bcf = _flicker_y(_luma(base, 32) / 255.0)
        out["flicker_rel"] = max(lf / max(blf, FLICKER_FLOOR), cf / max(bcf, FLICKER_FLOOR))
        out["color_shift"] = _color_shift(col, color_stats(base))
        if E is not None:
            out["drift"] = style_drift(E, dino.embed(base))
    if ccip is not None and ref_feats is not None and len(ref_feats) and len(frames):
        out.update(ccip.identity(frames, ref_feats))
    return out

In [ ]:
%%writefile lora_video_tools/remedy.py
"""보완: 체크포인트 지표 → 증상 → 다음 실행의 FIXED 덮어쓰기 → run_many 형식 계획.

조사 노트 (영상 데이터 5절) 의 '증상 → 원인 → 조정표' 를 옮겼다. 기준값·조정 폭은 판단값이라 첫 파일럿에서 고친다.
  * step_metrics[step]: motion_ratio, flicker_rel, color_shift, drift_rel, id_pass, id_slope, memo, consistency
    (모두 선택, 못 잰 값·없는 키는 None 으로 본다). 0 단계(베이스)는 보지 않는다. consistency 는 표시용 (규칙 없음).
    color_shift·drift_rel 은 트리거 없는(class word) 클립에서 잰 값이다 (트리거 클립은 캐릭터 색이 바뀌는 게 정상).
  * val_loss: {step: 홀드아웃(학습에 안 쓴) 샘플의 고정 검증 손실}. 학습 샘플로 잰 손실은 암기해도 계속 내려가 과적합을
    못 보므로 None 으로 넘긴다. 그러면 손실 규칙은 모두 건너뛰고, 과적합은 암기(memo)·고른 단계 뒤 악화(그림체 드리프트 등)
    로만 보며, 추천 단계는 품질 지표로 고른다 (못 고르면 마지막 단계).
  * 판정: 점 지표 (움직임·깜빡임·색·그림체·클립 안 정체성) 는 pick_best_step 이 고른 단계에서, 정체성 추세
    (underfit / identity_weak) 와 과적합 (검증 손실 상승·암기·고른 단계 뒤 악화) 은 단계 전체에서 본다.
    입력이 없는 규칙은 건너뛴다.
  * 한 실행에 2개까지만 바꾼다 (효과를 가려내려고). PRIORITY 순서로, ALLOWED 범위로 자르고, 그대로인 값과 fixed 에 없는
    키는 건너뛴다 (계열이 쓰지 않는 키는 호출하는 쪽이 fixed 에서 뺀다). rank 를 바꾸면 alpha 도 같은 값
    (ComfyUI 는 alpha 없는 LoRA 를 alpha = rank 로 적용).
  * 다음 실행 계획 (next_run_plan) 은 이번 실행의 덮어쓰기 (base_overrides) 위에 이번 제안을 얹는다: 앞선 보완이 이어지고,
    이번 실행과 다른 값은 이번 제안 (2개까지) 뿐이다.
  * 움직임 손실의 target_scope "kvo": 정지 이미지로 어텐션 Q/K/V/O 를 학습하면 움직임이 줄고 K/V/O 만 학습하면 유지됐다
    (V-Warper, arXiv 2512.12375, CogVideoX-5B). Wan 에서는 실측 전.
자동 제안은 사람이 컨택트 시트를 보고 승인한 뒤 실행한다.
"""
import math
import numbers
import re
from collections.abc import Mapping
from pathlib import Path

THR = dict(
    id_pass=0.90,             # CCIP 통과율 (같은 캐릭터로 판정된 프레임 비율) 하한
    id_rise=0.02,             # 마지막 두 체크포인트 사이 통과율이 이보다 더 오르면 아직 배우는 중 (underfit)
    id_slope=0.002,           # 클립 안에서 CCIP 차이가 프레임마다 이보다 더 커지면 정체성 흔들림
    motion_low=0.6,           # 베이스 대비 움직임 비율 하한·상한
    motion_high=1.6,
    best_motion=(0.7, 1.3),   # pick_best_step 이 받아들이는 움직임 비율
    flicker_rel=1.5,          # 베이스 대비 깜빡임 (밝기 luma_flicker·대비 contrast_flicker 비율 가운데 큰 쪽)
    color_shift=0.10,         # class-word 클립의 채도·평균색 상대 변화 (베이스 대비)
    drift_rel=2.0,            # class-word 그림체 드리프트 / 베이스 시드 간 변동
    memo=0.95,                # 새 프롬프트 프레임과 가장 가까운 학습 프레임의 DINOv2 cos (암기)
    val_rise=0.02,            # 홀드아웃 검증 손실이 최저점보다 2% 넘게 오르면 과적합 (val_loss 가 None 이면 안 봄)
)
# min_motion 단위 = video_io subject_motion: 초당 화면 긴 변의 비율 (0.05 = 1초에 긴 변의 5%).
# 0.05 는 video_io.STATIC_MOTION 과 같은 값 = 노트북 EDA 의 '거의 정지한 클립' 기준 (832x480·16fps 에서 약 2.6px/프레임 아래)
MIN_MOTION_ON = 0.05
# 숫자 키는 (하한, 상한), 나머지는 고를 수 있는 값.
# min_motion 상한 0.2 (832x480·16fps 에서 약 10px/프레임): 그보다 높이면 보통 움직임 클립까지 빠진다
ALLOWED = dict(
    max_train_steps=(250, 6000), learning_rate=(1e-5, 3e-4), rank=(4, 64), low_steps_share=(0.5, 1.0),
    shift_low=(1.0, 5.0), shift_high=(3.0, 12.0), shift=(1.0, 12.0), image_share=(0.0, 1.0),
    high_uses_images=(False, True), min_motion=(0.0, 0.2), min_frames=(17, 81), caption_dropout=(0.0, 0.2),
    preservation=("off", "dop"), target_scope=("all", "attn", "kvo"),
)
_up = lambda v, x, cap: max(v, min(x, cap))   # x 로 올리되 cap 까지 (이미 cap 위면 그대로, 내리지 않음)
_rank_up = lambda v: _up(v, v * 2, 32)
_low_up = lambda v: _up(v, v + 0.1, 0.9)
# 증상 → [(키, 지금 값 → 다음 값)], 앞에서부터 쓴다
PLAYBOOK = {
    "overfit": [("max_train_steps", lambda v: int(v * 0.6)), ("learning_rate", lambda v: v * 0.5),
                ("caption_dropout", lambda v: max(v, 0.05)), ("preservation", lambda v: "dop")],
    "underfit": [("max_train_steps", lambda v: int(v * 1.5)), ("rank", _rank_up),
                 ("learning_rate", lambda v: _up(v, v * 1.5, 2e-4))],
    "identity_weak": [("low_steps_share", _low_up), ("rank", _rank_up),
                      ("shift_low", lambda v: v - 1.0), ("shift", lambda v: v - 1.0)],   # 낮은 shift = 저노이즈(디테일) 쪽
    "motion_loss": [("image_share", lambda v: v - 0.2),     # 섞인 데이터에서 클립을 더 뽑기 (이미지뿐이면 노트북이 이 키를 빼고 넘김)
                    ("high_uses_images", lambda v: False),                   # 이미지는 저노이즈 전문가만
                    ("target_scope", lambda v: "kvo"),                       # V-Warper: K/V/O 만 → 움직임 유지
                    ("min_motion", lambda v: max(v, MIN_MOTION_ON)),         # 거의 정지한 클립 빼기
                    ("shift_high", lambda v: v + 2.0), ("shift", lambda v: v + 1.0)],
    "identity_drift": [("min_frames", lambda v: v + 16), ("low_steps_share", _low_up), ("rank", _rank_up)],
    "flicker": [("learning_rate", lambda v: v * 0.5), ("min_frames", lambda v: v + 16)],
    "color_shift": [("learning_rate", lambda v: v * 0.7), ("preservation", lambda v: "dop")],
    "style_drift": [("preservation", lambda v: "dop"), ("max_train_steps", lambda v: int(v * 0.8))],
    "motion_excess": [("learning_rate", lambda v: v * 0.7), ("low_steps_share", _low_up)],
}
PRIORITY = ["overfit", "underfit", "identity_weak", "motion_loss", "identity_drift", "flicker", "color_shift",
            "style_drift", "motion_excess"]
MAX_CHANGES = 2
_SEV = ("low", "medium", "high")


def _num(v):
    """숫자로 읽을 수 있고 유한하면 float, 아니면 None"""
    try:
        v = float(v)
    except (TypeError, ValueError):
        return None
    return v if math.isfinite(v) else None


def _step(s):
    """단계 키 (JSON 에서 읽은 '250' 도 됨). 정수가 아니면 None"""
    v = _num(s)
    return int(v) if v is not None and v == int(v) else None


def _steps(step_metrics):
    """{단계: 지표 dict} (0 단계·정수가 아닌 키는 뺌, dict 가 아닌 값은 빈 dict)"""
    return {k: (dict(m) if isinstance(m, Mapping) else {}) for s, m in (step_metrics or {}).items()
            if (k := _step(s)) is not None and k > 0}


def _losses(val_loss):
    return {k: v for s, x in (val_loss or {}).items() if (k := _step(s)) is not None and (v := _num(x)) is not None}


def _thr(thr):
    """기준값: 빠진 키는 THR 값으로 채운다"""
    return {**THR, **(thr or {})}


def _sev(ratio):
    """기준 대비 몇 배 나쁜가 → 정도"""
    return "high" if ratio >= 2.0 else "medium" if ratio >= 1.25 else "low"


def _point(m, thr):
    """한 체크포인트의 점 지표 판정: {이름: (근거, 정도)}. identity·memo 는 과적합 근거로만 쓴다."""
    g = lambda k: _num(m.get(k))
    r, f, c, d, sl, mm, p = (g(k) for k in ("motion_ratio", "flicker_rel", "color_shift", "drift_rel", "id_slope", "memo", "id_pass"))
    out = {}
    if r is not None and r < thr["motion_low"]:
        out["motion_loss"] = (f"베이스 대비 움직임 {r:.2f} < {thr['motion_low']}", _sev(thr["motion_low"] / max(r, 1e-6)))
    if r is not None and r > thr["motion_high"]:
        out["motion_excess"] = (f"베이스 대비 움직임 {r:.2f} > {thr['motion_high']}", _sev(r / thr["motion_high"]))
    if f is not None and f > thr["flicker_rel"]:
        out["flicker"] = (f"베이스 대비 밝기·대비 깜빡임 {f:.2f}배 > {thr['flicker_rel']}", _sev(f / thr["flicker_rel"]))
    if c is not None and c > thr["color_shift"]:
        out["color_shift"] = (f"class-word 채도·평균색 변화 {c:.1%} > {thr['color_shift']:.0%}", _sev(c / thr["color_shift"]))
    if d is not None and d > thr["drift_rel"]:
        out["style_drift"] = (f"class-word 그림체 변화가 시드 변동의 {d:.2f}배 > {thr['drift_rel']}", _sev(d / thr["drift_rel"]))
    if sl is not None and sl > thr["id_slope"]:
        out["identity_drift"] = (f"클립 안 CCIP 차이가 프레임마다 {sl:.4f} 증가 > {thr['id_slope']}", _sev(sl / thr["id_slope"]))
    if mm is not None and mm > thr["memo"]:
        out["memo"] = (f"학습 프레임과 DINOv2 cos {mm:.3f} > {thr['memo']} (암기)", _sev((1 - thr["memo"]) / max(1 - mm, 1e-6)))
    if p is not None and p < thr["id_pass"]:
        out["identity"] = (f"CCIP 통과율 {p:.2f} < {thr['id_pass']}", _sev(thr["id_pass"] / max(p, 1e-6)))
    return out


_QUALITY = ("motion_ratio", "flicker_rel", "drift_rel")


def _quality_ok(m, thr):
    r, f, d = (_num(m.get(k)) for k in _QUALITY)
    lo, hi = thr["best_motion"]
    return (r is None or lo <= r <= hi) and (f is None or f <= thr["flicker_rel"]) and (d is None or d <= thr["drift_rel"])


def pick_best_step(step_metrics, val_loss, thr=THR, after=None):
    """추천 체크포인트 단계 (없으면 None). 후보는 step_metrics 의 단계 (0 제외), 없으면 val_loss 의 단계.

    정체성(id_pass)을 쟀으면: 정체성 통과 + 품질 (움직임 비율 best_motion 안, 깜빡임·그림체 기준 안) 인 가장 이른 단계 →
    정체성만 통과한 가장 이른 단계 → (통과 단계 없음) 정체성이 가장 높은 단계 (품질 통과 단계 우선).
    after: A14B 처럼 전문가를 차례로 학습할 때 첫 전문가의 마지막 단계 (전문가가 하나면 None). 정체성으로 고를 때 그 뒤
    단계 (두 전문가의 LoRA 를 함께 쓴 미리보기) 에 정체성·품질 통과 단계가 있으면 그 가운데 가장 이른 단계를 먼저 고른다.
    정체성은 첫 전문가(저노이즈)가 배우므로, 이것이 없으면 학습이 잘될수록 뒤 전문가 LoRA 가 추천에서 빠진다.
    정체성을 안 쟀으면: '가장 이른 단계' 는 덜 배운 단계일 뿐이라, 품질 통과 단계 가운데 검증 손실이 가장 낮은 단계
    (같으면 늦은 단계, 검증 손실이 없으면 가장 늦은 단계). 품질 통과 단계도 없으면 검증 손실 최저, 그것도 없으면 마지막 단계.
    A14B 처럼 전문가를 차례로 학습하면 노트북이 첫 전문가의 마지막 검증 손실을 뒤 단계에 이어 붙이므로, 같은 손실이면
    뒤 전문가까지 학습한 단계를 고른다.
    val_loss 는 홀드아웃 손실일 때만 넘긴다 (None 이면 품질 지표만으로 고르고, 못 고르면 마지막 단계).
    이 실행에서 한 번도 못 잰 지표는 보지 않는다. 다른 단계에서는 잰 지표가 빠진 단계는 품질 통과로 치지 않는다.
    """
    thr = _thr(thr)
    sm, vl = _steps(step_metrics), _losses(val_loss)
    steps = sorted(sm) or sorted(s for s in vl if s > 0)
    if not steps:
        return None

    def by_loss(ss):
        known = [s for s in ss if s in vl]
        return min(known, key=lambda s: (vl[s], -s)) if known else max(ss)

    keys = [k for k in _QUALITY if any(_num(m.get(k)) is not None for m in sm.values())]
    full = [s for s in steps if all(_num(sm.get(s, {}).get(k)) is not None for k in keys)] or steps
    q = [s for s in full if _quality_ok(sm.get(s, {}), thr)]
    ids = {s: _num(sm.get(s, {}).get("id_pass")) for s in steps}
    measured = [s for s in steps if ids[s] is not None]
    if not measured:
        return by_loss(q or full)
    ok = [s for s in measured if ids[s] >= thr["id_pass"]]
    if ok:
        both = [s for s in ok if s in q and after is not None and s > after]
        return (both or [s for s in ok if s in q] or ok)[0]
    pool = [s for s in q if ids[s] is not None] or measured
    return max(pool, key=lambda s: (ids[s], -s))


def diagnose(step_metrics, val_loss, thr=THR, after=None):
    """증상 목록 [{symptom, evidence, severity}] (PRIORITY 순서). severity: low / medium / high (기준 대비 1.25배·2배).
    val_loss=None (홀드아웃 손실 없음) 이면 손실 근거는 쓰지 않는다: 과적합은 암기·고른 단계 뒤 악화로만 본다.
    after 는 pick_best_step 과 같다 (점 지표를 추천 단계에서 보도록 같은 값을 넘긴다)."""
    thr = _thr(thr)
    sm, vl = _steps(step_metrics), _losses(val_loss)
    best = pick_best_step(step_metrics, val_loss, thr, after)
    out = []
    add = lambda name, ev, sev: out.append(dict(symptom=name, evidence=ev, severity=sev))

    # 정체성 추세: 어느 단계도 기준에 못 미쳤으면 마지막 구간이 오르는 중이면 underfit, 아니면 identity_weak
    ids = [(s, _num(sm[s].get("id_pass"))) for s in sorted(sm)]
    ids = [(s, p) for s, p in ids if p is not None]
    underfit = False
    if ids and max(p for _, p in ids) < thr["id_pass"]:
        s1, p1 = ids[-1]
        rise = p1 - ids[-2][1] if len(ids) > 1 else 0.0
        sev = _sev(thr["id_pass"] / max(p1, 1e-6))
        if rise > thr["id_rise"]:
            underfit = True
            add("underfit", f"CCIP 통과율 {p1:.2f} < {thr['id_pass']} 인데 마지막 구간 {rise:+.2f} 오르는 중 ({s1}단계)", sev)
        else:
            add("identity_weak", f"CCIP 통과율 최고 {max(p for _, p in ids):.2f} < {thr['id_pass']}, "
                                 f"마지막 구간 {rise:+.2f} ({s1}단계)", sev)

    pts = _point(sm.get(best, {}), thr) if best is not None else {}

    # 과적합: 검증 손실 상승 (정체성이 아직 오르는 중이면 보지 않음), 암기, 고른 단계 뒤 악화
    ev = []
    if len(vl) >= 2 and not underfit:
        s_min, s_last = min(vl, key=lambda s: (vl[s], s)), max(vl)
        # 이어 붙인 값 (A14B: 첫 전문가의 마지막 손실을 뒤 단계에 복사) 은 처음 잰 단계로 적는다
        s_last = min((s for s in vl if s > s_min and vl[s] == vl[s_last]), default=s_last)
        rise = (vl[s_last] - vl[s_min]) / max(abs(vl[s_min]), 1e-12)
        if s_min < s_last and rise > thr["val_rise"]:
            ev.append((f"검증 손실 {s_min}단계 {vl[s_min]:.4f} → {s_last}단계 {vl[s_last]:.4f} ({rise:+.1%})",
                       _sev(rise / thr["val_rise"])))
    if "memo" in pts:
        ev.append((f"{best}단계 {pts['memo'][0]}", pts["memo"][1]))
    last = max((s for s, m in sm.items() if any(_num(v) is not None for v in m.values())), default=None)   # 잰 값이 있는 마지막 단계
    if best is not None and last is not None and best < last:
        later = _point(sm[last], thr)
        worse = [k for k in later if k not in pts]
        if worse:
            ev.append((f"{best}단계 뒤 나빠짐 ({last}단계: " + "; ".join(later[k][0] for k in worse) + ")", "medium"))
    if ev:
        add("overfit", " / ".join(e for e, _ in ev), max((s for _, s in ev), key=_SEV.index))

    for name in ("identity_drift", "motion_loss", "motion_excess", "flicker", "color_shift", "style_drift"):
        if name in pts:
            add(name, f"{best}단계: {pts[name][0]}", pts[name][1])
    return sorted(out, key=lambda s: PRIORITY.index(s["symptom"]))


def _isnum(v):
    return isinstance(v, numbers.Real) and not isinstance(v, bool)


def _clamp(key, v, allowed):
    """허용 범위로 자르고 숫자는 6자리로 다듬는다. 고를 수 있는 값이 아니면 None."""
    lim = allowed.get(key)
    if lim is None:
        return v
    if len(lim) == 2 and all(_isnum(x) for x in lim):
        v = min(max(v, lim[0]), lim[1])
        return int(v) if isinstance(v, numbers.Integral) else float(f"{float(v):.6g}")
    return v if v in lim else None


def propose(fixed, symptoms, allowed=ALLOWED, max_changes=MAX_CHANGES):
    """증상 → (fixed_overrides, changes). changes: [{key, old, new, symptom, evidence}].

    계열에서 쓰지 않는 키 (전문가가 하나인 계열의 low_steps_share·shift_low·shift_high·high_uses_images, I2V 의
    image_share 등) 는 fixed 에서 빼고 넘긴다. 없는 키·None·종류가 맞지 않는 값은 건너뛰므로 바꿀 기회를 헛쓰지 않는다.
    fixed·symptoms 가 None 이면 빈 것으로 본다.
    """
    found = {}
    for s in symptoms or ():
        if isinstance(s, Mapping) and s.get("symptom") in PLAYBOOK:
            found.setdefault(s["symptom"], s)
    changes, cur = [], dict(fixed or {})
    for name in (p for p in PRIORITY if p in found):
        for key, fn in PLAYBOOK.get(name, ()):
            if len(changes) >= max_changes:
                break
            if cur.get(key) is None or any(c["key"] == key for c in changes):
                continue
            old = cur[key]
            try:
                raw = fn(old)
                new = _clamp(key, raw, allowed)
            except (TypeError, ValueError):                # 종류가 다른 값 (예: 문자열 단계 수) → 건너뜀
                continue
            if new is None or new == old or (_isnum(old) and _isnum(raw) and (raw - old) * (new - old) <= 0):
                continue                                  # 그대로이거나, 범위 때문에 거꾸로 바뀌는 값 (예: 4단계 → 하한 250)
            cur[key] = new
            changes.append(dict(key=key, old=old, new=new, symptom=name, evidence=found[name].get("evidence", "")))
    overrides = {c["key"]: c["new"] for c in changes}
    if "rank" in overrides:
        overrides["alpha"] = overrides["rank"]
    return overrides, changes


def next_run_name(name):
    """niki_wan_v1 → niki_wan_v2, niki_v01 → niki_v02 (자릿수 유지), niki_v9 → niki_v10, niki → niki_v2"""
    m = re.fullmatch(r"(.*[_-][vV])(\d+)", name)
    return f"{m.group(1)}{int(m.group(2)) + 1:0{len(m.group(2))}d}" if m else f"{name}_v2"


def _abs(p):
    """지금 작업 폴더 기준 절대 경로 (빈 값은 그대로)"""
    return str(Path(p).expanduser().resolve()) if p else p


def next_run_plan(user, overrides, changes, best_step, base_overrides=None):
    """run_many.py 목록 형식 {"defaults": {}, "runs": [실행 1개]}. USER 값을 옮기고 실행 이름만 판을 올린다.
    USER 에 TARGET_SCOPE·EXPERT·CLASS_WORD·CAPTION_FIXED_WORDS 가 없거나 비어 있으면 빈 문자열 (run_many 는 건너뜀).

    경로는 지금 작업 폴더(노트북이 이 경로들을 읽은 곳) 기준 절대 경로로 바꾼다. run_many 는 상대 경로를 목록 파일
    (= 이번 실행 폴더) 기준으로 읽으므로, 그대로 두면 데이터셋을 못 찾거나 다음 결과가 이번 실행 폴더 안에 생긴다.
    DATASET_DIR·OUTPUT_DIR (비어 있으면 작업 폴더)·VALIDATION_FILE 은 늘, MODEL_PATH 는 로컬에 있을 때만
    (Hugging Face 저장소 이름은 그대로).
    fixed_overrides = base_overrides (이번 실행에 적용한 덮어쓰기 dict, 노트북의 OVERRIDES) 위에 overrides (이번 제안) 를
    덮은 것. 앞선 보완이 이어지고, 이번 실행과 다른 값은 이번 제안뿐이다. rank 가 있으면 alpha = rank.
    overrides·changes·base_overrides 가 None 이면 빈 것으로 본다.
    """
    opt = lambda k, d="": user.get(k) or d
    mp = user["MODEL_PATH"]
    fixed = {**dict(base_overrides or {}), **dict(overrides or {})}
    if "rank" in fixed:
        fixed["alpha"] = fixed["rank"]
    run = dict(family=user["MODEL_FAMILY"], model_path=_abs(mp) if mp and Path(mp).expanduser().exists() else mp,
               dataset_dir=_abs(user["DATASET_DIR"]), output_dir=_abs(opt("OUTPUT_DIR", ".")), trigger=user["TRIGGER"],
               run_name=next_run_name(user["RUN_NAME"]), target_scope=opt("TARGET_SCOPE"), expert=opt("EXPERT"),
               class_word=opt("CLASS_WORD"), validation_file=_abs(opt("VALIDATION_FILE")),
               caption_fixed_words=opt("CAPTION_FIXED_WORDS"), fixed_overrides=fixed,
               why=[{k: c.get(k) for k in ("key", "old", "new", "symptom", "evidence")} for c in changes or ()],
               previous=dict(run_name=user["RUN_NAME"], best_step=None if best_step is None else int(best_step)))
    return {"defaults": {}, "runs": [run]}

In [ ]:
%%writefile lora_video_adapters/wan22_t2v.py
"""Wan2.2 T2V-A14B adapter: Wan-AI/Wan2.2-T2V-A14B-Diffusers (WanPipeline with two denoisers).

Training math is the shared Wan code in wan_common.py (mirrors diffusers v0.42.1 pipeline_wan.py):
  x0 = (vae.encode(px).latent_dist.mode() - latents_mean) / latents_std ; x_t = (1 - s) * x0 + s * noise ;
  target = noise - x0 ; expert(hidden_states=x_t, timestep=1000 * s, encoder_hidden_states=UMT5 at 512 tokens).
Experts: `transformer` = original high_noise_model, used by the pipeline while t >= boundary_ratio * 1000 (0.875 in
model_index.json); `transformer_2` = low_noise_model below. Each expert trains its own LoRA on its sigma range only
(t = 1000 * s): high [0.875, 1], low [0, 0.875) as in musubi-tuner's T2V split; two files, the low one loads with
load_into_transformer_2. Text only; still images are F = 1 clips (WanPipeline accepts num_frames 1).
Validation = official Wan2.2 wan/configs/wan_t2v_A14B.py: sample_steps 40, sample_shift 12.0,
sample_guide_scale (3.0, 4.0) in (low, high) order -> diffusers guidance_scale 4.0 (high), guidance_scale_2 3.0
(low); shared_config sample_fps 16, frame_num 81. The -Diffusers repo ships UniPC flow_shift 3.0, so generate()
always sets the shift explicitly.
"""

from diffusers import WanPipeline

try:  # dev: video_adapters/ on sys.path; notebook: lora_video_tools/ on sys.path or importable as a package
    import wan_common as W
except ImportError:
    from lora_video_tools import wan_common as W

FAMILY = "wan22_t2v"
PIPELINE_CLASS = WanPipeline
EXPERTS = ["transformer", "transformer_2"]  # high noise first
EXPERT_TAGS = {"transformer": "high", "transformer_2": "low"}
CONDITIONING = "t2v"
FPS = 16
SPATIAL_FACTOR = 8  # vae_stride (4, 8, 8)
SPATIAL_MULTIPLE = 16  # 8 x patch 2
TEMPORAL_FACTOR = 4
TOKEN_DIV = 16
VAL_STEPS = 40
VAL_SHIFT = 12.0
VAL_GUIDANCE = (4.0, 3.0)  # (high-noise expert, low-noise expert)
NEGATIVE_PROMPT = W.NEGATIVE_PROMPT
LORA_TARGETS = W.LORA_TARGETS
BOUNDARY_RATIO = 0.875  # wan_t2v_A14B.py boundary; the loaded pipeline's own boundary_ratio is what sigma_range uses
_SPEC = dict(two_experts=True, in_channels=16, z_dim=16, expand_timesteps=False)

encode_prompts = W.encode_prompts
lora_to_comfy = W.lora_to_comfy
save_lora = W.save_lora
load_lora = W.load_lora


def sigma_range(pipe, expert):
    return W.sigma_range(pipe, expert, EXPERTS)


def load_pipeline(model_path, dtype):
    return W.load_pipeline(model_path, dtype, PIPELINE_CLASS, FAMILY, **_SPEC)


def tiny_pipeline(dtype):
    # diffusers v0.42.1 tests/pipelines/wan/test_wan_22.py Wan22 two-expert components (boundary 0.875)
    pipe = W.tiny_pipeline(PIPELINE_CLASS, dtype, two_experts=True, boundary_ratio=BOUNDARY_RATIO)
    W.check_pipeline(pipe, FAMILY, **_SPEC)
    return pipe


def encode_video(pipe, frames, device, dtype, condition):  # dtype unused: the VAE runs in its own dtype (fp32)
    return W.encode_video(pipe, frames, device, conditioning=CONDITIONING, multiple=SPATIAL_MULTIPLE,
                          condition=condition)


def forward_loss(pipe, denoiser, x0, cond, text, noise, sigma, mode):
    return W.flow_loss(denoiser, x0, noise, sigma, text, conditioning=CONDITIONING, mode=mode, cond=cond)


def generate(pipe, text, negative, *, height, width, num_frames, generator, image=None, steps=None, shift=None,
             guidance=None, device=None, offload=False):
    return W.generate(pipe, text, negative, height=height, width=width, num_frames=num_frames, generator=generator,
                      image=image, steps=VAL_STEPS if steps is None else steps,
                      shift=VAL_SHIFT if shift is None else shift,
                      guidance=VAL_GUIDANCE if guidance is None else guidance, device=device, offload=offload,
                      conditioning=CONDITIONING)

In [ ]:
%%writefile lora_video_adapters/wan22_i2v.py
"""Wan2.2 I2V-A14B adapter: Wan-AI/Wan2.2-I2V-A14B-Diffusers (WanImageToVideoPipeline with two denoisers; also
Index-AniSora V3.2 once converted to that layout).

Training math is the shared Wan code in wan_common.py (mirrors diffusers v0.42.1 pipeline_wan_i2v.py, the original
wan/image2video.py and the musubi-tuner / ai-toolkit / diffusion-pipe Wan2.2 I2V losses):
  x0 = (vae.encode(px).latent_dist.mode() - latents_mean) / latents_std ; x_t = (1 - s) * x0 + s * noise ;
  y = cat([mask4, (vae.encode(cat([frame0, zeros(F - 1)])).mode() - latents_mean) / latents_std]) exactly as
  prepare_latents (mask4 = pixel mask [1, 0, ..., 0] with frame 0 repeated 4x, viewed as 4 channels per latent frame) ;
  expert(hidden_states=cat([x_t, y]) (16 + 4 + 16 = 36 channels), timestep=1000 * s, encoder_hidden_states=UMT5 at
  512 tokens) ; loss = MSE(pred, noise - x0) over ALL latent frames, the first one included (as those trainers do).
No CLIP image tokens (transformer image_dim null; Wan2.1 I2V had them). Clips only: with F = 1 the condition already
holds x0, so forward_loss refuses single-latent-frame samples, and mode "t2v" altogether.
Experts: `transformer` = high_noise_model, used while t >= boundary_ratio * 1000 (0.9 in model_index.json), trains
sigma [0.9, 1]; `transformer_2` = low_noise_model, sigma [0, 0.9) (musubi-tuner's I2V split 900-1000 / 0-900). Two
files; the low one loads with load_into_transformer_2.
Validation = official Wan2.2 wan/configs/wan_i2v_A14B.py: sample_steps 40, sample_shift 5.0, sample_guide_scale
(3.5, 3.5) -> guidance_scale 3.5 (high), guidance_scale_2 3.5 (low); shared_config sample_fps 16, frame_num 81. The
-Diffusers repo ships UniPC flow_shift 3.0, so generate() always sets the shift. generate() needs the start image.
"""

from diffusers import WanImageToVideoPipeline

try:  # dev: video_adapters/ on sys.path; notebook: lora_video_tools/ on sys.path or importable as a package
    import wan_common as W
except ImportError:
    from lora_video_tools import wan_common as W

FAMILY = "wan22_i2v"
PIPELINE_CLASS = WanImageToVideoPipeline
EXPERTS = ["transformer", "transformer_2"]  # high noise first
EXPERT_TAGS = {"transformer": "high", "transformer_2": "low"}
CONDITIONING = "i2v_concat"
FPS = 16
SPATIAL_FACTOR = 8  # vae_stride (4, 8, 8)
SPATIAL_MULTIPLE = 16  # 8 x patch 2
TEMPORAL_FACTOR = 4
TOKEN_DIV = 16
VAL_STEPS = 40
VAL_SHIFT = 5.0
VAL_GUIDANCE = (3.5, 3.5)  # (high-noise expert, low-noise expert)
NEGATIVE_PROMPT = W.NEGATIVE_PROMPT
LORA_TARGETS = W.LORA_TARGETS
BOUNDARY_RATIO = 0.9  # wan_i2v_A14B.py boundary; the loaded pipeline's own boundary_ratio is what sigma_range uses
_SPEC = dict(two_experts=True, in_channels=36, z_dim=16, expand_timesteps=False)

encode_prompts = W.encode_prompts
lora_to_comfy = W.lora_to_comfy
save_lora = W.save_lora
load_lora = W.load_lora


def sigma_range(pipe, expert):
    return W.sigma_range(pipe, expert, EXPERTS)


def load_pipeline(model_path, dtype):
    # rejects T2V-A14B (in_channels 16), Wan2.1 I2V (one expert, image_dim / image_encoder) and TI2V-5B (z 48)
    return W.load_pipeline(model_path, dtype, PIPELINE_CLASS, FAMILY, **_SPEC)


def tiny_pipeline(dtype):
    # diffusers v0.42.1 tests/pipelines/wan/test_wan_22_image_to_video.py Wan22 two-expert I2V components (in 36,
    # out 16, no image encoder), boundary 0.9 as the real model_index.json (the test uses 0.875)
    pipe = W.tiny_pipeline(PIPELINE_CLASS, dtype, in_channels=36, two_experts=True, boundary_ratio=BOUNDARY_RATIO)
    W.check_pipeline(pipe, FAMILY, **_SPEC)
    return pipe


def encode_video(pipe, frames, device, dtype, condition):  # dtype unused: the VAE runs in its own dtype (fp32)
    return W.encode_video(pipe, frames, device, conditioning=CONDITIONING, multiple=SPATIAL_MULTIPLE,
                          condition=condition)


def forward_loss(pipe, denoiser, x0, cond, text, noise, sigma, mode):
    return W.flow_loss(denoiser, x0, noise, sigma, text, conditioning=CONDITIONING, mode=mode, cond=cond)


def generate(pipe, text, negative, *, height, width, num_frames, generator, image=None, steps=None, shift=None,
             guidance=None, device=None, offload=False):
    return W.generate(pipe, text, negative, height=height, width=width, num_frames=num_frames, generator=generator,
                      image=image, steps=VAL_STEPS if steps is None else steps,
                      shift=VAL_SHIFT if shift is None else shift,
                      guidance=VAL_GUIDANCE if guidance is None else guidance, device=device, offload=offload,
                      conditioning=CONDITIONING)

In [ ]:
%%writefile lora_video_adapters/wan22_ti2v_5b.py
"""Wan2.2 TI2V-5B adapter: Wan-AI/Wan2.2-TI2V-5B-Diffusers (WanPipeline, expand_timesteps true, one denoiser).

Training math is the shared Wan code in wan_common.py; the TI2V parts mirror the expand_timesteps branches of
diffusers v0.42.1 pipeline_wan.py (T2V) and pipeline_wan_i2v.py (I2V) and the original wan/textimage2video.py:
  x0 = (vae.encode(px).latent_dist.mode() - latents_mean) / latents_std with the Wan2.2 VAE (z 48, 2x2 patchify,
  16x spatial, 4x temporal) ; x_t = (1 - s) * x0 + s * noise ; target = noise - x0 ;
  input = (1 - m) * x0 + m * x_t ; per-token timestep = (m[0, 0][:, ::2, ::2] * 1000 * s).flatten(), shape
  (1, f * h/2 * w/2), as the pipeline's (mask[0][0][:, ::2, ::2] * t).flatten() ;
  "t2v" (images F = 1 and clips): m = 1 everywhere -> input x_t, every token at 1000 * s, loss over all frames ;
  "i2v" (clips): m = 0 on latent frame 0 -> clean first frame with timestep 0 on its tokens, loss over latent frames
  1: only (DiffSynth / ai-toolkit: the clean frame carries no loss).
The I2V pipeline conditions on vae.encode(frame0) alone; the Wan VAE is causal in time, so that equals x0[:, :, :1].
Validation = official Wan2.2 wan/configs/wan_ti2v_5B.py: sample_steps 50, sample_shift 5.0 (the -Diffusers repo
ships 5.0 too), sample_guide_scale 5.0, sample_fps 24, frame_num 121, vae_stride (4, 16, 16) -> H, W multiples of 32.
generate() runs T2V without an image and I2V with one (WanImageToVideoPipeline over the same modules). A 5B LoRA
does not load into the A14B models (inner dim 3072 vs 5120).
"""

from diffusers import WanPipeline

try:  # dev: video_adapters/ on sys.path; notebook: lora_video_tools/ on sys.path or importable as a package
    import wan_common as W
except ImportError:
    from lora_video_tools import wan_common as W

FAMILY = "wan22_ti2v_5b"
PIPELINE_CLASS = WanPipeline
EXPERTS = ["transformer"]
EXPERT_TAGS = {"transformer": "main"}
CONDITIONING = "ti2v"
FPS = 24  # wan_ti2v_5B.py sample_fps
SPATIAL_FACTOR = 16  # vae_stride (4, 16, 16)
SPATIAL_MULTIPLE = 32  # 16 x patch 2
TEMPORAL_FACTOR = 4
TOKEN_DIV = 32
VAL_STEPS = 50
VAL_SHIFT = 5.0
VAL_GUIDANCE = (5.0, 5.0)
NEGATIVE_PROMPT = W.NEGATIVE_PROMPT
LORA_TARGETS = W.LORA_TARGETS
_SPEC = dict(two_experts=False, in_channels=48, z_dim=48, expand_timesteps=True)

encode_prompts = W.encode_prompts
lora_to_comfy = W.lora_to_comfy
save_lora = W.save_lora
load_lora = W.load_lora


def sigma_range(pipe, expert):
    return W.sigma_range(pipe, expert, EXPERTS)


def load_pipeline(model_path, dtype):
    # model_index.json of the -Diffusers repo: WanPipeline, expand_timesteps true, no transformer_2 / boundary_ratio;
    # rejects Wan2.1 and A14B folders (z 16, in 16 / 36, two experts) and z-48 folders without expand_timesteps
    return W.load_pipeline(model_path, dtype, PIPELINE_CLASS, FAMILY, **_SPEC)


def tiny_pipeline(dtype):
    # diffusers v0.42.1 tests/pipelines/wan/test_wan_22.py Wan225B components (VAE z 48, patch 2, 16x spatial,
    # transformer in/out 48, expand_timesteps); UniPC flow_shift 3.0 as the test, so the harness sees generate() set 5.0
    pipe = W.tiny_pipeline(PIPELINE_CLASS, dtype, in_channels=48, out_channels=48, z_dim=48, expand_timesteps=True)
    W.check_pipeline(pipe, FAMILY, **_SPEC)
    return pipe


def encode_video(pipe, frames, device, dtype, condition):  # dtype unused; no "cond": I2V conditions on x0[:, :, :1]
    return W.encode_video(pipe, frames, device, conditioning=CONDITIONING, multiple=SPATIAL_MULTIPLE,
                          condition=condition)


def forward_loss(pipe, denoiser, x0, cond, text, noise, sigma, mode):
    return W.flow_loss(denoiser, x0, noise, sigma, text, conditioning=CONDITIONING, mode=mode, cond=cond)


def generate(pipe, text, negative, *, height, width, num_frames, generator, image=None, steps=None, shift=None,
             guidance=None, device=None, offload=False):
    return W.generate(pipe, text, negative, height=height, width=width, num_frames=num_frames, generator=generator,
                      image=image, steps=VAL_STEPS if steps is None else steps,
                      shift=VAL_SHIFT if shift is None else shift,
                      guidance=VAL_GUIDANCE if guidance is None else guidance, device=device, offload=offload,
                      conditioning=CONDITIONING)

In [ ]:
%%writefile lora_video_adapters/wan21.py
"""Wan2.1 T2V adapter: Wan-AI/Wan2.1-T2V-1.3B-Diffusers or Wan-AI/Wan2.1-T2V-14B-Diffusers (WanPipeline, one denoiser).

Training math is the shared Wan code in wan_common.py (mirrors diffusers v0.42.1 pipeline_wan.py):
  x0 = (vae.encode(px).latent_dist.mode() - latents_mean) / latents_std ; x_t = (1 - s) * x0 + s * noise ;
  target = noise - x0 ; transformer(hidden_states=x_t, timestep=1000 * s, encoder_hidden_states=UMT5 at 512 tokens).
Text only (CONDITIONING "t2v"); still images are F = 1 clips (WanPipeline accepts num_frames 1). One expert, so
sigma_range is (0, 1) and the LoRA file loads into `transformer`.
Validation defaults = official Wan2.1 generate.py for --task t2v-1.3B and t2v-14B (wan/configs/wan_t2v_*.py hold no
sampling values): sample_steps 50, sample_shift 5.0, sample_guide_scale 5.0, sample_fps 16, frame_num 81. Other
documented values: the Wan2.1 README advises 1.3B to use guide 6 with shift 8-12 (generate(shift=8.0,
guidance=(6.0, 6.0))), and its diffusers snippet uses flow_shift 3.0 for 480P. The -Diffusers repos ship UniPC
flow_shift 3.0, so generate() always sets the shift explicitly.
"""

from diffusers import WanPipeline

try:  # dev: video_adapters/ on sys.path; notebook: lora_video_tools/ on sys.path or importable as a package
    import wan_common as W
except ImportError:
    from lora_video_tools import wan_common as W

FAMILY = "wan21"
PIPELINE_CLASS = WanPipeline
EXPERTS = ["transformer"]
EXPERT_TAGS = {"transformer": "main"}
CONDITIONING = "t2v"
FPS = 16  # shared_config.py sample_fps
SPATIAL_FACTOR = 8  # vae_stride (4, 8, 8)
SPATIAL_MULTIPLE = 16  # 8 x patch 2
TEMPORAL_FACTOR = 4
TOKEN_DIV = 16
VAL_STEPS = 50
VAL_SHIFT = 5.0
VAL_GUIDANCE = (5.0, 5.0)
NEGATIVE_PROMPT = W.NEGATIVE_PROMPT
LORA_TARGETS = W.LORA_TARGETS
_SPEC = dict(two_experts=False, in_channels=16, z_dim=16, expand_timesteps=False)

encode_prompts = W.encode_prompts
lora_to_comfy = W.lora_to_comfy
save_lora = W.save_lora
load_lora = W.load_lora


def sigma_range(pipe, expert):
    return W.sigma_range(pipe, expert, EXPERTS)


def load_pipeline(model_path, dtype):
    return W.load_pipeline(model_path, dtype, PIPELINE_CLASS, FAMILY, **_SPEC)


def tiny_pipeline(dtype):
    # diffusers v0.42.1 tests/pipelines/wan/test_wan.py components, but with the UniPC scheduler the real
    # Wan2.1-T2V-*-Diffusers repos ship (the test uses FlowMatchEulerDiscreteScheduler(shift=7.0)).
    pipe = W.tiny_pipeline(PIPELINE_CLASS, dtype)
    W.check_pipeline(pipe, FAMILY, **_SPEC)
    return pipe


def encode_video(pipe, frames, device, dtype, condition):  # dtype unused: the VAE runs in its own dtype (fp32)
    return W.encode_video(pipe, frames, device, conditioning=CONDITIONING, multiple=SPATIAL_MULTIPLE,
                          condition=condition)


def forward_loss(pipe, denoiser, x0, cond, text, noise, sigma, mode):
    return W.flow_loss(denoiser, x0, noise, sigma, text, conditioning=CONDITIONING, mode=mode, cond=cond)


def generate(pipe, text, negative, *, height, width, num_frames, generator, image=None, steps=None, shift=None,
             guidance=None, device=None, offload=False):
    return W.generate(pipe, text, negative, height=height, width=width, num_frames=num_frames, generator=generator,
                      image=image, steps=VAL_STEPS if steps is None else steps,
                      shift=VAL_SHIFT if shift is None else shift,
                      guidance=VAL_GUIDANCE if guidance is None else guidance, device=device, offload=offload,
                      conditioning=CONDITIONING)

In [ ]:
for _d in (Path("lora_video_tools").resolve(), Path("lora_video_adapters").resolve()):
    if str(_d) not in sys.path:
        sys.path.insert(0, str(_d))
for _m in ["wan_common", "video_io", "video_metrics", "remedy"] + sorted(p.stem for p in Path("lora_video_adapters").glob("*.py")):
    if _m in sys.modules:                     # 셀을 다시 실행하면 새 파일로 다시 불러온다
        importlib.reload(sys.modules[_m])
import video_io as V
import video_metrics as VM
import remedy as RM
SUPPORTED = sorted(p.stem for p in Path("lora_video_adapters").glob("*.py"))
if FAMILY not in SUPPORTED:
    raise ValueError(f"{'LORA_NB_TEST' if TEST_FAMILY else 'MODEL_FAMILY'}='{FAMILY}' 는 지원하지 않는 계열입니다. "
                     f"고를 수 있는 값: {SUPPORTED}")
A = importlib.import_module(FAMILY)
for _name in ["FAMILY", "PIPELINE_CLASS", "EXPERTS", "EXPERT_TAGS", "CONDITIONING", "FPS", "SPATIAL_FACTOR",
              "SPATIAL_MULTIPLE", "TEMPORAL_FACTOR", "TOKEN_DIV", "VAL_STEPS", "VAL_SHIFT", "VAL_GUIDANCE",
              "NEGATIVE_PROMPT", "LORA_TARGETS", "sigma_range", "load_pipeline", "tiny_pipeline", "encode_prompts",
              "encode_video", "forward_loss", "lora_to_comfy", "save_lora", "load_lora", "generate"]:
    if not hasattr(A, _name):
        raise AttributeError(f"어댑터 {FAMILY} 에 {_name} 가 없습니다")
if FIXED["target_scope"] not in A.LORA_TARGETS:
    raise ValueError(f"target_scope='{FIXED['target_scope']}' 는 쓸 수 없습니다: {list(A.LORA_TARGETS)}")
for _k, _v in OVERRIDES.items():                  # 보완 규칙(remedy.ALLOWED)에 범위·선택지가 있는 값은 그 안인지
    _al = RM.ALLOWED.get(_k)
    if isinstance(_al, tuple) and len(_al) == 2 and all(isinstance(x, (int, float)) for x in _al) \
            and isinstance(_v, (int, float)):
        if not _al[0] <= _v <= _al[1]:
            raise ValueError(f"FIXED_OVERRIDES: {_k}={_v} 가 허용 범위 [{_al[0]}, {_al[1]}] 밖입니다")
    elif isinstance(_al, tuple) and _al and all(isinstance(x, str) for x in _al) and _v not in _al:
        raise ValueError(f"FIXED_OVERRIDES: {_k}='{_v}' 는 {', '.join(_al)} 중 하나여야 합니다")
# 학습 셀(3-1·4-3)이 쓰는 값으로 같은 객체를 한 번 만들어 본다: 값이 틀리면 모델(수십 GB)을 불러오기 전에 멈춘다
try:
    _buckets = V.make_buckets(FIXED["token_budget"], FIXED["frame_options"], FIXED["aspects"], FIXED["max_area"],
                              A.TOKEN_DIV, A.TEMPORAL_FACTOR, A.SPATIAL_MULTIPLE)
    _clip_buckets = [b for b in _buckets if b[0] >= FIXED["min_frames"]]          # 3-1 의 CLIP_BUCKETS 와 같음
    V.windows(FIXED["min_frames"], FIXED["min_frames"], FIXED["window_mode"], FIXED["windows_per_segment"],
              np.random.default_rng(0))
    _o = torch.optim.AdamW([torch.nn.Parameter(torch.zeros(1))], lr=FIXED["learning_rate"], betas=FIXED["betas"],
                           weight_decay=FIXED["weight_decay"], eps=FIXED["eps"])
    get_scheduler(FIXED["lr_scheduler"], optimizer=_o, num_warmup_steps=FIXED["warmup_steps"],
                  num_training_steps=FIXED["max_train_steps"])
except Exception as _e:
    raise ValueError(f"FIXED_OVERRIDES {OVERRIDES} 로는 학습을 준비할 수 없습니다: {type(_e).__name__}: {_e}") from None
# 오류 없이 학습을 망치는 값: clip_grad_norm_(…, 0) 은 '자르지 않음' 이 아니라 기울기를 0 으로 만들어 LoRA 가 0 으로 남는다
_bad = [k for k, good in (("max_grad_norm", FIXED["max_grad_norm"] > 0), ("logit_std", FIXED["logit_std"] > 0),
                           ("dropout", 0 <= FIXED["dropout"] < 1), ("caption_dropout", 0 <= FIXED["caption_dropout"] < 1),
                           ("image_share", 0 <= FIXED["image_share"] <= 1),
                           ("init_lora_weights", FIXED["init_lora_weights"] == "gaussian"),
                           ("mixed_precision", FIXED["mixed_precision"] in ("bf16", "fp16", "fp32")),
                           ("frame_options", max(FIXED["frame_options"]) >= FIXED["min_frames"]),
                           ("allow_upscale", FIXED["allow_upscale"] >= 1), ("aspects·max_area", bool(_buckets))) if not good]
if _bad:
    raise ValueError(f"FIXED_OVERRIDES 의 {', '.join(_bad)} 값을 쓸 수 없습니다: max_grad_norm·logit_std 는 0 보다 크고, "
                     "dropout·caption_dropout 은 0 이상 1 미만, image_share 는 0~1, init_lora_weights 는 gaussian, "
                     "mixed_precision 은 bf16·fp16·fp32 중 하나, frame_options 에는 min_frames 이상인 값이 있어야 하고, "
                     "allow_upscale 은 1 이상 (1.0 = 업스케일 없음), aspects·max_area 로 버킷이 하나 이상 나와야 합니다")
NEGATIVE = VALIDATION_NEGATIVE if VALIDATION_NEGATIVE is not None else A.NEGATIVE_PROMPT

# 3-1 의 버킷 고르기. 아래 데이터셋 확인(모델을 불러오기 전)과 3-1 이 같은 규칙을 쓰도록 여기서 정의한다
def own_buckets(w, h):
    """원본이 기본 버킷보다 작을 때 쓸 버킷: 비율마다 원본 안에 들어가는 가장 큰 넓이를 상한으로 만든다 (업스케일 없음).
    그 넓이가 max_area 의 1/8 보다 작으면 만들지 않는다 (너무 작은 원본은 학습에서 뺀다)."""
    out = set()
    for a in FIXED["aspects"]:
        cap = min(FIXED["max_area"], min(w, h * a) * min(h, w / a))
        if cap >= FIXED["max_area"] / 8:
            out.update(V.make_buckets(FIXED["token_budget"], FIXED["frame_options"], (a,), cap,
                                      A.TOKEN_DIV, A.TEMPORAL_FACTOR, A.SPATIAL_MULTIPLE))
    return sorted(out)

def fit_bucket(buckets, n, w, h, image):
    """기본 버킷에서 고르고, 없으면 원본 크기로 만든 버킷에서 고른다 (기본 버킷이 맞는 파일은 결과가 그대로다)."""
    b = V.pick_bucket(buckets, n, w, h, prefer="frames", allow_upscale=FIXED["allow_upscale"])
    if b is None:
        own = [x for x in own_buckets(w, h) if (x[0] == 1 if image else x[0] >= FIXED["min_frames"])]
        b = V.pick_bucket(own, n, w, h, prefer="frames", allow_upscale=FIXED["allow_upscale"])
    return b

# 데이터셋을 모델(수십 GB)보다 먼저 확인한다
if not TEST_FAMILY:
    _media = [f for f in DATASET_PATH.iterdir() if f.is_file() and f.suffix.lower() in V.IMG_EXT | V.VID_EXT]
    if not _media:
        raise FileNotFoundError(f"{DATASET_PATH} 에 이미지·영상 파일이 없습니다 (확장자 {sorted(V.IMG_EXT | V.VID_EXT)})")
    _videos = [f for f in _media if f.suffix.lower() in V.VID_EXT or V.is_animated_image(f)]
    print(f"데이터셋 {DATASET_PATH.resolve()}: 파일 {len(_media)}개 (영상·움직이는 그림 {len(_videos)}개)")

    def _is_clip(f):
        """3-1 이 학습 창을 고를 수 있는 영상: 모델 fps 로 리샘플한 프레임 수(V.resample_to_fps 와 같은 식)와 원본 크기로
        3-1 의 fit_bucket 이 클립 버킷을 고른다. 1프레임 영상은 정지 그림으로 다룬다.
        (장면 전환·멈춤으로 줄어드는 깨끗한 구간과 min_motion 은 1-2 분석 뒤에야 알아서 여기서는 보지 않는다)"""
        try:
            m = V.probe(f)
            n = math.floor(m["duration"] * A.FPS + 0.5 + 1e-9)
            return m["frames"] > 1 and fit_bucket(_clip_buckets, n, m["width"], m["height"], False) is not None
        except Exception:
            return False

    if A.CONDITIONING == "i2v_concat" and not any(_is_clip(f) for f in _videos):
        _fm = min((b[0] for b in _clip_buckets), default=FIXED["min_frames"])   # 가장 짧은 학습 창 (3-1 과 같음)
        raise ValueError(f"{A.FAMILY} 는 클립이 있어야 학습합니다 (1프레임 이미지는 조건 채널에 정답이 들어가 배우는 것이 없음). "
                         f"모델 fps({A.FPS})로 {_fm}프레임(약 {_fm / A.FPS:.1f}초, min_frames {FIXED['min_frames']} 이상인 가장 "
                         f"짧은 학습 창) 이상이고, 쓸 수 있는 넓이가 max_area 의 1/8({FIXED['max_area'] // 8:,}화소) 이상인 클립이 "
                         "필요합니다. 이미지만 있으면 T2V-A14B·TI2V-5B·Wan2.1 모델 경로로 먼저 학습하고, 클립이 생기면 I2V 로 학습합니다.")
print("어댑터:", A.FAMILY, "| 파이프라인:", A.PIPELINE_CLASS.__name__, "| 전문가:", A.EXPERTS, "| 조건:", A.CONDITIONING,
      f"| {A.FPS}fps, H·W 는 {A.SPATIAL_MULTIPLE} 배수, 프레임 4n+1")
print("LoRA 대상:", A.LORA_TARGETS[FIXED["target_scope"]])

### 0-4. 장치·정밀도 결정, 시드 고정, 모델 불러오기

In [ ]:
def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(FIXED["seed"])
if TEST_FAMILY or SMOKE:
    DEVICE = torch.device("cuda" if TEST_DEVICE == "cuda" and torch.cuda.is_available() else "cpu")
else:
    if not torch.cuda.is_available():
        raise RuntimeError("GPU 가 보이지 않습니다 (torch.cuda.is_available() = False). 실제 모델은 GPU 에서만 학습합니다.\n"
                           "컨테이너를 GPU 와 함께 띄웠는지(docker run --gpus all, 안에서 nvidia-smi), torch 가 CUDA 판"
                           "(torch.__version__ 끝이 +cu…)인지 확인하고 커널을 다시 시작합니다.")
    DEVICE = torch.device("cuda")
if FIXED["mixed_precision"] == "bf16" and DEVICE.type == "cuda" and torch.cuda.is_bf16_supported():
    WEIGHT_DTYPE = torch.bfloat16
elif FIXED["mixed_precision"] in ("bf16", "fp16") and DEVICE.type == "cuda":
    WEIGHT_DTYPE = torch.float16
else:
    WEIGHT_DTYPE = torch.float32
print("장치:", DEVICE, "| 가중치 dtype:", WEIGHT_DTYPE)

def check_model_path(mp):
    """0-2 의 detect_family 가 확인한 경로. 로컬 폴더면 ~ 를 펼친 절대 경로, 아니면 Hugging Face 저장소 이름."""
    mp = norm_model_path(mp)
    p = Path(mp).expanduser()
    return str(p.resolve()) if p.is_dir() else mp

# 같은 커널에서 다시 실행하면 앞 실행의 모델·학습 상태(A14B 면 수십 GB)를 먼저 놓아 준다.
# 셀 오류·중지 뒤에는 IPython 이 마지막 traceback 의 프레임(지역 변수에 모델)을 쥐고 있으므로 그것부터 비운다.
# 프레임이 쥔 함수의 클로저(체크포인트 재계산, DOP 의 run)도 모델을 쥐므로 traceback 자체도 놓는다 (sys.last_*, %debug 용 사본,
# 이 셀을 오류 셀 바로 다음에 실행하면 IPython 이 이 셀이 끝날 때까지 쥐는 앞 셀의 실행 결과 last_execution_result).
# 학습 셀(4-3)이 도중에 멈추면 전역 loss·pl 이 계산 그래프로 앞 모델의 가중치를 쥐고 있으므로 함께 지운다.
import traceback
try:
    traceback.clear_frames(getattr(sys, "last_traceback", None))
except Exception:
    pass
for _a in ("last_type", "last_value", "last_traceback", "last_exc"):
    if hasattr(sys, _a):
        setattr(sys, _a, None)
try:
    get_ipython().InteractiveTB.tb = None
    _r = get_ipython().last_execution_result
    if _r is not None:
        _r.error_in_exec = _r.error_before_exec = None
    del _r
except Exception:
    pass
for _name in ("pipe", "den", "optimizer", "params", "lr_scheduler", "_c", "p", "SAMPLES", "VAL_NOISE", "PREVIEW_HISTORY",
              "DINO", "CCIP", "TEXT", "VAL_TEXT", "DRIFT_TEXT", "PRES_TEXT", "ckpt_paths", "loss", "pl", "_captured"):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 불러오기 전에 메모리를 확인한다. A14B 는 미리보기·최종 생성 때 두 전문가와 텍스트 인코더가 모두 CPU 메모리에 있다(약 69GB).
# 모자라면 불러오는 도중이나 첫 미리보기에서 커널이 이유를 남기지 않고 죽는다 ('kernel died').
def model_weights_gb(folder, dtype):
    """구성요소별로 불러올 가중치 크기(GB)와, 가중치 파일이 다 있지 않은 구성요소 목록.
    safetensors 머리말만 읽어 원소 수를 세고, load_pipeline 처럼 VAE 는 fp32, 나머지는 dtype 으로 센다."""
    folder = Path(folder)
    nbytes = torch.tensor([], dtype=dtype).element_size()
    sizes, missing = {}, []
    for k, v in json.loads((folder / "model_index.json").read_text(encoding="utf-8")).items():
        sub = folder / k
        if not (isinstance(v, list) and v and v[0] is not None):
            continue
        if not sub.is_dir():
            missing.append(k)
            continue
        if not (sub / "config.json").is_file():        # 토크나이저·스케줄러 (가중치 없음)
            continue
        files = None
        for stem in ("diffusion_pytorch_model", "model"):
            ix = sub / f"{stem}.safetensors.index.json"
            if ix.is_file():   # 나뉜 파일: 목록의 조각이 모두 있어야 한다
                try:
                    wm = json.loads(ix.read_text(encoding="utf-8"))["weight_map"]
                    files = [sub / f for f in sorted(set(wm.values()))]
                except (OSError, ValueError, TypeError, KeyError, AttributeError) as e:
                    raise OSError(f"가중치 목록 파일을 읽을 수 없습니다: {ix} ({type(e).__name__}: {e}). 복사가 끊기거나 깨진 "
                                  "파일이면 hf download <저장소> --local-dir <모델 폴더> 로 다시 받습니다.") from None
            elif (sub / f"{stem}.safetensors").is_file():
                files = [sub / f"{stem}.safetensors"]
            if files:
                break
        if not files or not all(f.is_file() for f in files):
            missing.append(k)
            continue
        n = 0
        for f in files:
            size = f.stat().st_size
            try:
                with open(f, "rb") as fh:
                    hl = int.from_bytes(fh.read(8), "little")
                    if not 2 <= hl <= min(size - 8, 100_000_000):     # safetensors 머리말 길이 한도
                        raise ValueError(f"머리말 길이 {hl}")
                    head = json.loads(fh.read(hl))
                ts = [t for key, t in head.items() if key != "__metadata__"]
                end = max((t["data_offsets"][1] for t in ts), default=0)
                if size != 8 + hl + end:          # safetensors 도 크기가 정확히 맞아야 읽는다 (데이터가 잘리거나 덧붙은 파일)
                    raise ValueError(f"파일 크기가 머리말 기준 {8 + hl + end:,}바이트와 다름")
                n += sum(math.prod(t["shape"]) for t in ts)
            except (OSError, ValueError, TypeError, KeyError, AttributeError) as e:
                raise OSError(f"가중치 파일을 읽을 수 없습니다: {f} ({size:,}바이트, {type(e).__name__}: {e}). git-lfs 포인터·"
                              "0바이트·데이터가 잘린 파일이면 hf download <저장소> --local-dir <모델 폴더> 로 다시 받습니다.") from None
        sizes[k] = n * (4 if k == "vae" else nbytes) / 1e9
    return sizes, missing

def free_ram_gb():
    """(지금 쓸 수 있는 시스템 메모리, 다른 프로그램을 모두 멈췄을 때 이 커널이 더 쓸 수 있는 메모리, 상한을 정한 컨테이너
    메모리 한도 또는 None) GB. psutil 이 없으면 (None, None, None).
    리눅스 컨테이너에 메모리 한도(cgroup)가 있으면 그 한도 안에서 잰다. cgroup v2 는 /proc/self/cgroup 의 경로와 그 위 단계의
    memory.max 를 모두 본다 (--cgroupns=host 컨테이너는 /sys/fs/cgroup 이 호스트 루트다). v1 은 hierarchical_memory_limit 도 본다.
    커널이 회수하는 파일 캐시(inactive·active file)는 빈 것으로 센다 ('file'·'cache' 는 공유 메모리까지 들어가 쓰지 않는다)."""
    try:
        import psutil
        vm = psutil.virtual_memory()
        free, cap, rss = vm.available, vm.total, psutil.Process().memory_info().rss
    except Exception:
        return None, None, None
    lim_gb, rel = None, ""
    try:
        rel = next((l[3:].strip().strip("/") for l in Path("/proc/self/cgroup").read_text().splitlines()
                    if l.startswith("0::")), "")
    except OSError:
        pass
    parts = [] if ".." in rel.split("/") else [x for x in rel.split("/") if x]
    v2 = [("/sys/fs/cgroup/" + "/".join(parts[:i])).rstrip("/") for i in range(len(parts) + 1)]
    for d, lim, cur in [(x, "memory.max", "memory.current") for x in v2] + \
                       [("/sys/fs/cgroup/memory", "memory.limit_in_bytes", "memory.usage_in_bytes")]:
        try:
            raw = (Path(d) / lim).read_text().strip()
            stat = dict(l.split()[:2] for l in (Path(d) / "memory.stat").read_text().splitlines() if len(l.split()) >= 2)
            limit = min(1 << 62 if raw == "max" else int(raw), int(stat.get("hierarchical_memory_limit", 1 << 62)))
            if limit >= 1 << 60:
                continue
            reclaim = sum(int(stat.get("total_" + k, stat.get(k, 0))) for k in ("inactive_file", "active_file"))
            free = min(free, limit - int((Path(d) / cur).read_text()) + reclaim)
            if limit < cap:
                cap, lim_gb = limit, limit / 1e9
        except (OSError, ValueError):
            pass
    return free / 1e9, (cap - rss) / 1e9, lim_gb

def gpu_need_gb(folder, w, tokens, offload):
    """GPU 에 비어 있어야 할 메모리(GB) 추정 [추정: 실제 폭의 Wan 블록을 RTX 5060 에서 1~6층으로 재어 층 수로 늘림].
    - LoRA: 대상 선형층마다 rank × (입력 + 출력) 파라미터, fp32 파라미터·기울기·AdamW 상태 2개로 파라미터당 16바이트.
    - 학습 단계: 가장 큰 디노이저 + LoRA + 토큰 수 × 토큰당 바이트 + 여유 2GB.
      토큰당 = 체크포인트로 남기는 층마다의 블록 입력(dim) + 다시 계산·역전파하는 한 블록의 작업분(16 × (dim + ffn)), 가중치 dtype 기준.
    - 생성(미리보기·최종): 오프로드(A14B 기본)면 전문가 하나 + LoRA + 생성 토큰 × 토큰당 바이트의 1/4 (추론은 활성값을 남기지
      않음, 실측 약 0.2배) + 여유 2GB. 생성 토큰은 max(preview_frames, final_frames) 의 잠재 프레임 × max_area 토큰.
      오프로드가 없으면 디노이저 모두 + LoRA + VAE + VAE 디코드(출력 화소당 약 23KB, fp32) + 여유 2GB.
    (디노이저 이름, 학습 몫, 생성 몫, 토큰당 바이트, LoRA 몫) 를 돌려준다."""
    nb = torch.tensor([], dtype=WEIGHT_DTYPE).element_size()
    dens = [k for k in w if k.startswith("transformer")]
    den = max(dens, key=w.get)
    cfg = json.loads((Path(folder) / den / "config.json").read_text(encoding="utf-8"))
    dim = cfg["num_attention_heads"] * cfg["attention_head_dim"]
    per_tok = nb * dim * cfg["num_layers"] + 16 * nb * (dim + cfg["ffn_dim"])
    lin = sum(2 * dim if t.startswith("attn") else dim + cfg["ffn_dim"] for t in A.LORA_TARGETS[FIXED["target_scope"]])
    lora = 16 * FIXED["rank"] * lin * cfg["num_layers"] / 1e9
    train = w[den] + lora + per_tok * tokens / 1e9 + 2
    if offload:
        gen_tok = ((max(FIXED["preview_frames"], FIXED["final_frames"]) - 1) // A.TEMPORAL_FACTOR + 1) \
            * (FIXED["max_area"] // A.TOKEN_DIV ** 2)
        gen = w[den] + lora + per_tok / 4 * gen_tok / 1e9 + 2
    else:
        gen = sum(w[k] for k in dens) + lora + w.get("vae", 0.0) + 23e-6 * FIXED["max_area"] + 2
    return den, train, gen, per_tok, lora

t0 = time.time()
_mdir = None if TEST_FAMILY else check_model_path(USER["MODEL_PATH"])
_wdir = _mdir if _mdir is not None and Path(_mdir).is_dir() else None
if os.environ.get("LORA_SKIP_RAM_CHECK") == "1":
    if _mdir is not None:
        print("참고: LORA_SKIP_RAM_CHECK=1 이라 불러오기 전 메모리 확인을 건너뜁니다.")
    _wdir = None
elif _mdir is not None and _wdir is None:   # Hugging Face 저장소 이름: 이미 받아 둔 사본이 있으면 그것으로 센다
    try:
        from huggingface_hub import snapshot_download
        _wdir = snapshot_download(_mdir, local_files_only=True)
    except Exception:
        print("참고: 아직 받지 않은 저장소라 메모리 확인을 건너뜁니다 (불러오기 전에 내려받습니다).")
_w, _miss = model_weights_gb(_wdir, WEIGHT_DTYPE) if _wdir is not None else ({}, [])
if _miss:
    print("참고: 가중치 파일이 다 있지 않아 메모리 확인을 건너뜁니다:", _miss, "(저장소 이름을 주었으면 불러오기 전에 내려받습니다)")
_up = lambda x: math.ceil(x * 10) / 10          # 오류 문구: 필요량은 올리고 남은 양은 내려, 같은 숫자로 보이지 않게
_dn = lambda x: math.floor(x * 10) / 10         # (안내 줄은 소수 둘째 자리라 오류 숫자는 늘 그 값의 보수적인 한 자리)
if _w and not _miss:
    _need, (_free, _room, _lim) = sum(_w.values()), free_ram_gb()
    print("불러올 가중치(GB):", {k: round(v, 2) for k, v in _w.items()}, f"| 합 {_need:.2f}GB |",
          f"쓸 수 있는 시스템 메모리 {_free:.2f}GB" if _free is not None else "시스템 메모리 확인 못 함 (psutil 없음)")
    if _free is not None and _room < _need + 8:
        _nbw = torch.tensor([], dtype=WEIGHT_DTYPE).element_size()
        # 더 작은 계열 [공식 저장소 머리말로 잰 bf16 GB: 텍스트 인코더 + 디노이저, VAE(fp32)] 가운데 이 커널에 들어가는 것만 권한다
        _small = [n for n, g, v in (("Wan2.1-T2V-14B", 39.94, 0.51), ("Wan2.2-TI2V-5B", 21.36, 2.82),
                                    ("Wan2.1-T2V-1.3B", 14.2, 0.51))
                  if g * _nbw / 2 + v + 8 <= _room and g * _nbw / 2 + v < _need - 0.1]
        raise MemoryError(
            f"이 {'컨테이너' if _lim else '서버'}의 메모리로는 이 모델을 불러올 수 없습니다: 가중치 {_up(_need):.1f}GB + 여유 8GB 가 "
            f"필요한데, 다른 프로그램을 모두 멈춰도 이 커널이 더 쓸 수 있는 메모리는 약 {_dn(_room):.1f}GB 입니다.\n"
            + (f"컨테이너 메모리 한도(docker run --memory, 지금 약 {_lim:.0f}GB)를 늘립니다." if _lim else
               "컨테이너 메모리 한도가 아니라 서버 RAM 이 모자랍니다. RAM 이 더 큰 서버를 씁니다.")
            + (f" 또는 더 작은 모델({', '.join(_small)})을 씁니다." if _small else ""))
    if _free is not None and _free < _need + 8:
        raise MemoryError(
            f"시스템 메모리가 모자랍니다: 가중치 {_up(_need):.1f}GB + 여유 8GB 가 필요한데 {_dn(_free):.1f}GB 만 비어 있습니다. "
            "미리보기·최종 생성 때는 모든 구성요소가 CPU 메모리에 올라가므로 이대로면 도중에 커널이 죽습니다.\n"
            "다른 프로그램(ComfyUI 실행 공간, 다른 노트북 커널)을 멈추고 이 셀부터 다시 실행합니다. "
            "확인을 건너뛰려면 환경변수 LORA_SKIP_RAM_CHECK=1")
    if DEVICE.type == "cuda":
        _gfree, _gtotal = (x / 1e9 for x in torch.cuda.mem_get_info(DEVICE))
        _bk = max(_w, key=_w.get)   # 한 번에 GPU 에 올리는 가장 큰 구성요소 (A14B 는 전문가 하나, 5B·1.3B 는 텍스트 인코더)
        _clip = any(_is_clip(f) for f in _videos)
        # 학습 토큰: 클립은 토큰 예산까지, 이미지(1프레임)는 max_area 에 묶인다
        _tok = FIXED["token_budget"] if _clip else min(FIXED["max_area"] // A.TOKEN_DIV ** 2, FIXED["token_budget"])
        _offl = FIXED["offload_sampling"] and len(A.EXPERTS) > 1
        try:
            _den, _gtrain, _gprev, _ptok, _lora = gpu_need_gb(_wdir, _w, _tok, _offl)
        except (KeyError, ValueError, OSError) as _e:
            print("참고: 학습 GPU 메모리를 추정하지 못해 가장 큰 구성요소로만 봅니다:", repr(_e))
            _den, _gtrain, _gprev, _ptok, _lora = _bk, 0.0, 0.0, None, 0.0
        _gneed = max(_w[_bk] + 4, _gtrain, _gprev)
        print(f"GPU 메모리: 비어 있음 {_gfree:.2f}GB / 전체 {_gtotal:.1f}GB | 필요 추정 {_gneed:.2f}GB = 가장 큰 값 "
              f"[{_bk} 올리기 {_w[_bk] + 4:.2f}, 학습 {_gtrain:.2f} ({_tok:,}토큰, {'클립 예산' if _clip else '이미지 한 장 최대'})"
              + (f", 생성 {_gprev:.2f} ({'전문가 하나씩' if _offl else '디노이저 모두 + VAE 디코드'})" if _gprev else "") + "]")
        # 지금 빈 메모리로 학습 단계에 들어가는 토큰 수. 생성 몫이 이미 들어가지 않으면 토큰 예산을 줄여도 소용없으므로 0
        _fit = int((_gfree - _w[_den] - _lora - 2) * 1e9 / _ptok) if _ptok and max(_w[_bk] + 4, _gprev) <= _gfree else 0
        if _clip and _ptok:
            print(f"  지금 비어 있는 GPU 메모리로 들어가는 클립 token_budget 약 {max(_fit, 0):,} [추정]")
        if _gfree < _gneed:
            raise MemoryError(
                f"GPU 메모리가 모자랍니다: {_up(_gneed):.1f}GB 이상 비어 있어야 하는데 {_dn(_gfree):.1f}GB 입니다 [추정]. "
                "ComfyUI 대기열 등 GPU 를 쓰는 작업을 멈추고 이 셀부터 다시 실행합니다."
                + ("\n클립 토큰 예산을 줄여도 됩니다 (지금 덮어쓴 값은 그대로 둠): LORA_FIXED_OVERRIDES='"
                   + json.dumps({**OVERRIDES, "token_budget": _fit // 1000 * 1000}) + "'"
                   if _clip and _gneed == _gtrain and _fit >= 1000 else "")
                + "\n추정이 맞지 않는다고 보면 환경변수 LORA_SKIP_RAM_CHECK=1 로 확인을 건너뜁니다.")
        if _gtotal - _gfree > 2:
            print(f"참고: 다른 프로그램이 GPU 메모리 약 {_gtotal - _gfree:.1f}GB 를 쓰고 있습니다. 학습 중에 그 프로그램이 "
                  "메모리를 더 쓰면 도중에 멈출 수 있습니다.")
pipe = A.tiny_pipeline(WEIGHT_DTYPE) if TEST_FAMILY else A.load_pipeline(_mdir, WEIGHT_DTYPE)
if not isinstance(pipe, A.PIPELINE_CLASS):
    raise TypeError(f"불러온 파이프라인 {type(pipe).__name__} 이 {A.PIPELINE_CLASS.__name__} 가 아닙니다.")
if TEST_FAMILY:   # 자동 인식 확인: 작은 모델의 설정 파일만 폴더에 써 두고 계열을 맞히는지 본다
    _cfg = Path(tempfile.mkdtemp()) / "cfg"
    pipe.save_config(_cfg)
    for _n in ("transformer", "transformer_2", "vae"):
        if getattr(pipe, _n, None) is not None:
            getattr(pipe, _n).save_config(_cfg / _n)
    assert detect_family(str(_cfg)) == TEST_FAMILY, (detect_family(str(_cfg)), TEST_FAMILY)
    print("모델 계열 자동 인식 확인:", TEST_FAMILY)
for _c in pipe.components.values():
    if isinstance(_c, torch.nn.Module):
        _c.requires_grad_(False)
        _c.eval()
MODULES = [k for k, v in pipe.components.items() if isinstance(v, torch.nn.Module)]

def place(on_device=()):
    """on_device 에 든 구성요소만 장치에 올리고 나머지는 CPU 로 내린다."""
    for k in MODULES:
        m = getattr(pipe, k, None)
        if isinstance(m, torch.nn.Module):
            m.to(DEVICE if k in on_device else "cpu")
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

place([])
print(pd.DataFrame([(k, type(getattr(pipe, k)).__name__, sum(p.numel() for p in getattr(pipe, k).parameters()) / 1e6)
                    for k in MODULES], columns=["구성요소", "클래스", "파라미터(백만)"]).to_string(index=False))
print(f"불러온 가중치 {sum(p.numel() * p.element_size() for k in MODULES for p in getattr(pipe, k).parameters()) / 1e9:.2f}GB")
if len(A.EXPERTS) > 1:
    print("전문가 σ 구간:", {A.EXPERT_TAGS[e]: tuple(round(x, 4) for x in A.sigma_range(pipe, e)) for e in A.EXPERTS})
print(f"불러오기 {time.time() - t0:.1f}초")

## 1. EDA (데이터 살펴보기)

### 1-1. 파일 목록과 메타데이터

- 영상은 헤더 값이 틀리는 경우가 많아(VFR, webm·mkv 의 프레임 수 0 등) PyAV 로 패킷 시각을 모아 프레임 수·fps·길이를 계산한다.
- 휴대폰 영상의 회전 정보와 화소 비율(SAR)을 반영한 '보이는 크기'를 쓴다.

In [ ]:
IMG_EXT, VID_EXT = V.IMG_EXT, V.VID_EXT
if TEST_FAMILY:
    DATA_DIR = Path(tempfile.mkdtemp()) / "data"
    TEST_EXPECT = V.make_test_dataset(DATA_DIR, USER["TRIGGER"])
    FIXED_WORDS = list(TEST_EXPECT.get("fixed_words") or FIXED_WORDS)   # 시험 데이터에 넣어 둔 고정 외형 낱말
else:
    DATA_DIR, TEST_EXPECT = DATASET_PATH, None
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"DATASET_DIR 가 없습니다: {DATA_DIR}")

def read_caption(path):
    """UTF-8(BOM 포함)·UTF-16·CP949 캡션을 읽는다 (이미지 노트북과 같음)."""
    b = path.read_bytes()
    if b.startswith((b"\xff\xfe", b"\xfe\xff")):
        return b.decode("utf-16").strip()
    if len(b) >= 4 and b"\x00" in b:
        even, odd = b[0::2], b[1::2]
        if odd.count(0) > len(odd) * 0.6:
            return b.decode("utf-16-le", errors="replace").strip()
        if even.count(0) > len(even) * 0.6:
            return b.decode("utf-16-be", errors="replace").strip()
    try:
        return b.decode("utf-8-sig").strip()
    except UnicodeDecodeError:
        pass
    try:
        text = b.decode("cp949").strip()
        problems.append((path.name, "캡션을 CP949 로 읽음 (UTF-8 로 저장 권장)"))
        return text
    except UnicodeDecodeError:
        problems.append((path.name, "캡션 인코딩을 알 수 없어 일부 글자를 바꿈"))
        return b.decode("utf-8", errors="replace").strip()

def load_still(path):
    """정지 그림 한 장 (np.uint8 H×W×3). 1프레임짜리 영상 파일(정지 GIF·1프레임 mp4 등)은 PyAV 로 첫 프레임을 읽는다."""
    path = Path(path)
    if path.suffix.lower() in IMG_EXT and not V.is_animated_image(path):
        return V.load_image(path)
    fr, _ = V.read_frames(path)
    return fr[0]

TRIGGER_RE = re.compile(r"(?<!\w)" + re.escape(USER["TRIGGER"]) + r"(?![A-Za-z0-9_])")
rows, problems = [], []
for p in sorted(DATA_DIR.iterdir()):
    ext = p.suffix.lower()
    if ext not in IMG_EXT and ext not in VID_EXT:
        continue
    try:
        if ext in IMG_EXT and not V.is_animated_image(p):   # 움직이는 webp·png·gif 는 클립으로 읽는다
            arr = V.load_image(p)
            meta = dict(kind="image", width=arr.shape[1], height=arr.shape[0], frames=1, fps=float("nan"),
                        duration=0.0, vfr=False, rotation=0, codec="image", has_alpha=False, hdr=False)
        else:
            m = V.probe(p)
            meta = dict(kind="video", width=m["width"], height=m["height"], frames=m["frames"], fps=m.get("fps", float("nan")),
                        duration=m.get("duration", 0.0), vfr=bool(m.get("vfr", False)), rotation=int(m.get("rotation", 0)),
                        codec=m.get("codec", ""), has_alpha=bool(m.get("has_alpha", False)), hdr=bool(m.get("hdr", False)))
            if meta["frames"] <= 1:
                meta["kind"] = "image"              # 한 프레임짜리 영상(예: 정지 GIF)은 이미지로 다룬다
    except Exception as e:
        problems.append((p.name, f"열 수 없음: {type(e).__name__}: {e}")); continue
    cap_path = p.with_suffix(".txt")
    caption = read_caption(cap_path) if cap_path.exists() else ""
    rows.append(dict(file=p.name, **meta, aspect=round(meta["width"] / max(1, meta["height"]), 3), caption=caption,
                     has_caption=bool(caption), has_trigger=bool(TRIGGER_RE.search(caption))))
if not rows:
    raise RuntimeError(f"{DATA_DIR} 에 이미지·영상이 없습니다")
df = pd.DataFrame(rows)
n_img, n_vid = int((df.kind == "image").sum()), int((df.kind == "video").sum())
print(f"이미지 {n_img}개, 클립 {n_vid}개")
print(df[["file", "kind", "width", "height", "frames", "fps", "duration", "vfr", "rotation", "codec"]].to_string(index=False))

### 1-2. 클립 신호 분석 (검은·멈춘 구간, 장면 전환, 움직임, 흐림, 중복)

- 검은 프레임·멈춤은 ffmpeg 의 blackdetect·freezedetect 기준, 장면 전환은 PySceneDetect 기본 기준을 numpy 로 옮긴 것이다.
- 2D 셀 애니메이션은 같은 그림을 2·3프레임 보여 주는 '홀드'가 정상이라, 멈춤과 따로 `hold_ratio`·`drawing_fps` 로 보여 준다.
- 움직임은 프레임 차이(깜빡임에 속는다) 대신 블록 정합 변위로 잰다: `subject_motion`(피사체), `camera_motion`(화면 전체).
  - 단위는 '초당 화면 긴 변 대비 이동 비율'이라 해상도·fps 가 달라도 비교할 수 있다. 0.05 보다 작으면 거의 정지한 클립이다.
- 중복은 클립마다 같은 위치 8프레임의 pHash 거리로 찾는다 (8비트 이하면 '중복 의심', 자동으로 지우지 않음).

In [ ]:
SIGNALS, EVENTS, TIMESTAMPS, SIGNATURES = {}, {}, {}, {}
extra = []
for r in df.itertuples():
    path = DATA_DIR / r.file
    if r.kind == "image":
        arr = load_still(path)
        frames = arr[None]
        s = V.summarize_clip(frames, 1.0)
        SIGNATURES[r.file] = V.clip_signature(frames, k=1)
        extra.append(dict(file=r.file, sha1=hashlib.sha1(arr.tobytes()).hexdigest(), **{k: s.get(k) for k in ("blur", "sat")}))
        continue
    frames, ts = V.read_frames(path, max_side=FIXED["analysis_side"] * 3)
    TIMESTAMPS[r.file] = ts
    fps = r.fps if r.fps == r.fps and r.fps > 0 else A.FPS
    sig = V.frame_signals(frames, fps, analysis_side=FIXED["analysis_side"])
    ev = V.detect_events(sig, fps, timestamps=ts)        # 시각을 주면 VFR 의 긴 정지 화면도 멈춤으로 잡는다
    s = V.summarize_clip(frames, fps, sig, ev)
    SIGNALS[r.file], EVENTS[r.file] = sig, ev
    SIGNATURES[r.file] = V.clip_signature(frames, k=8)
    extra.append(dict(file=r.file, sha1="", decoded=len(frames), **{k: s.get(k) for k in (
        "subject_motion", "camera_motion", "static", "hold_ratio", "drawing_fps", "blur", "n_cuts", "n_black",
        "n_freeze", "sat")}))
df = df.merge(pd.DataFrame(extra), on="file", how="left")
for _c in ("subject_motion", "camera_motion", "static", "hold_ratio", "drawing_fps", "n_cuts", "n_black", "n_freeze",
           "decoded", "blur", "sat"):
    if _c not in df.columns:             # 이미지만 있는 데이터셋: 클립 열을 빈 값으로 둔다
        df[_c] = np.nan

# 중복: 이미지는 화소가 같으면 빼고, 클립은 pHash 거리가 가까우면 사람이 보도록 표시만 한다
dup_img = df.kind.eq("image") & df.duplicated("sha1", keep="first") & df.sha1.ne("")
for f in df.loc[dup_img, "file"]:
    problems.append((f, "같은 이미지가 이미 있어 제외"))
df = df.loc[~dup_img].reset_index(drop=True)
vids = list(df.loc[df.kind == "video", "file"])
DUP_PAIRS = []
for i, a in enumerate(vids):
    for b in vids[i + 1:]:
        d = V.signature_distance(SIGNATURES[a], SIGNATURES[b])
        if d <= 8:
            DUP_PAIRS.append((a, b, d)); problems.append((b, f"{a} 와 거의 같은 클립 (pHash 거리 {d:.1f}, 확인 필요)"))

for r in df.itertuples():
    if r.kind != "video":
        continue
    if r.fps == r.fps and r.fps < A.FPS - 0.5:
        problems.append((r.file, f"원본 {r.fps:.1f}fps < 모델 {A.FPS}fps: 리샘플하면 같은 프레임이 반복됩니다"))
    if r.vfr:
        problems.append((r.file, "가변 프레임률(VFR): 시각 기준으로 리샘플합니다"))
    if r.n_cuts:
        problems.append((r.file, f"장면 전환 {int(r.n_cuts)}곳: 창은 전환을 넘지 않게 고릅니다"))
    if r.n_black or r.n_freeze:
        problems.append((r.file, f"검은 구간 {int(r.n_black)}개, 멈춘 구간 {int(r.n_freeze)}개: 학습 창에서 뺍니다"))
    if r.subject_motion == r.subject_motion and r.subject_motion < V.STATIC_MOTION:
        problems.append((r.file, f"거의 움직이지 않는 클립 (움직임 {r.subject_motion:.3f} < {V.STATIC_MOTION}, 단위: 초당 화면 긴 변 비율)"))
    if r.hold_ratio is not None and r.hold_ratio > 0.3:
        problems.append((r.file, f"홀드 비율 {r.hold_ratio:.2f} (셀 애니메이션이면 정상, fps 를 올려 변환한 흔적일 수도 있음)"))
    if r.hdr:
        problems.append((r.file, "HDR·10비트 영상: 색이 바랠 수 있습니다"))
print(df[["file", "kind", "subject_motion", "camera_motion", "hold_ratio", "n_cuts", "n_black", "n_freeze", "blur"]]
      .round(3).to_string(index=False))

### 1-3. 캡션 검사

- 트리거가 없으면 앞에 붙이고, 캡션이 없으면 트리거만으로 학습한다 (이미지 노트북과 같음).
- 클립 캡션에는 동작과 카메라 표현이 있어야 하고, EDA 측정값과 모순되면 표시한다 (예: 'static camera' 인데 화면 전체가 크게 움직임).
- `CAPTION_FIXED_WORDS` 에 적은 고정 외형 낱말이 캡션에 있으면 표시한다.

In [ ]:
cap_issue_col = []
for r in df.itertuples():
    summary = None if r.kind == "image" else dict(subject_motion=r.subject_motion, camera_motion=r.camera_motion)
    iss = V.caption_issues(r.caption, TRIGGER_RE, r.kind == "video", FIXED_WORDS, summary) if r.caption else ["캡션 없음"]
    cap_issue_col.append("; ".join(iss))
    if not r.caption:
        problems.append((r.file, "캡션 파일 없음 → 트리거만으로 학습"))
    elif iss:
        problems.append((r.file, "캡션: " + "; ".join(iss)))
df["caption_issues"] = cap_issue_col
df["train_caption"] = [c if t else (f"{USER['TRIGGER']}, {c}" if c else USER["TRIGGER"])
                       for c, t in zip(df.caption, df.has_trigger)]
df["caption_words"] = [len(c.split()) for c in df.train_caption]
print(df[["file", "kind", "caption_words", "caption_issues"]].to_string(index=False))

# 트리거 대신 넣을 일반 낱말 (그림체 변화 확인·보존 학습). auto 면 캡션에 가장 많이 나온 후보 낱말
CLASS_CANDIDATES = ("girl", "boy", "woman", "man", "child", "kid", "baby", "person", "cat", "dog", "bird", "animal",
                    "creature", "robot", "monster", "character")
KO_CLASS = {"소녀": "girl", "여자아이": "girl", "소년": "boy", "남자아이": "boy", "여자": "woman", "남자": "man", "아이": "child"}
CLASS_WORD = USER["CLASS_WORD"].strip()
if CLASS_WORD in ("", "auto"):
    _PLURAL = dict(girls="girl", boys="boy", women="woman", men="man", children="child", kids="kid", babies="baby",
                   people="person", persons="person", cats="cat", dogs="dog", birds="bird", animals="animal",
                   creatures="creature", robots="robot", monsters="monster", characters="character")
    _words = [_PLURAL.get(w, w) for c in df.caption for w in re.findall(r"[a-z]+", c.lower())]
    _cnt = Counter(w for w in _words if w in CLASS_CANDIDATES)
    _she = sum(w in ("she", "her", "hers", "herself") for w in _words)
    _he = sum(w in ("he", "him", "his", "himself") for w in _words)
    _ko = Counter(v for c in df.caption for k, v in KO_CLASS.items()        # 앞뒤 낱말 경계 (조사 허용, '아이스크림' 은 아님)
                  if re.search(r"(?<![가-힣])" + k + r"(?:가|이|는|은|를|을|와|과|의|도|들)?(?![가-힣])", c))
    if _cnt:
        CLASS_WORD, _why = _cnt.most_common(1)[0][0], "캡션에 가장 많이 나온 낱말"
    elif _she != _he:
        CLASS_WORD, _why = ("girl" if _she > _he else "boy"), "캡션의 대명사(she/he)"
    elif _ko:
        CLASS_WORD, _why = _ko.most_common(1)[0][0], "캡션의 한국어 낱말"
    else:
        CLASS_WORD, _why = "person", "캡션에 알맞은 낱말이 없어 기본값. 직접 정하려면 USER 의 CLASS_WORD 를 girl·boy 등으로 바꾼다"
    print(f"\nCLASS_WORD = {CLASS_WORD} ({_why})")
DRIFT_PROMPTS = [_TRIG.sub(lambda m: CLASS_WORD, p) for p in VALIDATION_PROMPTS[:1]]   # 트리거 없는 같은 장면

### 1-4. 문제 목록과 요약

In [ ]:
print(f"이미지 {int((df.kind == 'image').sum())}개, 클립 {int((df.kind == 'video').sum())}개 (중복 이미지 제외 뒤)")
print(df[["width", "height", "frames", "fps", "duration", "caption_words"]].describe().round(2).to_string())
print("\n문제 목록:" if problems else "\n문제 없음")
for f, msg in problems:
    print(f"  - {f}: {msg}")
if A.CONDITIONING == "i2v_concat" and not (df.kind == "video").any():
    raise ValueError(f"{A.FAMILY} 는 클립이 있어야 학습합니다 (1프레임 이미지는 조건 채널에 정답이 들어가 배우는 것이 없음). "
                     "이미지만 있으면 T2V-A14B·TI2V-5B·Wan2.1 모델 경로로 먼저 학습하고, 클립이 생기면 I2V 로 학습합니다.")

if TEST_FAMILY:   # 합성 데이터의 알려진 문제(회전·VFR·검은 구간·멈춤·장면 전환·중복·정지 클립)를 바르게 찾았는지 확인
    V.check_test_expectations(df, EVENTS, DUP_PAIRS, TEST_EXPECT)
    print("시험 데이터 확인 통과")

## 2. 시각화

### 2-1. 분포

In [ ]:
vd = df[df.kind == "video"]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.3))
if len(vd):
    axes[0].hist(vd.duration, bins=10, color="#7B3342"); axes[0].set_title("클립 길이 (초)")
    axes[1].hist(vd.fps, bins=10, color="#4A372E"); axes[1].axvline(A.FPS, ls="--", c="k"); axes[1].set_title(f"fps (점선 = 모델 {A.FPS})")
    axes[2].scatter(vd.duration, vd.subject_motion, c=np.where(vd.vfr, "tab:red", "tab:blue"))
    for _, r in vd.iterrows():
        axes[2].annotate(r.file[:12], (r.duration, r.subject_motion), fontsize=6)
    axes[2].axhline(V.STATIC_MOTION, ls=":", c="gray")
    axes[2].set_title("길이 대 움직임 (빨강 = VFR, 점선 = 정지 기준)"); axes[2].set_xlabel("초"); axes[2].set_ylabel("초당 긴 변 비율")
else:
    for ax in axes[:3]:
        ax.axis("off"); ax.text(0.5, 0.5, "클립 없음", ha="center")
axes[3].scatter(df.width, df.height, c=np.where(df.kind == "video", "tab:blue", "tab:orange"))
axes[3].set_title("해상도 (주황 = 이미지)"); axes[3].set_xlabel("W"); axes[3].set_ylabel("H")
plt.tight_layout(); plt.show()

### 2-2. 컨택트 시트 (클립마다 같은 간격 8프레임, 이미지는 전체에서 고르게 8장, 비율 유지)

In [ ]:
_rows, _labels = [], []
for r in df.itertuples():
    if r.kind == "video":
        fr, _ = V.read_frames(DATA_DIR / r.file, max_side=FIXED["analysis_side"] * 3)
        _rows.append(fr); _labels.append(f"{r.file} | {r.frames}f {r.fps:.1f}fps")
_all = list(df.loc[df.kind == "image", "file"])
_pick = [_all[j] for j in np.linspace(0, len(_all) - 1, min(8, len(_all))).round().astype(int)] if _all else []
_imgs = [load_still(DATA_DIR / f) for f in _pick]
if _imgs:   # 정사각형으로 늘이지 않고 여백을 둔다 (캐릭터 비율이 보이게)
    _rows.append(np.stack([VM.Dino.letterbox(i, 128) for i in _imgs])); _labels.append(f"이미지 {len(_imgs)}/{len(_all)}장")
if _rows:
    sheet = V.contact_sheet(_rows, labels=_labels, n=8, thumb_h=96)
    plt.figure(figsize=(14, 1.3 * len(_rows) + 1)); plt.imshow(sheet); plt.axis("off"); plt.show()

### 2-3. 클립별 신호 곡선 (세로선 = 장면 전환, 회색 = 멈춤, 검정 = 검은 구간)

In [ ]:
_files = list(SIGNALS)[:8]
if _files:
    fig, axes = plt.subplots(len(_files), 1, figsize=(13, 2.0 * len(_files)), squeeze=False)
    for ax, f in zip(axes[:, 0], _files):
        sig, ev = SIGNALS[f], EVENTS[f]
        ax.plot(sig["content"], label="장면 차이 (HSV)", lw=1)
        ax.plot(np.asarray(sig["mad"]) * 255, label="밝기 차이 ×255", lw=1)
        ax.plot(sig["motion_p95"], label="움직임 p95 (px)", lw=1)
        for c in ev["cuts"]:
            ax.axvline(c - 1, c="r", ls="--", lw=1)
        for a, b in ev["freeze"]:
            ax.axvspan(a, b - 1, color="gray", alpha=0.3)
        for a, b in ev["black"]:
            ax.axvspan(a, b - 1, color="k", alpha=0.3)
        ax.set_title(f"{f} | hold {ev['hold_ratio']:.2f}, drawing fps {ev['drawing_fps']:.1f}", fontsize=9)
    axes[0, 0].legend(fontsize=7, ncol=3)
    plt.tight_layout(); plt.show()
else:
    print("클립 없음")

## 3. 전처리

### 3-1. 모델 fps 로 리샘플하고 학습 창 고르기

- 시각 기준으로 모델 fps 격자 시각에 화면에 떠 있는 원본 프레임을 고른다 (ffmpeg fps 필터와 같음, 늘이거나 줄이지 않음).
  - VFR·GIF 에서 오래 떠 있는 프레임은 그만큼 반복되고, 0.25초 이상이면 1-2 의 멈춤 검사가 학습 창에서 뺀다.
- 장면 전환·검은·멈춘 구간을 뺀 '깨끗한 구간' 안에서만 창을 고른다.
- 창 길이는 4n+1 프레임이고, 버킷은 토큰 예산(`token_budget`) 안에서 프레임 수를 우선한다. 업스케일은 하지 않는다.
  - 원본이 기본 버킷보다 작으면(예: TI2V-5B 에 480p 클립, 640x360 클립) 원본 안에 들어가는 더 작은 버킷을 쓴다. 쓸 수 있는 넓이(지원 비율 `aspects` 로 잘라 남는 넓이)가 `max_area` 의 1/8 보다 작은 원본은 뺀다.
  - 미리보기·최종 생성은 가장 흔한 학습 버킷의 비율로, 넓이는 `max_area` 로 만든다 (작은 버킷이 생성 크기를 끌어내리지 않게).
- `min_motion` 보다 움직임이 작은 클립은 뺀다.
- 학습에서 뺀 파일과 이유는 셀 끝에 출력한다.

In [ ]:
RUN_DIR = (OUTPUT_PATH / (USER["RUN_NAME"] + (f"_test_{TEST_FAMILY}" if TEST_FAMILY else ""))).resolve()   # 결과 폴더 (절대 경로)
if RUN_DIR.exists() and TEST_FAMILY:
    shutil.rmtree(RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)

BUCKETS = V.make_buckets(FIXED["token_budget"], FIXED["frame_options"], FIXED["aspects"], FIXED["max_area"],
                         A.TOKEN_DIV, A.TEMPORAL_FACTOR, A.SPATIAL_MULTIPLE)
IMG_BUCKETS = [b for b in BUCKETS if b[0] == 1]
CLIP_BUCKETS = [b for b in BUCKETS if b[0] >= FIXED["min_frames"]]
# own_buckets·fit_bucket (원본이 기본 버킷보다 작을 때의 버킷)은 0-3 에서 정의했다: 0-3 의 클립 확인과 같은 규칙

rng = np.random.default_rng(FIXED["seed"])
PLAN = []          # 학습 샘플 계획: 디코딩은 3-4 캐시 때 한 번만
_n_prob = len(problems)
_fmin = min((f for f in FIXED["frame_options"] if f >= FIXED["min_frames"]), default=FIXED["min_frames"])   # 가장 짧은 학습 창
for r in df.itertuples():
    if r.kind == "image":
        if A.CONDITIONING == "i2v_concat":
            continue                                    # I2V 학습에는 쓰지 않는다 (검증 시작 이미지로 쓴다)
        b = fit_bucket(IMG_BUCKETS, 1, r.width, r.height, True)
        if b is None:   # 넓이가 1/8 이상인데 빠지면 원인은 지원 비율로 자르기다 (예: 1600x150 → 가장 커야 273x150)
            _a = max(FIXED["aspects"], key=lambda a: min(r.width, r.height * a) * min(r.height, r.width / a))
            problems.append((r.file, f"이미지 {r.width}x{r.height} 가 작아 맞는 버킷이 없음 (업스케일 금지, 쓸 수 있는 넓이가 "
                                     "max_area 의 1/8 미만)" if r.width * r.height < FIXED["max_area"] / 8 else
                             f"이미지 {r.width}x{r.height} 에서 지원 비율(aspects)로 자르면 남는 크기(가장 커야 "
                             f"{min(r.width, r.height * _a):.0f}x{min(r.height, r.width / _a):.0f})가 max_area 의 1/8 "
                             "미만이라 맞는 버킷이 없음 (업스케일 금지)")); continue
        PLAN.append(dict(file=r.file, kind="image", start=0, F=1, H=b[1], W=b[2], tokens=b[3], caption=r.train_caption))
        continue
    if r.subject_motion is not None and r.subject_motion < FIXED["min_motion"]:
        problems.append((r.file, f"움직임 {r.subject_motion:.3f} < min_motion {FIXED['min_motion']}: 학습에서 뺌")); continue
    ts = TIMESTAMPS[r.file]
    idx = V.resample_to_fps(ts, A.FPS)                  # 모델 fps 격자 → 원본 프레임 번호
    seg_of = np.full(len(ts), -1)
    for j, (a, b) in enumerate(V.clean_segments(len(ts), EVENTS[r.file])):
        seg_of[a:b] = j
    lab = seg_of[idx]                                   # 리샘플한 프레임마다 깨끗한 구간 번호
    starts, k = [], 0
    while k < len(lab):
        e = k
        while e < len(lab) and lab[e] == lab[k]:
            e += 1
        if lab[k] >= 0:
            starts.append((k, e))
        k = e
    used = 0
    for a, e in starts:
        b = fit_bucket(CLIP_BUCKETS, e - a, r.width, r.height, False)
        if b is None:
            continue
        for s in V.windows(e - a, b[0], FIXED["window_mode"], FIXED["windows_per_segment"], rng):
            PLAN.append(dict(file=r.file, kind="clip", start=a + s, F=b[0], H=b[1], W=b[2], tokens=b[3],
                             caption=r.train_caption, src_idx=idx[a + s:a + s + b[0]].tolist()))
            used += 1
    if not used:
        _long = max((e - a for a, e in starts), default=0)    # 모델 fps 로 센 가장 긴 깨끗한 구간
        problems.append((r.file, "깨끗한 구간이 없어 쓰지 않음 (장면 전환·검은·멈춘 구간뿐)" if not starts else
                         f"깨끗한 구간이 가장 길어야 {_long}프레임(모델 {A.FPS}fps 기준)이라 가장 짧은 학습 창 {_fmin}프레임보다 "
                         "짧아 쓰지 않음" if _long < _fmin else
                         f"{r.width}x{r.height} 에 맞는 버킷이 없어 쓰지 않음 (업스케일 금지, 쓸 수 있는 넓이가 max_area 의 1/8 미만)"))
_dropped = problems[_n_prob:]
if _dropped:
    print("학습에서 뺀 파일:")
    for _f, _why in _dropped:
        print(f"  - {_f}: {_why}")
plan_df = pd.DataFrame(PLAN)
if plan_df.empty:
    raise RuntimeError("학습 샘플이 하나도 없습니다."
                       + ("".join(f"\n  - {_f}: {_why}" for _f, _why in _dropped) if _dropped else " 1 EDA 의 문제 목록을 확인하세요."))
# 검증 손실용 파일: 학습에서 통째로 뺀다 (같은 클립의 다른 창이 학습에 섞이지 않게 파일 단위)
_kind_of = {}
for p in PLAN:
    _kind_of.setdefault(p["file"], p["kind"])
_quota = Counter(_kind_of.values())
_quota = {k: n // 10 for k, n in _quota.items()}          # 종류(이미지·클립)마다 10개당 1개까지
HOLDOUT = set()
for f in np.random.default_rng(FIXED["seed"] + 3).permutation(sorted(_kind_of)).tolist():
    if len(HOLDOUT) >= FIXED["holdout_files"]:
        break
    if _quota[_kind_of[f]] > 0:
        HOLDOUT.add(f)
        _quota[_kind_of[f]] -= 1
for p in PLAN:
    p["holdout"] = p["file"] in HOLDOUT
print("검증 손실용으로 학습에서 뺀 파일:", sorted(HOLDOUT) or "없음 (종류마다 파일이 10개 미만이라 학습 샘플로 잰다)")
for p in PLAN:
    assert (p["F"] - 1) % A.TEMPORAL_FACTOR == 0 and p["H"] % A.SPATIAL_MULTIPLE == 0 and p["W"] % A.SPATIAL_MULTIPLE == 0, p
print("버킷 후보:", [(b[0], b[1], b[2]) for b in BUCKETS])
print(f"학습 샘플 {len(PLAN)}개 (이미지 {int((plan_df.kind == 'image').sum())}, 클립 창 {int((plan_df.kind == 'clip').sum())})")

### 3-2. 버킷 분포

In [ ]:
bc = Counter(f"{p['F']}f {p['W']}x{p['H']}" for p in PLAN)
plt.figure(figsize=(max(6, len(bc) * 1.2), 3)); plt.bar(list(bc.keys()), list(bc.values()), color="#4A372E")
plt.xticks(rotation=30); plt.title("(프레임, 너비x높이) 버킷별 샘플 수"); plt.tight_layout(); plt.show()
print(plan_df.groupby(["kind", "F", "W", "H"]).size().rename("샘플").reset_index().to_string(index=False))

### 3-3. 모델이 실제로 볼 창 미리보기 (모델 fps, 자르기 뒤)

In [ ]:
def window_frames(p, cache=None):
    """계획 p 의 프레임을 디코딩해 버킷 크기로 자른다. cache 에 클립별 디코딩 결과를 둔다."""
    if p["kind"] == "image":
        return V.resize_crop_frames(load_still(DATA_DIR / p["file"])[None], p["H"], p["W"])
    if cache is not None and p["file"] in cache:
        fr = cache[p["file"]]
    else:
        fr, _ = V.read_frames(DATA_DIR / p["file"])
        if cache is not None:
            cache.clear(); cache[p["file"]] = fr
    return V.resize_crop_frames(fr[p["src_idx"]], p["H"], p["W"])

_prev = [p for p in PLAN if p["kind"] == "clip"][:3] or PLAN[:1]
_clips = [window_frames(p) for p in _prev]
sheet = V.contact_sheet(_clips, labels=[f"{p['file']} @{p['start']} {p['F']}f" for p in _prev], n=8, thumb_h=96)
plt.figure(figsize=(14, 1.4 * len(_clips) + 1)); plt.imshow(sheet); plt.axis("off"); plt.show()
for j, (p, c) in enumerate(zip(_prev, _clips)):
    if p["kind"] == "clip":
        out = V.save_mp4(RUN_DIR / f"preprocess_preview_{j}.mp4", c, A.FPS)
        if Video is not None and not TEST_FAMILY:
            display(Video(str(out), embed=True))

### 3-4. 잠재값·텍스트 미리 계산 (캐시)

- 텍스트 인코더(UMT5)와 VAE 는 한 번만 쓰고 GPU 에서 내린다. 학습 중에는 학습하는 전문가만 GPU 에 둔다.
- 잠재값은 VAE 사후분포의 평균(mode)을 `(z - latents_mean) / latents_std` 로 정규화한 값이다 (diffusers 파이프라인과 같음).
- I2V 는 창의 첫 프레임으로 조건(마스크 4채널 + [첫 프레임, 0] 의 잠재값)을 만든다.

In [ ]:
def to_dev(d):
    return None if d is None else {k: (v.to(DEVICE) if torch.is_tensor(v) else v) for k, v in d.items()}

captions = sorted({p["caption"] for p in PLAN})
CAP_IDX = {c: i for i, c in enumerate(captions)}
place(["text_encoder"])
with torch.no_grad():
    TEXT = A.encode_prompts(pipe, captions, DEVICE, WEIGHT_DTYPE)
    VAL_TEXT = A.encode_prompts(pipe, VALIDATION_PROMPTS, DEVICE, WEIGHT_DTYPE)
    DRIFT_TEXT = A.encode_prompts(pipe, DRIFT_PROMPTS, DEVICE, WEIGHT_DTYPE)
    NEG_TEXT = A.encode_prompts(pipe, [NEGATIVE], DEVICE, WEIGHT_DTYPE)[0]
    EMPTY_TEXT = A.encode_prompts(pipe, [""], DEVICE, WEIGHT_DTYPE)[0] if FIXED["caption_dropout"] > 0 else None
    PRES_TEXT = A.encode_prompts(pipe, [TRIGGER_RE.sub(lambda m: CLASS_WORD, c) for c in captions], DEVICE, WEIGHT_DTYPE) \
        if FIXED["preservation"] == "dop" else None
place(["vae"])
SAMPLES, THUMBS, _cache = [], [], {}
with torch.no_grad():
    for p in tqdm(PLAN, desc="VAE"):
        fr = window_frames(p, _cache)
        need_cond = A.CONDITIONING == "i2v_concat" and p["kind"] == "clip"
        enc = A.encode_video(pipe, fr, DEVICE, WEIGHT_DTYPE, condition=need_cond)
        assert torch.isfinite(enc["x0"]).all(), f"{p['file']} 잠재값에 NaN/Inf"
        SAMPLES.append(dict(**{k: p[k] for k in ("file", "kind", "F", "H", "W", "tokens", "holdout")}, cap=CAP_IDX[p["caption"]],
                            x0=enc["x0"], cond=enc.get("cond")))
        if len(THUMBS) < 16:
            THUMBS.append(fr[len(fr) // 2])
    # 잠재값 왕복 확인: 첫 샘플을 다시 그림으로 풀어 원본과 비교한다 (정규화가 틀리면 색이 크게 바뀜)
    _p0 = PLAN[0]; _x = SAMPLES[0]["x0"].to(DEVICE, pipe.vae.dtype)
    _mean = torch.tensor(pipe.vae.config.latents_mean, device=DEVICE, dtype=_x.dtype).view(1, -1, 1, 1, 1)
    _std = torch.tensor(pipe.vae.config.latents_std, device=DEVICE, dtype=_x.dtype).view(1, -1, 1, 1, 1)
    _dec = pipe.vae.decode(_x * _std + _mean, return_dict=False)[0]
    ROUNDTRIP = ((_dec[0, :, 0].float().clamp(-1, 1).permute(1, 2, 0).cpu().numpy() + 1) * 127.5).round().astype(np.uint8)
    del _x, _mean, _std, _dec                     # 학습·생성 동안 GPU 에 남기지 않는다 (클립이면 수백 MB)
place([])
_cache.clear()
ORIG0 = window_frames(_p0)[0]
_mse = float(((ROUNDTRIP.astype(np.float32) - ORIG0.astype(np.float32)) ** 2).mean())
ROUNDTRIP_PSNR = None if _mse == 0 else round(10 * math.log10(255 ** 2 / _mse), 2)
print(f"캐시 샘플 {len(SAMPLES)}개 | 캡션 {len(captions)}종 | 잠재값 왕복 PSNR {ROUNDTRIP_PSNR} dB (실제 모델은 보통 25dB 이상)")

### 3-5. 전처리 결과 확인

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
axes[0].imshow(ORIG0); axes[0].set_title("원본 (자른 뒤)"); axes[0].axis("off")
axes[1].imshow(ROUNDTRIP); axes[1].set_title(f"VAE 왕복 ({ROUNDTRIP_PSNR} dB)"); axes[1].axis("off")
z_mean = [float(s["x0"].mean()) for s in SAMPLES]; z_std = [float(s["x0"].std()) for s in SAMPLES]
axes[2].scatter(z_mean, z_std, c=["tab:orange" if s["kind"] == "image" else "tab:blue" for s in SAMPLES])
axes[2].set_title("잠재값 평균-표준편차 (주황 = 이미지)"); axes[2].set_xlabel("mean"); axes[2].set_ylabel("std")
plt.tight_layout(); plt.show()
print(f"잠재값 전체 평균 {np.mean(z_mean):.3f}, 표준편차 평균 {np.mean(z_std):.3f} (정규화가 맞으면 대략 0, 1 근처)")

## 4. 학습

- 전문가마다 LoRA 를 붙이고, 전문가 차례대로 학습한다 (A14B 기본: 저노이즈 → 고노이즈). 학습하는 전문가만 GPU 에 둔다.
- 노이즈 단계 σ 는 logit-normal 을 shift 로 옮긴 분포에서 뽑되, 그 전문가의 구간 안에 들어올 때까지 다시 뽑는다 (분포 모양 보존).
- 손실: x_t = (1-σ)·x0 + σ·noise, 목표 = noise - x0, 디노이저에는 t = 1000·σ 를 준다.
- 저장(`save_every`)과 검증(`validate_every`)은 전체 단계 기준이다. 검증마다 고정 노이즈 검증 손실과 고정 시드 미리보기 클립을 남긴다.
- 미리보기는 0 단계(=베이스 모델) 클립과 비교해 움직임·깜빡임·색 변화를 잰다.

### 4-1. LoRA 붙이기와 전문가 계획

In [ ]:
TARGETS = list(A.LORA_TARGETS[FIXED["target_scope"]])
lora_cfg = LoraConfig(r=FIXED["rank"], lora_alpha=FIXED["alpha"], lora_dropout=FIXED["dropout"],
                      init_lora_weights=FIXED["init_lora_weights"], target_modules=TARGETS)

# 전문가 계획: [(전문가, 단계 수)]
has_clip = any(s["kind"] == "clip" and not s["holdout"] for s in SAMPLES)
if len(A.EXPERTS) == 1:
    PHASES = [(A.EXPERTS[0], FIXED["max_train_steps"])]
else:
    HIGH, LOW = A.EXPERTS
    mode = "both" if USER["EXPERT"] == "auto" else USER["EXPERT"]
    high_ok = has_clip or FIXED["high_uses_images"]
    if mode == "high" and not high_ok:
        raise ValueError("고노이즈 전문가는 기본으로 클립으로만 학습합니다. 클립을 넣거나 high_uses_images=True 로 덮어씁니다.")
    if mode == "both" and high_ok:
        n_low = round(FIXED["max_train_steps"] * FIXED["low_steps_share"])
        PHASES = [(LOW, n_low), (HIGH, FIXED["max_train_steps"] - n_low)]
    elif mode == "both":
        PHASES = [(LOW, FIXED["max_train_steps"])]
        _why = "학습에 쓸 클립 창이 없어 (3-1 의 '학습에서 뺀 파일' 참고)" if (df.kind == "video").any() else "클립이 없어"
        print(f"참고: {_why} 고노이즈 전문가는 학습하지 않습니다. 저노이즈 LoRA 만 {FIXED['max_train_steps']}단계 학습합니다.")
    else:
        PHASES = [(LOW if mode == "low" else HIGH, FIXED["max_train_steps"])]
PHASES = [(e, n) for e, n in PHASES if n > 0]
POOL, VAL_ITEMS = {}, {}
for e, _ in PHASES:
    is_high = len(A.EXPERTS) > 1 and e == A.EXPERTS[0]
    fits = lambda s: s["kind"] == "clip" or not is_high or FIXED["high_uses_images"]
    POOL[e] = [i for i, s in enumerate(SAMPLES) if fits(s) and not s["holdout"]]
    if not POOL[e]:
        raise RuntimeError(f"{A.EXPERT_TAGS[e]} 전문가가 학습할 샘플이 없습니다")
    _hold = [i for i, s in enumerate(SAMPLES) if fits(s) and s["holdout"]]
    VAL_ITEMS[e] = (_hold[:FIXED["val_loss_items"]], True) if _hold else (POOL[e][:FIXED["val_loss_items"]], False)

for e, _ in PHASES:
    den = getattr(pipe, e)
    den.add_adapter(lora_cfg)
    for n, p in den.named_parameters():
        p.requires_grad_("lora_" in n)
        if "lora_" in n:
            p.data = p.data.float()            # 학습 파라미터는 fp32
    n_mod = len({n.rsplit(".lora_", 1)[0] for n, _ in den.named_parameters() if "lora_A" in n})
    if n_mod == 0:
        raise RuntimeError(f"LoRA 를 붙일 층을 찾지 못했습니다: {TARGETS}")
    print(f"{A.EXPERT_TAGS[e]:>5} 전문가 ({e}): LoRA 층 {n_mod}개, 파라미터 "
          f"{sum(p.numel() for n, p in den.named_parameters() if 'lora_' in n) / 1e6:.2f}M, 학습 {dict(PHASES)[e]}단계, "
          f"σ 구간 {tuple(round(x, 3) for x in A.sigma_range(pipe, e))}, 샘플 {len(POOL[e])}개, "
          f"검증 손실 {'따로 뺀 파일' if VAL_ITEMS[e][1] else '학습 샘플 (과적합 판단에 쓰지 않음)'}")

### 4-2. 학습 준비: 노이즈 단계·검증 손실·미리보기 (DINO·CCIP 불러오기)

In [ ]:
def expert_shift(e):
    if len(A.EXPERTS) == 1:
        return FIXED["shift"]
    return FIXED["shift_high"] if e == A.EXPERTS[0] else FIXED["shift_low"]

def sample_sigma(gen, e):
    """logit-normal → shift → 전문가 구간 안으로 거절 샘플링. 거의 안 들어오면 구간 안 균등으로 대신한다."""
    lo, hi = A.sigma_range(pipe, e)
    sh = expert_shift(e)
    for _ in range(2000):
        u = torch.sigmoid(torch.normal(FIXED["logit_mean"], FIXED["logit_std"], (1,), generator=gen))
        s = sh * u / (1 + (sh - 1) * u)
        if lo <= float(s) < hi or (hi >= 1.0 and float(s) >= lo):
            return s.clamp(1e-3, 1 - 1e-3)
    return (lo + (hi - lo) * torch.rand((1,), generator=gen)).clamp(1e-3, 1 - 1e-3)

# 검증 손실: 전문가 구간 안의 고정 σ 4개 · 고정 노이즈 · 고정 샘플 → 단계끼리 비교 가능
_vg = torch.Generator("cpu").manual_seed(FIXED["seed"] + 7)
VAL_NOISE = {i: torch.randn(SAMPLES[i]["x0"].shape, generator=_vg) for i in range(len(SAMPLES))}

def sample_mode(s, gen):
    if A.CONDITIONING == "i2v_concat":
        return "i2v"
    if A.CONDITIONING == "ti2v" and s["kind"] == "clip" and float(torch.rand((1,), generator=gen)) < FIXED["ti2v_i2v_share"]:
        return "i2v"
    return "t2v"

@torch.no_grad()
def validation_loss(e):
    den = getattr(pipe, e); was = den.training; den.eval()
    lo, hi = A.sigma_range(pipe, e)
    vals = []
    for i in VAL_ITEMS[e][0]:
        s = SAMPLES[i]
        mode = "i2v" if A.CONDITIONING == "i2v_concat" else "t2v"
        for q in (0.125, 0.375, 0.625, 0.875):
            sig = torch.tensor([lo + (min(hi, 1 - 1e-3) - lo) * q], device=DEVICE)
            vals.append(float(A.forward_loss(pipe, den, s["x0"].to(DEVICE), to_dev({"cond": s["cond"]}) if s["cond"] is not None else None,
                                             to_dev(TEXT[s["cap"]]), VAL_NOISE[i].to(DEVICE), sig, mode)))
    den.train(was)
    return float(np.mean(vals))

def fit_hw(h, w, area, m):
    """(h, w) 의 비율을 지키며 면적 area 안에서 m 의 배수인 가장 큰 (H, W)"""
    s = math.sqrt(area / (h * w))
    return max(m, int(h * s) // m * m), max(m, int(w * s) // m * m)

# 미리보기·최종 생성 크기: 학습 샘플에서 가장 흔한 (H, W) 의 비율, 넓이는 max_area (0-4 생성 추정과 같은 크기).
#   3-1 이 작은 원본에 쓴 작은 버킷이나 긴 클립의 줄인 버킷이 생성 크기를 끌어내리지 않게 한다.
_hw = Counter((s["H"], s["W"]) for s in SAMPLES if s["kind"] == "clip") or Counter((s["H"], s["W"]) for s in SAMPLES)
PREVIEW_HW = fit_hw(*_hw.most_common(1)[0][0], FIXED["max_area"], A.SPATIAL_MULTIPLE)
# I2V 시작 이미지: VALIDATION_FILE images → 데이터셋 이미지 → 클립 첫 프레임.
# 가운데를 잘라 미리보기 크기에 맞추지 않고, 이미지마다 그 비율에 맞는 생성 크기를 쓴다 (세로 전신 그림의 머리가 잘리지 않게).

def start_image(arr):
    """시작 이미지를 미리보기와 같은 면적, 자기 비율의 생성 크기로 줄인다 (배수에 맞추느라 가장자리만 조금 잘림)"""
    return V.resize_crop_frames(arr[None], *fit_hw(*arr.shape[:2], PREVIEW_HW[0] * PREVIEW_HW[1], A.SPATIAL_MULTIPLE))[0]

if VALIDATION_IMAGES:
    START_IMAGES = [start_image(load_still(Path(x))) for x in VALIDATION_IMAGES]
else:
    _imgs = [r.file for r in df.itertuples() if r.kind == "image"][:4]
    START_IMAGES = [start_image(load_still(DATA_DIR / f)) for f in _imgs] or \
                   [start_image(window_frames(p)[0]) for p in [q for q in PLAN if q["kind"] == "clip"][:2]]
NEEDS_IMAGE = A.CONDITIONING == "i2v_concat"
OFFLOAD = bool(FIXED["offload_sampling"]) and DEVICE.type == "cuda" and len(A.EXPERTS) > 1
ACTIVE = []          # 지금 GPU 에 둘 학습 중 전문가

def generate_clip(text, seed, frames, steps=None, image=None, hw=None):
    """캐시한 임베딩으로 고정 시드 클립을 만든다. 끝나면 학습 배치를 되돌린다."""
    flags = {e: getattr(pipe, e).training for e in A.EXPERTS}
    for e in A.EXPERTS:
        getattr(pipe, e).eval()
    if not OFFLOAD:
        place(list(A.EXPERTS) + ["vae"])
    try:
        h, w = hw or (image.shape[:2] if image is not None else PREVIEW_HW)   # 시작 이미지가 있으면 그 크기로
        img = Image.fromarray(image) if image is not None else None
        return A.generate(pipe, to_dev(text), to_dev(NEG_TEXT), height=h, width=w, num_frames=frames,
                          generator=torch.Generator("cpu").manual_seed(seed), image=img, steps=steps,
                          device=DEVICE, offload=OFFLOAD)
    finally:
        for e, t in flags.items():
            getattr(pipe, e).train(t)
        place(ACTIVE)

PREVIEW = [(VAL_TEXT[i], VALIDATION_SEEDS[i], "trigger", i) for i in range(min(FIXED["preview_prompts"], len(VAL_TEXT)))]
if A.CONDITIONING == "ti2v" and any(s["kind"] == "clip" for s in SAMPLES):   # TI2V-5B 가 클립으로 배우는 I2V 모드도 본다
    PREVIEW += [(VAL_TEXT[0], VALIDATION_SEEDS[0], "i2v", 0)]
PREVIEW += [(DRIFT_TEXT[0], VALIDATION_SEEDS[0], "class_word", 0)]          # 마지막은 늘 트리거 없는 클립
DINO = VM.Dino.load("cpu") if FIXED["use_dino"] else None
CCIP = VM.CCIP.load() if FIXED["use_ccip"] else None
REF_FEATS = CCIP.features(THUMBS) if CCIP is not None and THUMBS else None
TRAIN_EMB = DINO.embed(THUMBS) if DINO is not None and THUMBS else None   # 생성 클립과 같은 letterbox 전처리
_pv_hw = sorted({tuple(START_IMAGES[i % len(START_IMAGES)].shape[:2]) if NEEDS_IMAGE or kind == "i2v" else PREVIEW_HW
                 for _, _, kind, i in PREVIEW})   # run_previews 와 같은 식: I2V·'i2v' 미리보기는 시작 이미지 크기
print("미리보기:", f"{len(PREVIEW)}개 × {FIXED['preview_frames']}프레임 {', '.join(f'{w}x{h}' for h, w in _pv_hw)}, "
      f"{FIXED['preview_steps']} step", "| DINO", "켬" if DINO else "끔", "| CCIP", "켬" if CCIP else "끔", "| 생성 오프로드", OFFLOAD)

In [ ]:
PREVIEW_HISTORY, STEP_METRICS, CALIB = {}, {}, None

def run_previews(step):
    global CALIB
    clips = []
    for j, (text, seed, kind, i) in enumerate(PREVIEW):
        img = START_IMAGES[i % len(START_IMAGES)] if NEEDS_IMAGE or kind == "i2v" else None
        clip = generate_clip(text, seed, FIXED["preview_frames"], FIXED["preview_steps"], image=img)
        clips.append(clip)
        V.save_mp4(RUN_DIR / f"preview_step{step:05d}_{j}_{kind}.mp4", clip, A.FPS)
    PREVIEW_HISTORY[step] = clips
    base = PREVIEW_HISTORY.get(0)
    if step == 0 and DINO is not None:      # 그림체 변화 기준: 같은 프롬프트 다른 시드의 베이스 차이
        alt = generate_clip(PREVIEW[-1][0], PREVIEW[-1][1] + 7, FIXED["preview_frames"], FIXED["preview_steps"],
                            image=START_IMAGES[0] if NEEDS_IMAGE else None)
        CALIB = VM.style_drift(DINO.embed(alt), DINO.embed(clips[-1]))
    if base is None:
        return
    mets = [VM.clip_metrics(c, base=b, dino=DINO, ccip=CCIP, ref_feats=REF_FEATS) for c, b in zip(clips, base)]
    trig = [m for m, (_, _, kind, _) in zip(mets, PREVIEW) if kind == "trigger"]
    mean = lambda key, ms: (float(np.mean([m[key] for m in ms if m.get(key) is not None]))
                            if any(m.get(key) is not None for m in ms) else None)
    drift = mets[-1].get("drift")
    memo = None
    if TRAIN_EMB is not None and not NEEDS_IMAGE:   # I2V 는 학습 데이터 그림에서 시작하므로 닮음이 암기의 증거가 아니다
        emb = np.concatenate([DINO.embed(c) for c, (_, _, kind, _) in zip(clips, PREVIEW) if kind == "trigger"])
        memo = float((emb @ TRAIN_EMB.T).max())
    # 색 변화·그림체 변화는 트리거 없는 클립으로 본다 (트리거 클립은 캐릭터 색이 바뀌는 것이 정상)
    STEP_METRICS[step] = dict(motion_ratio=mean("motion_ratio", trig), flicker_rel=mean("flicker_rel", trig),
                              color_shift=mets[-1].get("color_shift"), consistency=mean("consistency", trig),
                              drift_rel=(drift / CALIB) if drift is not None and CALIB else None,
                              id_pass=mean("id_pass", trig), id_slope=mean("id_slope", trig), memo=memo)

def save_ckpt(e, step):
    d = RUN_DIR / f"checkpoint-{step:05d}"
    d.mkdir(exist_ok=True)
    sd = {k: v.detach().to("cpu", torch.float32).contiguous() for k, v in get_peft_model_state_dict(getattr(pipe, e)).items()}
    meta = dict(family=A.FAMILY, expert=A.EXPERT_TAGS[e], step=str(step), rank=str(FIXED["rank"]), alpha=str(FIXED["alpha"]),
                trigger=USER["TRIGGER"], base_model=USER["MODEL_PATH"], target_scope=FIXED["target_scope"])
    return A.save_lora(sd, str(d / f"{USER['RUN_NAME']}_{A.EXPERT_TAGS[e]}.safetensors"), meta)

# 그래디언트 체크포인트: 켠 것과 끈 것의 손실이 같은지 한 번 확인한 뒤 켠다 (이미지 노트북과 같음)
def enable_gc(e):
    den = getattr(pipe, e)
    if not (FIXED["gradient_checkpointing"] and getattr(den, "_supports_gradient_checkpointing", False)):
        return "끔"
    s = SAMPLES[POOL[e][0]]
    g = torch.Generator("cpu").manual_seed(0)
    nz = torch.randn(s["x0"].shape, generator=g).to(DEVICE)
    sig = torch.tensor([sum(A.sigma_range(pipe, e)) / 2], device=DEVICE)
    mode = "i2v" if A.CONDITIONING == "i2v_concat" else "t2v"
    cond = to_dev({"cond": s["cond"]}) if s["cond"] is not None else None
    with torch.no_grad():
        off = float(A.forward_loss(pipe, den, s["x0"].to(DEVICE), cond, to_dev(TEXT[s["cap"]]), nz, sig, mode))
    den.enable_gradient_checkpointing()
    with torch.enable_grad():
        on = float(A.forward_loss(pipe, den, s["x0"].to(DEVICE), cond, to_dev(TEXT[s["cap"]]), nz, sig, mode).detach())
    den.zero_grad(set_to_none=True)
    tol = 1e-4 if WEIGHT_DTYPE == torch.float32 else 2e-2
    if abs(on - off) > tol * max(1.0, abs(off)):
        den.disable_gradient_checkpointing()
        return f"끔 (켜면 손실이 달라짐 {off:.5f} → {on:.5f})"
    return f"켬 (확인 손실 {off:.5f} / {on:.5f})"

### 4-3. 학습 루프

In [ ]:
# 그림체 보존(DOP): LoRA 를 끈 출력(베이스)과 켠 출력의 차이 — v2 이미지 노트북과 같은 방식
_captured = []

def _capture(module, args, kwargs, out):
    out = getattr(out, "sample", out)
    _captured.append(out[0] if isinstance(out, (tuple, list)) else out)

def preservation_loss(den, x0, cond, text, noise, sigma, mode):
    def run():
        _captured.clear()
        h = den.register_forward_hook(_capture, with_kwargs=True)
        try:
            A.forward_loss(pipe, den, x0, cond, text, noise, sigma, mode)
        finally:
            h.remove()
        if len(_captured) != 1:
            raise RuntimeError(f"디노이저가 {len(_captured)}번 불렸습니다")
        return _captured.pop()
    den.disable_adapters()
    try:
        with torch.no_grad():
            prior = run().detach()
    finally:
        den.enable_adapters()
    return torch.mean((run().float() - prior.float()) ** 2)

history, val_history, ckpt_paths = [], [], defaultdict(dict)
gen = torch.Generator("cpu").manual_seed(FIXED["seed"])
order = defaultdict(list)
nan_steps, global_step, t_start = 0, 0, time.time()
step_sec, train_peak, all_peak = 0.0, 0, 0   # 학습 단계만의 시간·최대 GPU 메모리, 미리보기 생성까지 포함한 최대 GPU 메모리
if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()
run_previews(0)                               # 0 단계 = 베이스 모델 (LoRA B=0)

def pick(e):
    pool = POOL[e]
    kinds = {SAMPLES[i]["kind"] for i in pool}
    kind = ("image" if float(torch.rand((1,), generator=gen)) < FIXED["image_share"] else "clip") if len(kinds) == 2 else kinds.pop()
    if not order[(e, kind)]:
        cand = [i for i in pool if SAMPLES[i]["kind"] == kind]
        order[(e, kind)] = [cand[j] for j in torch.randperm(len(cand), generator=gen).tolist()]
    return order[(e, kind)].pop()

for e, n_steps in PHASES:
    den = getattr(pipe, e)
    ACTIVE[:] = [e]; place(ACTIVE)
    den.train()
    print(f"\n[{A.EXPERT_TAGS[e]} 전문가] 그래디언트 체크포인트:", enable_gc(e))
    params = [p for n, p in den.named_parameters() if "lora_" in n]
    optimizer = torch.optim.AdamW(params, lr=FIXED["learning_rate"], betas=FIXED["betas"],
                                  weight_decay=FIXED["weight_decay"], eps=FIXED["eps"])
    lr_scheduler = get_scheduler(FIXED["lr_scheduler"], optimizer=optimizer, num_warmup_steps=FIXED["warmup_steps"],
                                 num_training_steps=n_steps)
    val_history.append((global_step, A.EXPERT_TAGS[e], validation_loss(e)))
    pbar = tqdm(range(1, n_steps + 1), desc=f"학습 {A.EXPERT_TAGS[e]}")
    for k in pbar:
        global_step += 1
        t_step = time.time()
        if DEVICE.type == "cuda":   # 학습 단계만의 최대 메모리를 미리보기 생성과 따로 잰다
            all_peak = max(all_peak, torch.cuda.max_memory_allocated())
            torch.cuda.reset_peak_memory_stats()
        optimizer.zero_grad(set_to_none=True)
        i = pick(e); s = SAMPLES[i]
        x0 = s["x0"].to(DEVICE)
        noise = torch.randn(x0.shape, generator=gen).to(DEVICE)
        sigma = sample_sigma(gen, e).to(DEVICE)
        mode = sample_mode(s, gen)
        cond = to_dev({"cond": s["cond"]}) if s["cond"] is not None else None
        drop = EMPTY_TEXT is not None and float(torch.rand((1,), generator=gen)) < FIXED["caption_dropout"]
        text = to_dev(EMPTY_TEXT if drop else TEXT[s["cap"]])
        loss = A.forward_loss(pipe, den, x0, cond, text, noise, sigma, mode)
        pres = float("nan")
        if torch.isfinite(loss):
            loss.backward()
            if PRES_TEXT is not None and global_step % FIXED["preservation_every"] == 0:
                pl = preservation_loss(den, x0, cond, to_dev(PRES_TEXT[s["cap"]]), noise, sigma, mode)
                if torch.isfinite(pl):
                    (FIXED["preservation_weight"] * pl).backward(); pres = float(pl.detach())
                else:
                    loss = pl
        grad_norm = float(torch.nn.utils.clip_grad_norm_(params, FIXED["max_grad_norm"])) if torch.isfinite(loss) else float("nan")
        if not math.isfinite(grad_norm):
            nan_steps += 1
            loss = loss.detach(); pl = None          # 건너뛴 단계의 계산 그래프(활성값)를 다음 단계까지 들고 있지 않는다
            optimizer.zero_grad(set_to_none=True)
            if nan_steps > FIXED["max_nan_steps"]:
                raise RuntimeError(f"NaN/Inf 손실·기울기가 {nan_steps}번 나와 멈춥니다. 학습률·데이터를 확인하세요.")
        else:
            optimizer.step(); lr_scheduler.step()
            history.append(dict(step=global_step, expert=A.EXPERT_TAGS[e], loss=float(loss.detach()), pres_loss=pres,
                                lr=lr_scheduler.get_last_lr()[0], grad_norm=grad_norm, sigma=float(sigma),
                                mode=mode, kind=s["kind"], frames=s["F"], tokens=s["tokens"], sec=time.time() - t_start))
            pbar.set_postfix(loss=f"{float(loss.detach()):.4f}")
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
            train_peak = max(train_peak, torch.cuda.max_memory_allocated())
        step_sec += time.time() - t_step
        if global_step % FIXED["save_every"] == 0 or k == n_steps:
            ckpt_paths[e][global_step] = save_ckpt(e, global_step)
        if global_step % FIXED["validate_every"] == 0 or k == n_steps:
            val_history.append((global_step, A.EXPERT_TAGS[e], validation_loss(e)))
            run_previews(global_step)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
del optimizer, lr_scheduler, params           # AdamW 상태(LoRA 파라미터당 8바이트)를 5절 생성 동안 GPU 에 남기지 않는다
loss = pl = None
ACTIVE[:] = []; place([])
loop_sec = time.time() - t_start
train_sec = step_sec                          # 학습 단계만 (미리보기·검증 손실·저장 제외)
if DEVICE.type == "cuda":
    all_peak = max(all_peak, torch.cuda.max_memory_allocated())
peak_gb = all_peak / 1e9 if DEVICE.type == "cuda" else float("nan")            # 미리보기 생성까지 포함
train_peak_gb = train_peak / 1e9 if DEVICE.type == "cuda" else float("nan")    # 학습 단계만
MAX_TOKENS = max((r["tokens"] for r in history), default=0)
pd.DataFrame(history).to_csv(RUN_DIR / "train_log.csv", index=False)
pd.DataFrame(val_history, columns=["step", "expert", "val_loss"]).to_csv(RUN_DIR / "val_loss.csv", index=False)
print(f"학습 {len(history)}단계: 학습 {train_sec / 60:.1f}분 + 미리보기·검증·저장 {(loop_sec - train_sec) / 60:.1f}분, "
      f"NaN 건너뜀 {nan_steps}번")
if DEVICE.type == "cuda":
    print(f"최대 GPU 메모리: 학습 단계 {train_peak_gb:.1f}GB (가장 큰 샘플 {MAX_TOKENS}토큰) | 미리보기 생성 포함 {peak_gb:.1f}GB")

## 5. 검증

### 5-1. 손실 곡선 (전문가별)

검증 손실은 학습에서 뺀 파일(3-1 절)로 잰다. 이미지·클립 종류마다 10개 미만이면 학습 샘플로 재므로, 암기할수록 계속 내려가 과적합을 보여 주지 못한다.

In [ ]:
h = pd.DataFrame(history)
vh = pd.DataFrame(val_history, columns=["step", "expert", "val_loss"])
fig, axes = plt.subplots(1, 3, figsize=(16, 3.4))
for tag, g in h.groupby("expert"):
    axes[0].plot(g.step, g.loss, alpha=0.3, label=f"{tag} 단계 손실")
    axes[0].plot(g.step, g.loss.ewm(alpha=0.05).mean(), label=f"{tag} 이동 평균")
axes[0].legend(fontsize=7); axes[0].set_title("학습 손실")
for tag, g in vh.groupby("expert"):
    axes[1].plot(g.step, g.val_loss, marker="o", label=tag)
axes[1].legend(fontsize=7); axes[1].set_title("검증 손실 (고정 노이즈·σ, 전문가 구간별)")
axes[2].plot(h.step, h.lr); axes[2].set_title("학습률")
plt.tight_layout(); plt.show()
print(vh.to_string(index=False))

### 5-2. 단계별 미리보기 (같은 프롬프트·같은 시드)

- 위에서 아래로 단계가 진행된다. 맨 위는 0 단계(베이스 모델)다.
- 마지막 줄의 프롬프트는 트리거를 일반 낱말로 바꾼 것이라 학습해도 베이스와 비슷해야 한다 (그림체 변화 확인).
- 표의 `motion_ratio` 는 베이스 대비 움직임 크기다. 0.6 보다 작으면 움직임이 줄어든 것이다.

In [ ]:
steps_ = sorted(PREVIEW_HISTORY)
for j, (_, _, kind, i) in enumerate(PREVIEW):
    sheet = V.contact_sheet([PREVIEW_HISTORY[s][j] for s in steps_], labels=[f"step {s}" for s in steps_], n=6, thumb_h=110)
    plt.figure(figsize=(13, 1.4 * len(steps_) + 0.6)); plt.imshow(sheet); plt.axis("off")
    plt.title(("[트리거 없음] " if kind == "class_word" else "") + (VALIDATION_PROMPTS[i] if kind == "trigger" else DRIFT_PROMPTS[0])[:90], fontsize=9)
    plt.show()
print(pd.DataFrame(STEP_METRICS).T.round(3).to_string())
_last = sorted(RUN_DIR.glob(f"preview_step{steps_[-1]:05d}_0_*.mp4"))
if _last and Video is not None and not TEST_FAMILY:
    display(Video(str(_last[0]), embed=True))

### 5-3. 저장한 LoRA 파일 다시 불러오기 확인

- 마지막 저장 파일(ComfyUI 형식 키)을 diffusers 로 다시 불러와 학습한 값과 같은지 본다.
- 모든 키가 `diffusion_model.blocks.` 로 시작하는지도 확인한다. ComfyUI 에서는 경고 `lora key not loaded` 가 0건이어야 한다 (5-6 절).

In [ ]:
RELOAD = {}
for e, _ in PHASES:
    den = getattr(pipe, e)
    final_step = max(ckpt_paths[e])
    path = ckpt_paths[e][final_step]
    from safetensors.torch import load_file
    keys = list(load_file(path).keys())
    bad = [k for k in keys if not k.startswith("diffusion_model.blocks.")]
    assert not bad and all(k.endswith((".lora_A.weight", ".lora_B.weight")) for k in keys), bad[:5]
    trained = {k.split(".lora_")[0] + ".lora_" + k.split(".lora_")[1].split(".")[0]: v.detach().float().cpu()
               for k, v in get_peft_model_state_dict(den).items()}
    den.delete_adapters("default")
    A.load_lora(pipe, path, e, adapter_name=f"reload_{A.EXPERT_TAGS[e]}")
    re_ = {k.split(".lora_")[0] + ".lora_" + k.split(".lora_")[1].split(".")[0]: v.detach().float().cpu()
           for k, v in get_peft_model_state_dict(den, adapter_name=f"reload_{A.EXPERT_TAGS[e]}").items()}
    assert set(re_) == set(trained), (len(re_), len(trained))
    diff = max(float((trained[k] - re_[k]).abs().max()) for k in trained)
    assert diff < 1e-6, f"저장 파일과 학습 값이 다릅니다 ({diff})"   # 어댑터가 파일의 fp32 값을 그대로 넣는다
    RELOAD[A.EXPERT_TAGS[e]] = dict(file=str(path), keys=len(keys), max_diff=diff)
    print(f"{A.EXPERT_TAGS[e]:>5}: {Path(path).name} 키 {len(keys)}개 (모두 ComfyUI 형식), 최대 차이 {diff:.2e}")
check = generate_clip(VAL_TEXT[0], VALIDATION_SEEDS[0], FIXED["preview_frames"], FIXED["preview_steps"],
                      image=START_IMAGES[0] if NEEDS_IMAGE else None)
assert check.ndim == 4 and check.std() > 0, "생성 클립이 한 색입니다"
print("다시 불러온 LoRA 로 생성 성공:", check.shape)

### 5-4. 최종 검증 세트 (LoRA 켬 / 끔 비교)

- 검증 프롬프트 전부를 공식 설정(step 수·shift·guidance)으로 그리고, 같은 시드로 LoRA 를 끈 베이스와 비교한다.
- 지표: 움직임 비율(베이스 대비), 깜빡임 비율, 채도 변화, 시간 일관성(DINO), 정체성 통과율(CCIP, 선택)
- 절댓값보다 베이스와의 비교로 본다. 최종 판정은 5-6 절대로 ComfyUI 에서 사람이 한다.

In [ ]:
FINAL_STEPS = FIXED["final_steps"] or A.VAL_STEPS
FINAL, FINAL_METRICS = [], []
for i, (text, seed) in enumerate(zip(VAL_TEXT, VALIDATION_SEEDS)):
    img = START_IMAGES[i % len(START_IMAGES)] if NEEDS_IMAGE else None
    lora = generate_clip(text, seed, FIXED["final_frames"], FINAL_STEPS, image=img)
    for e in A.EXPERTS:
        if getattr(getattr(pipe, e), "_hf_peft_config_loaded", False):
            getattr(pipe, e).disable_adapters()
    try:
        base = generate_clip(text, seed, FIXED["final_frames"], FINAL_STEPS, image=img)
    finally:
        for e in A.EXPERTS:
            if getattr(getattr(pipe, e), "_hf_peft_config_loaded", False):
                getattr(pipe, e).enable_adapters()
    V.save_mp4(RUN_DIR / f"final_{i:02d}_lora.mp4", lora, A.FPS); V.save_mp4(RUN_DIR / f"final_{i:02d}_base.mp4", base, A.FPS)
    m = VM.clip_metrics(lora, base=base, dino=DINO, ccip=CCIP, ref_feats=REF_FEATS)
    FINAL.append((lora, base)); FINAL_METRICS.append(dict(prompt=i, **m))
for i, (lora, base) in enumerate(FINAL):
    sheet = V.contact_sheet([base, lora], labels=["베이스", "LoRA"], n=6, thumb_h=110)
    plt.figure(figsize=(13, 3.2)); plt.imshow(sheet); plt.axis("off"); plt.title(VALIDATION_PROMPTS[i][:90], fontsize=9); plt.show()
FINAL_DF = pd.DataFrame(FINAL_METRICS)
print(FINAL_DF.round(3).to_string(index=False))

### 5-5. 새 프롬프트로 시험 (학습 데이터에 없던 장면)

- `EXTRA_PROMPTS` 를 바꿔 이 셀만 다시 실행해도 된다. 트리거가 없는 문장은 앞에 트리거를 붙인다.
- I2V 계열은 `EXTRA_IMAGE` 에 시작 이미지 경로를 준다 (비우면 검증 시작 이미지를 쓴다).

In [ ]:
EXTRA_PROMPTS = [
    "full body, static camera, she spins around once on a swing in a sunset park, clean anime illustration",
    "medium shot, camera slowly pushes in, she laughs and covers her mouth with her sleeve, snowy street, clean anime illustration",
]
EXTRA_IMAGE = ""
EXTRA_SEED = 2001

extra_prompts = [p.strip() if _TRIG.search(p) else f"{USER['TRIGGER']}, {p.strip()}" for p in EXTRA_PROMPTS]
place(["text_encoder"])
with torch.no_grad():
    extra_text = A.encode_prompts(pipe, extra_prompts, DEVICE, WEIGHT_DTYPE)
place([])
_img = start_image(load_still(Path(EXTRA_IMAGE))) if EXTRA_IMAGE else START_IMAGES[0]
extra = [generate_clip(t, EXTRA_SEED + j, FIXED["final_frames"], FINAL_STEPS, image=_img if NEEDS_IMAGE else None)
         for j, t in enumerate(extra_text)]
for j, c in enumerate(extra):
    V.save_mp4(RUN_DIR / f"extra_{j:02d}_seed{EXTRA_SEED + j}.mp4", c, A.FPS)
(RUN_DIR / "extra_prompts.txt").write_text("\n".join(f"{j:02d} seed {EXTRA_SEED + j}: {p}" for j, p in enumerate(extra_prompts)) + "\n",
                                           encoding="utf-8")
sheet = V.contact_sheet(extra, labels=[p[len(USER["TRIGGER"]) + 2:][:40] for p in extra_prompts], n=6, thumb_h=110)
plt.figure(figsize=(13, 1.6 * len(extra) + 0.6)); plt.imshow(sheet); plt.axis("off"); plt.show()
print(f"{len(extra)}개 저장:", RUN_DIR)

### 5-6. ComfyUI 에서 확인할 것 (노트북 밖)

- A14B 는 LoRA 가 전문가마다 하나씩 2개다. 고노이즈 모델 로더 뒤에 `_high` 파일, 저노이즈 모델 로더 뒤에 `_low` 파일을 각각 `LoraLoaderModelOnly` 로 건다 (키 이름이 같아서 바꿔 걸어도 경고가 나지 않는다).
  - 클립 없이 이미지만으로 학습하면 `_low` 파일 하나만 나온다. 고노이즈 모델에는 LoRA 를 걸지 않는다.
  - 두 전문가는 서로 다른 단계 폴더에 저장된다 (저노이즈 먼저, 고노이즈 나중). 6-2 절이 추천한 짝을 쓴다.
- ComfyUI 콘솔에 `lora key not loaded` 경고가 0건인지 본다.
- 랭크와 알파가 같으므로 강도 1.0 이 학습 때 세기다. 움직임이 줄었으면 고노이즈 LoRA 를 0.5 로 낮춰 비교한다.
- T2V-A14B 로 학습한 LoRA 를 I2V-A14B 에 걸어 효과가 이어지는지 따로 확인한다 (블록 모양은 같지만 효과는 미확인).
- 공식 설정으로 비교한다: I2V 40 step, shift 5.0 (diffusers 저장소 기본 3.0 과 다름), CFG 3.5·3.5 / T2V shift 12.0, CFG 4.0(고)·3.0(저).
- 공식 부정 프롬프트에는 '정지 화면'·'그림체' 단어가 들어 있다. 셀화 그림체가 흐려지면 부정 프롬프트를 바꿔 비교한다.

## 6. 보완

### 6-1. 증상 진단

- 미리보기 지표(단계별)와 검증 손실로 증상을 찾는다. 기준값은 `remedy.THR` 에 있다 [제안, 실측으로 다시 맞춘다].
- 지표가 없는 규칙(예: CCIP 를 끈 경우의 정체성)은 건너뛴다.

In [ ]:
SYMPTOM_KO = dict(overfit="과적합·암기", underfit="과소학습", identity_weak="정체성 약함", identity_drift="클립 안 정체성 흔들림",
                  motion_loss="움직임 손실", motion_excess="움직임 과다", flicker="깜빡임", color_shift="색 변화",
                  style_drift="그림체 변화")
_main = A.EXPERT_TAGS[PHASES[0][0]]
VAL_HELDOUT = VAL_ITEMS[PHASES[0][0]][1]
# 학습 샘플로 잰 손실은 암기할수록 계속 내려가므로 과적합·추천 단계 판단에 쓰지 않는다
VAL_MAIN = {int(s): v for s, t, v in val_history if t == _main} if VAL_HELDOUT else None
if VAL_MAIN and len(PHASES) > 1:   # 첫 전문가는 다음 전문가를 학습하는 동안 그대로라, 그 뒤 단계의 검증 손실은 마지막 값과 같다
    VAL_MAIN.update({int(s): VAL_MAIN[max(VAL_MAIN)] for s in STEP_METRICS if int(s) > PHASES[0][1]})
if not VAL_HELDOUT:
    print("참고: 검증 손실이 학습 샘플로 잰 값이라 진단에 쓰지 않습니다 (이미지·클립 종류마다 10개 이상이면 따로 뺀 파일로 잽니다).")
# I2V 미리보기는 학습 데이터 그림(시작 이미지)에서 시작해 베이스도 CCIP 를 통과한다: memo 처럼 통과율을 진단·추천에 쓰지 않는다
_SM = {int(k): (dict(v, id_pass=None) if NEEDS_IMAGE else dict(v)) for k, v in STEP_METRICS.items()}
_AFTER = PHASES[0][1] if len(PHASES) > 1 else None   # A14B: 이 단계 뒤 미리보기는 두 전문가의 LoRA 를 함께 쓴 것
if NEEDS_IMAGE and any(m.get("id_pass") is not None for m in STEP_METRICS.values()):
    print("참고: I2V 는 학습 데이터 그림에서 시작해 베이스도 CCIP 를 통과하므로, 정체성 통과율(5-2 표)은 진단·추천에 쓰지 않습니다.")
_MET_KO = dict(motion_ratio="움직임", flicker_rel="깜빡임", color_shift="색 변화", consistency="시간 일관성(DINO)",
               drift_rel="그림체 변화(DINO)", memo="암기(DINO)", id_pass="정체성(CCIP)", id_slope="클립 안 정체성(CCIP)")
_missing = [v for k, v in _MET_KO.items() if STEP_METRICS and all(m.get(k) is None for m in STEP_METRICS.values())
            and not (k == "memo" and NEEDS_IMAGE)]   # I2V 는 암기 지표를 쓰지 않는다
if _missing:
    print("값이 없어 관련 규칙을 건너뛴 지표:", ", ".join(_missing))
    if "움직임" in _missing:
        print("  움직임 비율은 베이스(LoRA 0) 미리보기가 거의 움직이지 않으면 내지 않는다. 움직임이 있는 검증 프롬프트를 쓴다.")
    if any("DINO" in m or "CCIP" in m for m in _missing):
        print("  DINO·CCIP 지표는 끄거나(LORA_FIXED_OVERRIDES 의 use_dino·use_ccip) 4-2 셀에서 불러오지 못하면 빠진다 "
              "(이유는 4-2 셀 출력).")
SYMPTOMS = RM.diagnose(_SM, VAL_MAIN, after=_AFTER)
if SYMPTOMS:
    print(pd.DataFrame([dict(증상=SYMPTOM_KO.get(s["symptom"], s["symptom"]), 근거=s["evidence"], 정도=s["severity"])
                        for s in SYMPTOMS]).to_string(index=False))
else:
    print("뚜렷한 증상 없음 (지표 기준). 컨택트 시트와 ComfyUI 결과로 최종 판단한다.")

### 6-2. 체크포인트 추천

In [ ]:
BEST_STEP = RM.pick_best_step(_SM, VAL_MAIN, after=_AFTER)
# 전문가는 차례로 학습하므로, 어느 단계의 미리보기는 '그때까지 학습한 전문가들의 그 시점 LoRA' 로 그린 것이다.
# 실제로 함께 잰 짝만 추천한다: 고른 단계 뒤에 학습한 전문가는 LoRA 없이(None), 앞서 끝난 전문가는 마지막 파일.
RECOMMEND, _start = {}, 0
for e, n in PHASES:
    steps_e = sorted(ckpt_paths[e])
    if BEST_STEP is None or BEST_STEP >= _start + n:
        RECOMMEND[A.EXPERT_TAGS[e]] = ckpt_paths[e][steps_e[-1]]
    elif BEST_STEP <= _start:
        RECOMMEND[A.EXPERT_TAGS[e]] = None
    else:
        cand = [s for s in steps_e if s <= BEST_STEP]
        RECOMMEND[A.EXPERT_TAGS[e]] = ckpt_paths[e][cand[-1]] if cand else None
    _start += n
print("추천 단계:", BEST_STEP)
for tag, pth in RECOMMEND.items():
    print(f"  {tag:>5}: {pth if pth else '없음 (고른 단계는 이 전문가를 학습하기 전이라 LoRA 없이 쓴다)'}")
_late = [A.EXPERT_TAGS[e] for e, _ in PHASES if RECOMMEND[A.EXPERT_TAGS[e]] is None and ckpt_paths[e]]
if _late:
    print(f"참고: {', '.join(_late)} LoRA 까지 함께 쓰려면 마지막 단계({global_step})의 짝(전문가마다 마지막 파일)을 씁니다. "
          "그 단계의 지표는 5-2 표에서 봅니다.")
print("ComfyUI 에서는 이 파일들을 model-inbox 절차로 등록해 5-6 절대로 확인한다.")

### 6-3. 다음 실행 설정 만들기

- 증상의 우선순위대로 한 번에 2개까지만 값을 바꾼다 (무엇이 효과였는지 알 수 있게).
- 결과는 `run_many.py` 목록 형식의 `next_run.json` 이다. 실행 이름은 판이 올라간다 (예: `_v1` → `_v2`).
- 바로 돌리지 말고, 컨택트 시트와 ComfyUI 결과를 보고 사람이 승인한 뒤 실행한다.

In [ ]:
# 이 계열·이 데이터에서 효과가 없는 값은 제안 후보에서 뺀다 (바꿀 수 있는 2개를 헛되이 쓰지 않게)
_drop = {"low_steps_share", "shift_low", "shift_high", "high_uses_images"} if len(A.EXPERTS) == 1 else {"shift"}
_trained = {e for e, _ in PHASES}
if len(A.EXPERTS) > 1 and A.EXPERTS[0] not in _trained:   # 고노이즈를 학습하지 않음 (클립 없음 또는 EXPERT=low)
    _drop |= {"shift_high", "low_steps_share", "high_uses_images"}
if len(A.EXPERTS) > 1 and A.EXPERTS[1] not in _trained:   # 저노이즈를 학습하지 않음 (EXPERT=high)
    _drop |= {"shift_low", "low_steps_share"}
_pool_kinds = [{SAMPLES[i]["kind"] for i in POOL[e]} for e in _trained]   # 학습한 전문가가 실제로 뽑은 샘플 종류
_kinds = set().union(*_pool_kinds)
if A.CONDITIONING == "i2v_concat" or not any(len(k) > 1 for k in _pool_kinds):   # image_share 는 두 종류가 섞인 풀에서만 쓴다
    _drop |= {"image_share"}
if "image" not in _kinds:
    _drop |= {"high_uses_images"}
if "clip" not in _kinds:                                   # 클립에만 쓰는 값
    _drop |= {"min_motion", "min_frames", "window_mode", "windows_per_segment", "ti2v_i2v_share"}
NEXT_OVERRIDES, CHANGES = RM.propose({k: v for k, v in FIXED.items() if k not in _drop}, SYMPTOMS)
# 이번 실행에 이미 덮어쓴 값(OVERRIDES)을 이어받아야 이전 보완이 되돌아가지 않는다
ALL_OVERRIDES = {**OVERRIDES, **NEXT_OVERRIDES}
if "rank" in ALL_OVERRIDES:
    ALL_OVERRIDES["alpha"] = ALL_OVERRIDES["rank"]
# 다음 실행에는 이번에 정한 계열·낱말·범위를 그대로 적는다 (auto 를 풀어 쓴 값)
#   계열은 사용자가 정한 그대로(보통 auto): next_run.json 에서 model_path 만 바꿔도 새 모델에 맞춰 다시 정한다
_user_next = dict(USER, MODEL_FAMILY=USER["MODEL_FAMILY"].strip() or "auto", CLASS_WORD=CLASS_WORD, TARGET_SCOPE=SCOPE)
NEXT_PLAN = RM.next_run_plan(_user_next, NEXT_OVERRIDES, CHANGES, BEST_STEP, base_overrides=OVERRIDES)
(RUN_DIR / "next_run.json").write_text(json.dumps(NEXT_PLAN, ensure_ascii=False, indent=2, default=str), encoding="utf-8")
if CHANGES:
    print(pd.DataFrame([dict(값=c.get("key"), 지금=c.get("old"), 다음=c.get("new"),
                             이유=SYMPTOM_KO.get(c.get("symptom"), c.get("symptom"))) for c in CHANGES]).to_string(index=False))
    print("\n다음 실행:", RUN_DIR / "next_run.json")
    print(f"  python run_many.py {RUN_DIR / 'next_run.json'} --notebook lora_training_video.ipynb")
    print("  또는 LORA_FIXED_OVERRIDES='" + json.dumps(ALL_OVERRIDES, ensure_ascii=False) + "' 와 새 LORA_RUN_NAME 으로 이 노트북 실행")
else:
    if SYMPTOMS:   # 증상은 있는데 이 계열·데이터에서 바꿀 수 있는 값이 없음
        print("증상(" + ", ".join(SYMPTOM_KO.get(s["symptom"], s["symptom"]) for s in SYMPTOMS) + ")은 있지만, "
              "이 모델·데이터에서 바꿀 수 있는 학습 값이 없습니다.")
        print("  예: 이미지만 있는 데이터의 움직임 손실은 클립을 넣어야 나아집니다.")
    else:
        print("바꿀 값 없음 (지표상 증상 없음).")
    print("  데이터·캡션을 보강하거나 더 이른 체크포인트를 비교합니다. next_run.json 은 판만 올린 같은 설정입니다:",
          RUN_DIR / "next_run.json")

### 6-4. 결과 요약 저장

In [ ]:
summary = dict(
    notebook="video", family=A.FAMILY, model_path=USER["MODEL_PATH"] if not TEST_FAMILY else "tiny-test", run_dir=str(RUN_DIR),
    trigger=USER["TRIGGER"], expert_plan=[(A.EXPERT_TAGS[e], n) for e, n in PHASES],
    family_auto=not TEST_FAMILY and USER["MODEL_FAMILY"].strip() in ("", "auto"), target_scope=FIXED["target_scope"],
    class_word=CLASS_WORD,
    data=dict(images=int((df.kind == "image").sum()), clips=int((df.kind == "video").sum()), samples=len(SAMPLES),
              buckets=dict(Counter(f"{s['F']}x{s['H']}x{s['W']}" for s in SAMPLES)), problems=len(problems)),
    fixed=dict(FIXED), overrides=OVERRIDES, steps_done=len(history), nan_steps=nan_steps,
    # train_minutes 는 학습 단계만, preview_minutes 는 학습 중 미리보기·검증 손실·저장. peak_gpu_gb 는 미리보기 생성까지 포함한
    # 최대, train_peak_gpu_gb 는 학습 단계만 (가장 큰 샘플이 max_train_tokens 토큰)
    train_minutes=round(train_sec / 60, 2), preview_minutes=round((loop_sec - train_sec) / 60, 2),
    peak_gpu_gb=None if math.isnan(peak_gb) else round(peak_gb, 2),
    train_peak_gpu_gb=None if math.isnan(train_peak_gb) else round(train_peak_gb, 2), max_train_tokens=MAX_TOKENS,
    val_loss=[dict(step=int(s), expert=t, val_loss=round(v, 5)) for s, t, v in val_history],
    checkpoints={A.EXPERT_TAGS[e]: {int(s): str(p) for s, p in d.items()} for e, d in ckpt_paths.items()},
    reload=RELOAD, holdout_files=sorted(HOLDOUT), val_heldout=VAL_HELDOUT,
    step_metrics={int(k): v for k, v in STEP_METRICS.items()},
    final_metrics=FINAL_METRICS, symptoms=SYMPTOMS, best_step=BEST_STEP, recommend=RECOMMEND,
    next_overrides=ALL_OVERRIDES, roundtrip_psnr=ROUNDTRIP_PSNR,
    # run_many.py 기록용: 따로 뺀 파일로 잰 검증 손실이 가장 낮은 저장 단계(첫 전문가), 추천 LoRA 파일 크기 합(MB),
    # 마지막 20단계 평균 손실
    best_val_step=min(((s, v) for s, t, v in val_history if t == _main and s in ckpt_paths[PHASES[0][0]]),
                      key=lambda x: x[1], default=(None, None))[0] if VAL_HELDOUT else None,
    lora_file_mb=round(sum(Path(p).stat().st_size for p in RECOMMEND.values() if p) / 1e6, 1),
    final_train_loss=round(float(np.mean([r["loss"] for r in history[-20:]])), 5) if history else None,
    versions={k: _md.version(k) for k in list(PINNED) + ["av", "ftfy"]},
)
(RUN_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1, default=str), encoding="utf-8")
print(json.dumps({k: summary[k] for k in ("family", "expert_plan", "data", "train_minutes", "preview_minutes", "peak_gpu_gb",
                                           "train_peak_gpu_gb", "max_train_tokens", "best_step", "recommend")},
                 ensure_ascii=False, indent=1, default=str))
if TEST_FAMILY:
    print(f"\nNOTEBOOK TEST OK {TEST_FAMILY}")